> **运行环境准备**：在执行后续代码之前，先完成两项基础设置。
> 1. `warnings.filterwarnings("ignore")`：关闭运行时的告警信息，避免 LangChain 旧版 API（如 `langchain_classic`）的 deprecation warning 干扰教学阅读。
> 2. 清理 `HTTP_PROXY` / `HTTPS_PROXY` / `ALL_PROXY` 等代理环境变量：防止本地或容器内的代理配置影响 vLLM、Xinference、Milvus 等服务的正常访问。

> **本课件的实跑记录**：92 个代码单元在 `env_rag_uv` 内核真实执行，耗时 381.1 秒，67 个单元有输出，0 报错。
> 预处理口径：单份 PDF 29 页 → 101 个标题分段 → 二次切分 121 个 chunk；Unstructured 路线 75 个章节块 → 88 个 chunk；两条路线的质量检查都通过。
> 本课件的 Markdown 预处理章节就是 `knowledge_md/` 的生成器，重跑之后 `git status --short knowledge_md` 仍然为空，说明语料可复现。
> 检索层评估（3 条）：Hit@5 `0.667`、MRR `0.667`，未命中的是「数据可靠性包括哪些层？」。
> LLM-as-Judge 平均分：RAG `3.22` 对比基线无 RAG `2.00`；其中第 1 题 RAG 只答了「我不知道」，被压到 `1.33` 对 `3.33`——这正是评估环节要暴露的问题，不是脚本故障。
> 前置服务：Milvus（`localhost:19530`）、vLLM、Xinference。LLM 文本与评分每次都会变，摘录只是**一次真实运行**的样本。

In [ ]:
import warnings
warnings.filterwarnings("ignore")


import os
for k in [
    "HTTP_PROXY",
    "HTTPS_PROXY",
    "ALL_PROXY",
    "http_proxy",
    "https_proxy",
    "all_proxy"
]:
    os.environ.pop(k, None)

# 项目说明

本项目是一个面向初学者的**检索增强生成（RAG）入门实践**，以 LangChain 为核心框架，完整演示如何从零搭建一个"本地知识库问答"系统。项目将带你依次了解 RAG 的基本原理、环境准备、文档加载与切分、向量化与向量存储、检索策略、大模型生成，以及简单的 LLM-as-Judge 评估方法。

## 1. 项目总览

整个 Notebook 按照"数据 → 索引 → 检索 → 生成 → 评估"的链路组织，帮助学习者建立 RAG 系统的端到端视角。主要内容包括：

1. **环境准备**：介绍 UV 虚拟环境搭建、依赖库安装及 vLLM / Xinference 模型服务配置；
2. **知识库构建**：使用 LangChain 加载 PDF、Markdown 等本地文档，进行文本切分、嵌入向量化，并存储到向量数据库；
3. **检索与生成**：基于向量检索召回相关片段，结合大语言模型生成带有上下文的回答；
4. **评估闭环**：通过 LLM-as-Judge 对比 RAG 回答与基线回答的忠实度、完整性和相关性；
5. **扩展方向**：简要介绍 Ragas 等系统化评估框架，为后续优化提供参考。

学习完本项目后，你将能够独立搭建一个可运行、可评估的本地知识库问答系统，并理解 RAG 链路中每一步的作用与调优点。

## 2. 关键库与环境说明

> **环境说明**：本 Notebook 使用 `uv` 创建的虚拟环境 `env_rag_uv` 运行，Python 版本为 3.12，Jupyter Kernel 名称为 `env_rag_uv`。下面列出本项目核心依赖库、用途说明以及实际安装版本号。实际运行时，也可以通过下方代码单元查看当前环境中各库的真实安装版本。

| 库名 | 版本 | 说明 |
|---|---|---|
| `langchain` | 1.4.7 | LangChain 主框架，提供链、提示词、模型接口等核心能力 |
| `langchain-core` | 1.7.0 | LangChain 核心抽象与运行时（Messages、Runnables、输出解析等） |
| `langchain-community` | 0.4.2 | 社区版集成（如 `PyMuPDFLoader` 等文档加载器） |
| `langchain-openai` | 1.7.2 | OpenAI 兼容接口，本项目用于 vLLM 大模型与 Xinference Embedding |
| `langchain-milvus` | 0.4.0 | Milvus 向量数据库的 LangChain 封装（需搭配 pymilvus 3.x） |
| `langchain-classic` | 1.0.8 | 兼容旧版 API（`RetrievalQA`、`ConversationalRetrievalChain`） |
| `langchain-text-splitters` | 1.1.3 | 文本切分器（`RecursiveCharacterTextSplitter` 等） |
| `langchain-cohere` | 未安装 | Cohere Rerank 云端精排（可选，缺 Key 时自动跳过） |
| `pymilvus` | 3.0.2 | Milvus Python SDK，用于连接与操作向量库 |
| `PyMuPDF` | 2.0 | PDF 解析器，`PyMuPDFLoader` 的底层依赖 |
| `rank-bm25` | 0.2.2 | 教学用手写 BM25 关键词检索 |
| `xinference-client` | 4.0.0 | Xinference 原生 REST 客户端（可选，Embedding/Rerank 走 OpenAI 兼容接口） |
| `python-dotenv` | 1.2.4 | 从 `.env` 文件读取服务地址与 API Key |
| `httpx` | 0.28.1 | HTTP 客户端，用于 vLLM / Xinference Rerank / Jina Rerank 调用 |
| `openai` | 3.37.0 | OpenAI SDK，OpenAI 兼容接口所需 |
| `pandas` | 2.3.3 | 数据处理与展示 |
| `numpy` | 2.5.4 | 数值计算 |
| `ipykernel` | 7.4.0 | Jupyter Notebook 运行所需 Kernel |

In [ ]:
# 查看当前 env_rag_uv 环境中关键库的实际安装版本
from importlib.metadata import version, PackageNotFoundError

packages = [
    ("langchain", "LangChain 主框架"),
    ("langchain-core", "LangChain 核心抽象"),
    ("langchain-community", "社区版组件与加载器"),
    ("langchain-openai", "OpenAI 兼容接口（vLLM / Xinference）"),
    ("langchain-milvus", "Milvus 向量库封装"),
    ("langchain-classic", "旧版 Chain API 兼容"),
    ("langchain-text-splitters", "文本切分器"),
    ("langchain-cohere", "Cohere Rerank（可选）"),
    ("pymilvus", "Milvus Python SDK"),
    ("PyMuPDF", "PDF 解析"),
    ("rank-bm25", "BM25 关键词检索（教学版）"),
    ("xinference-client", "Xinference 原生客户端（可选）"),
    ("python-dotenv", ".env 环境变量读取"),
    ("httpx", "HTTP 客户端"),
    ("openai", "OpenAI SDK"),
    ("pandas", "数据处理"),
    ("numpy", "数值计算"),
    ("ipykernel", "Jupyter Kernel"),
]

print(f"{'库名':<35} {'版本':<15} {'说明'}")
print("-" * 80)
for name, desc in packages:
    try:
        ver = version(name)
    except PackageNotFoundError:
        ver = "未安装"
    print(f"{name:<35} {ver:<15} {desc}")

**预期输出：**

```text
库名                                  版本              说明
--------------------------------------------------------------------------------
langchain                           1.4.7           LangChain 主框架
langchain-core                      1.7.0           LangChain 核心抽象
langchain-community                 0.4.2           社区版组件与加载器
langchain-openai                    1.7.2           OpenAI 兼容接口（vLLM / Xinference）
langchain-milvus                    0.4.0           Milvus 向量库封装
langchain-classic                   1.0.8           旧版 Chain API 兼容
langchain-text-splitters            1.1.3           文本切分器
langchain-cohere                    未安装             Cohere Rerank（可选）
pymilvus                            3.0.2           Milvus Python SDK
PyMuPDF                             2.0             PDF 解析
rank-bm25                           0.2.2           BM25 关键词检索（教学版）
xinference-client                   4.0.0           Xinference 原生客户端（可选）
python-dotenv                       1.2.4           .env 环境变量读取
httpx                               0.28.1          HTTP 客户端
openai                              3.37.0          OpenAI SDK
pandas                              2.3.3           数据处理
numpy                               2.5.4           数值计算
ipykernel                           7.4.0           Jupyter Kernel
```

## 3. 统一模型配置说明

本 notebook 统一使用以下三个模型配置：

- **大语言模型**：vLLM 部署的 Qwen3，通过 `langchain.chat_models.init_chat_model` 以 OpenAI 兼容协议调用；
- **Embedding 模型**：Xinference 部署的 Qwen3 Embedding，通过 `langchain_openai.OpenAIEmbeddings` 调用其 OpenAI 兼容接口；
- **Reranker 模型**：Xinference 部署的 Qwen3 Reranker，通过 OpenAI 兼容风格的 `{XINFERENCE_OPENAI_BASE_URL}/rerank` 接口调用。

三个服务的地址、模型名与密钥全部放在仓库根目录 `.env` 中，代码里只读环境变量，不写死任何 IP 或 Key。
下方统一定义 `my_llm`、`my_emb` 以及 `RERANKER_BASE_URL` / `RERANKER_MODEL` 配置常量。
后续章节为了便于单独运行，可能会在对应代码单元中重复定义同样配置。

In [ ]:
import os
import httpx
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings

# 读取仓库根目录 .env（notebook 位于子目录时会自动向上查找）
_ = load_dotenv(find_dotenv())

# ── 1. 大语言模型配置（vLLM，OpenAI 兼容接口）────────────────
my_llm = init_chat_model(
    model=os.environ["vLLM_QWEN_MODEL_ID"],
    model_provider="openai",
    base_url=os.environ["vLLM_QWEN_BASE_URL"],
    api_key=os.environ["vLLM_QWEN_API_KEY"],
    http_client=httpx.Client(trust_env=False),
    temperature=0.3,
    max_tokens=3000,
)

# ── 2. Embedding 模型配置（Xinference，OpenAI 兼容接口）───────
my_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)

# ── 3. Reranker 模型配置（Xinference，/v1/rerank）────────────
RERANKER_BASE_URL = os.environ["XINFERENCE_OPENAI_BASE_URL"]
RERANKER_MODEL = os.environ["XINFERENCE_RERANKER_MODEL_ID"]

# ── 4. 向量数据库地址 ────────────────────────────────────────
MILVUS_URI = os.environ["MILVUS_URI"]

print("LLM       :", os.environ["vLLM_QWEN_MODEL_ID"], "@", os.environ["vLLM_QWEN_BASE_URL"])
print("Embedding :", os.environ["XINFERENCE_EMBEDDING_MODEL_ID"], "@", os.environ["XINFERENCE_OPENAI_BASE_URL"])
print("Reranker  :", RERANKER_MODEL)
print("Milvus    :", MILVUS_URI)

**预期输出：**

```text
LLM       : my_qwen3.8_flash_next @ http://8.159.144.108:3791/v1
Embedding : my_qwen3_embedding_0.6b @ http://8.159.144.108:3797/v1
Reranker  : my_qwen3_reranker_0.6b
Milvus    : http://localhost:19530
```

In [ ]:
import os
import httpx

# Embedding 自检：确认向量维度与 Xinference 返回一致
vector = my_emb.embed_query("什么是人工智能")
print("向量维度：", len(vector))
print("前 5 维：", [round(v, 4) for v in vector[:5]])

# Rerank 自检：调用 Xinference 的 OpenAI 兼容 /rerank 接口
response = httpx.post(
    RERANKER_BASE_URL.rstrip("/") + "/rerank",
    headers={"Authorization": "Bearer " + os.environ["XINFERENCE_API_KEY"]},
    json={
        "model": RERANKER_MODEL,
        "query": "什么是人工智能",
        "documents": [
            "人工智能是研究如何让机器模拟人类智能的学科",
            "今天的天气很好，适合出去散步",
        ],
        "top_n": 2,
    },
    timeout=60,
    trust_env=False,
)
response.raise_for_status()
for item in response.json()["results"]:
    print(f"index={item['index']}  score={item['relevance_score']:.4f}")

**预期输出：**

```text
向量维度： 1024
前 5 维： [-0.0427, -0.0347, -0.0083, -0.0811, -0.0452]
index=0  score=1.0000
index=1  score=0.0001
```

# 数据预处理

## 1. PDF 数据简单预处理

为构建我们的本地知识库，我们需要对以多种类型存储的本地文档进行处理，读取本地文档并通过前文描述的 Embedding 方法将本地文档的内容转化为词向量来构建向量数据库。接下来以一些实际示例入手，来讲解如何对本地文档进行处理。  

### 1.1 数据读取
如果没有知识库文件，可以选择将知识库源数据放置在 knowledge_db 目录下

In [ ]:
# !cp -r /workspace/DataSet/rag2026/VMAX-S/   /workspace/llm-rag/knowledge_path/

PDF文档为例：   
可以使用 LangChain 的 PyMuPDFLoader 来读取知识库的 PDF 文件。PyMuPDFLoader 是 PDF 解析器中速度最快的一种，结果会包含 PDF 及其页面的详细元数据，并且每页返回一个文档。

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

# 创建一个 PyMuPDFLoader Class 实例，输入为待加载的 pdf 文档路径
loader = PyMuPDFLoader("../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf")

# 调用 PyMuPDFLoader Class 的函数 load 对 pdf 文件进行加载
pdf_pages = loader.load()

文档加载后储存在 `pdf_pages` 变量中:
- `pdf_pages` 的变量类型为 `List`
- 打印 `pdf_pages` 的长度可以看到 pdf 一共包含多少页

In [ ]:
print(f"载入后的变量类型为：{type(pdf_pages)}，",  f"该 PDF 一共包含 {len(pdf_pages)} 页")

**预期输出：**

```text
载入后的变量类型为：<class 'list'>， 该 PDF 一共包含 29 页
```

`pdf_pages` 中的每一元素为一个文档，变量类型为 `langchain_core.documents.base.Document`, 文档变量类型包含两个属性
- `page_content` 包含该文档的内容。
- `meta_data` 为文档相关的描述性数据。

In [ ]:
pdf_page = pdf_pages[20]
print(f"每一个元素的类型：{type(pdf_page)}.", 
    f"该文档的描述性数据：{pdf_page.metadata}", 
    f"查看该文档的内容:\n{pdf_page.page_content}", 
    sep="\n------\n")

**预期输出：**

```text
每一个元素的类型：<class 'langchain_core.documents.base.Document'>.
------
该文档的描述性数据：{'producer': 'Apache FOP Version 2.6', 'creator': 'DITA Open Toolkit', 'creationdate': '2023-05-23T21:45:33+08:00', 'source': '../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf', 'file_path': '../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf', 'total_pages': 29, 'format': 'PDF 1.4', 'title': '目录', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20230523214533+08'00'", 'page': 20}
------
查看该文档的内容:
7 可靠性设计
容错能力
ZXVMAX-S通过对关键软件资源的定时检测、实时任务监控、存储保护、数据校验、操作日志
信息保存等手段，可有效地防止小软件故障对系统所造成的冲击，提高了软件系统的容错能力
（即软件错误情况下的自愈能力）。
故障监视及处理
ZXVMAX-S具备自动检测与诊断系统软硬件故障的功能，可对故障硬件实施自动隔离、倒换、
重新启动、重新加载等处理。
满足漏洞扫描
为了避免存在安全漏洞使在网设备受外部攻击，需要使系统满足主流安全漏洞扫描工具的要求。
ZXVMAX-S满足Nessus和CD工具安全漏洞扫描的要求。
7.3 数据可靠性
ZXVMAX-S产品在不同分层架构下的数据都可以确保数据可靠和可恢复。

采集层
采集层运行环境部署采用冗余磁盘阵列设计，在服务器损坏、单一硬盘故障等场景下，通过
替换故障硬件后即可工作，数据不会丢失。同时，系统支持数据补充采集，确保系统数据连
续性。

存储共享层
存储共享层的MPP数据库，根据需要都设置了多个冗余备份，单一节点的损坏不会影响整系
统的运行，也不会导致数据丢失。在替换损坏硬件后，系统能自动进行数据冗余备份的补全
和负载均衡。
存储共享层的MPP数据库，设计运行在冗余磁盘阵列上，并且是具备双控制器的外置磁阵，
在确保性能的同时避免了磁阵数据丢失。

应用分析层
应用分析层操作系统和软件模块都部署在冗余磁盘阵列上，系统软件、配置数据等信息在单
一磁盘损坏，主机故障等情形下都不会丢失。
7.4 组网可靠性
ZXVMAX-S产品根据需要，可以通过交换机冗余备份，各主机网络的以太网隧道冗余备份，实
现对业务和设备的保护。
在单一交换机故障时，备用交换机会及时启用，避免业务中断。
在单一的网络传输故障时，备用的传输链路仍然生效，也能避免业务中断。
SJ-20220623151803-017 | 2023-03-30（R1.0）
17
```

In [ ]:
# 假设环境中缺少 NLTK 数据包中的 punkt 资源，这是 NLTK 库中用于分词的一个重要组件。解决这个问题的方法是按照报错信息中提到的步骤下载并安装 punkt 资源。
# import nltk
# nltk.download('punkt', quiet=True)

### 1.2 数据清洗

简单版本只做通用空白归一化：把连续换行和空格压缩为一个空格。它容易理解，但不会识别页眉页脚、标题、列表和 PDF 排版硬换行，作为后续优化版本的基线。

In [ ]:
import re
from langchain_core.documents import Document

def simple_clean_document(doc: Document) -> Document:
    text = re.sub(r"\s+", " ", doc.page_content).strip()
    metadata = dict(doc.metadata)
    metadata["page_number"] = metadata.get("page", 0) + 1
    metadata["preprocess_version"] = "simple"
    return Document(page_content=text, metadata=metadata)

simple_cleaned_pages = [
    simple_clean_document(page) for page in pdf_pages if page.page_content.strip()
]
print(simple_cleaned_pages[20].page_content[:500])

**预期输出：**

```text
7 可靠性设计 容错能力 ZXVMAX-S通过对关键软件资源的定时检测、实时任务监控、存储保护、数据校验、操作日志 信息保存等手段，可有效地防止小软件故障对系统所造成的冲击，提高了软件系统的容错能力 （即软件错误情况下的自愈能力）。 故障监视及处理 ZXVMAX-S具备自动检测与诊断系统软硬件故障的功能，可对故障硬件实施自动隔离、倒换、 重新启动、重新加载等处理。 满足漏洞扫描 为了避免存在安全漏洞使在网设备受外部攻击，需要使系统满足主流安全漏洞扫描工具的要求。 ZXVMAX-S满足Nessus和CD工具安全漏洞扫描的要求。 7.3 数据可靠性 ZXVMAX-S产品在不同分层架构下的数据都可以确保数据可靠和可恢复。  采集层 采集层运行环境部署采用冗余磁盘阵列设计，在服务器损坏、单一硬盘故障等场景下，通过 替换故障硬件后即可工作，数据不会丢失。同时，系统支持数据补充采集，确保系统数据连 续性。  存储共享层 存储共享层的MPP数据库，根据需要都设置了多个冗余备份，单一节点的损坏不会影响整系 统的运行，也不会导致数据丢失。在替换损坏硬件后，系统能自动进行数据冗余备份的补全 和负载均衡
```

### 1.3 文档分割

简单版本直接按固定字符长度切分，不感知章节边界。

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

simple_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50,
)
simple_split_docs = simple_text_splitter.split_documents(simple_cleaned_pages)
for index, chunk in enumerate(simple_split_docs):
    chunk.metadata["chunk_index"] = index

print(f"简单预处理切分后的文件数量：{len(simple_split_docs)}")

**预期输出：**

```text
简单预处理切分后的文件数量：54
```

## 2. PDF 数据预处理优化

前面的内容演示了 PDF 的基本加载与页面结构。本节在此基础上给出面向 ZXVMAX-S 产品说明书的优化处理流程：先恢复文档结构并清除页眉页脚等噪声，再按章节标题建立语义边界，最后仅对过长章节进行二次切分，同时保留来源、页码、业务类型、章节标题和 chunk 序号等元数据。

### 2.1 数据读取

优化版本读取同一份 PDF，确保后续对比只反映预处理策略的差异。

In [ ]:
optimized_loader = PyMuPDFLoader(
    "../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf"
)
optimized_pdf_pages = optimized_loader.load()
print(f"优化版本读取页数：{len(optimized_pdf_pages)}")

**预期输出：**

```text
优化版本读取页数：29
```

### 2.2 数据清洗

PDF 清洗不是简单地删除换行和空格，而是尽可能恢复文档原有的语义结构。对于本项目的 ZXVMAX-S 产品说明书，需要重点处理：页眉页脚、独立页码、PDF 排版产生的硬换行，以及 `` 等私有区项目符号。  

本节按三个步骤进行：先规范字符，再按规则过滤噪声，最后恢复段落结构。标题、列表、英文术语和必要空格会被保留。

定义可复用的清洗规则

In [ ]:
import re
import unicodedata
from pathlib import Path
from langchain_core.documents import Document

# ZXVMAX-S 文档的固定页脚，例如：SJ-... | 2023-03-30（R1.0）
FOOTER_PATTERN = re.compile(
    r"^SJ-\d{14,}-\d+\s*\|\s*\d{4}-\d{2}-\d{2}（R\d+\.\d+）$"
)
PAGE_NUMBER_PATTERN = re.compile(r"^\d{1,4}$")
HEADING_PATTERN = re.compile(
    r"^(?:第[一二三四五六七八九十百]+章|\d+(?:\.\d+)*)(?:\s+|\u00a0).+"
)
BULLET_PATTERN = re.compile(r"^[\uf06c•▪▫▶●○]\s*")
REPEATED_HEADER_PATTERN = re.compile(
    r"^ZXVMAX-S(?: 多维价值分析系统)? 产品描述（.+业务）$"
)

In [ ]:
def normalize_line(line: str) -> str:
    """统一 Unicode、空白和项目符号，不删除有语义的空格。"""
    # NFC 只统一 Unicode 组合形式，不会把中文标点转换成英文标点
    line = unicodedata.normalize("NFC", line)
    line = line.replace("\u00a0", " ").replace("\u3000", " ")
    line = re.sub(r"[ \t]+", " ", line).strip()
    if BULLET_PATTERN.match(line):
        line = BULLET_PATTERN.sub("- " , line)
    return line


def should_join(previous: str, current: str) -> bool:
    """判断换行是否由 PDF 排版造成；标题、列表和完整句子不合并。"""
    if not previous or not current:
        return False
    if current.startswith("- ") or previous.startswith("- "):
        return False
    if HEADING_PATTERN.match(previous) or HEADING_PATTERN.match(current):
        return False
    # 产品说明书中的无编号小节标题通常较短，如“容错能力”“采集层”
    if len(previous) <= 20 and not previous.endswith(("，", "、", "：", "；")):
        return False
    if previous.endswith(("。", "！", "？", "；", "：", "）", "】")):
        return False
    return True


def clean_pdf_document(doc: Document) -> Document:
    """清洗一个 PDF 页面，同时完整保留来源和页码 metadata。"""
    raw_text = doc.page_content.replace("\r\n", "\n").replace("\r", "\n")
    lines = []
    pending_bullet = False

    for raw_line in raw_text.split("\n"):
        line = normalize_line(raw_line)
        if not line:
            continue
        # PDF 中项目符号经常单独占一行，把它与下一行的列表标题合并
        if line == "- ":
            pending_bullet = True
            continue
        if pending_bullet:
            line = "- " + line
            pending_bullet = False
        if FOOTER_PATTERN.match(line) or PAGE_NUMBER_PATTERN.match(line):
            continue
        if REPEATED_HEADER_PATTERN.match(line):
            continue
        lines.append(line)

    paragraphs = []
    for line in lines:
        if paragraphs and should_join(paragraphs[-1], line):
            paragraphs[-1] += line
        else:
            paragraphs.append(line)

    metadata = dict(doc.metadata)
    metadata["source_name"] = Path(metadata.get("source", "")).name
    metadata["page_number"] = metadata.get("page", 0) + 1
    metadata["cleaned"] = True

    return Document(
        page_content="\n".join(paragraphs).strip(),
        metadata=metadata,
    )


# 对优化版本的全部页面清洗；保留原始页面方便做前后对照
cleaned_pdf_pages = [
    clean_pdf_document(page)
    for page in optimized_pdf_pages
    if page.page_content.strip()
]
cleaned_pdf_page = cleaned_pdf_pages[20]

# 清洗前后对照与基础质量检查
print("===== 清洗前（前 500 字）=====")
print(optimized_pdf_pages[20].page_content[:500])
print("\n===== 清洗后（前 500 字）=====")
print(cleaned_pdf_page.page_content[:500])
print("\n===== 质量统计 =====")
print("原始页数：", len(optimized_pdf_pages))
print("清洗后有效页数：", len(cleaned_pdf_pages))
print("原始字符数：", sum(len(page.page_content) for page in optimized_pdf_pages))
print("清洗后字符数：", sum(len(page.page_content) for page in cleaned_pdf_pages))
print("残留私有项目符号页数：", sum("\uf06c" in page.page_content for page in cleaned_pdf_pages))
print("当前页元数据：", cleaned_pdf_page.metadata)

**预期输出：**

```text
===== 清洗前（前 500 字）=====
7 可靠性设计
容错能力
ZXVMAX-S通过对关键软件资源的定时检测、实时任务监控、存储保护、数据校验、操作日志
信息保存等手段，可有效地防止小软件故障对系统所造成的冲击，提高了软件系统的容错能力
（即软件错误情况下的自愈能力）。
故障监视及处理
ZXVMAX-S具备自动检测与诊断系统软硬件故障的功能，可对故障硬件实施自动隔离、倒换、
重新启动、重新加载等处理。
满足漏洞扫描
为了避免存在安全漏洞使在网设备受外部攻击，需要使系统满足主流安全漏洞扫描工具的要求。
ZXVMAX-S满足Nessus和CD工具安全漏洞扫描的要求。
7.3 数据可靠性
ZXVMAX-S产品在不同分层架构下的数据都可以确保数据可靠和可恢复。

采集层
采集层运行环境部署采用冗余磁盘阵列设计，在服务器损坏、单一硬盘故障等场景下，通过
替换故障硬件后即可工作，数据不会丢失。同时，系统支持数据补充采集，确保系统数据连
续性。

存储共享层
存储共享层的MPP数据库，根据需要都设置了多个冗余备份，单一节点的损坏不会影响整系
统的运行，也不会导致数据丢失。在替换损坏硬件后，系统能自动进行数据冗余备份的补全
和负载均衡

===== 清洗后（前 500 字）=====
7 可靠性设计
容错能力
ZXVMAX-S通过对关键软件资源的定时检测、实时任务监控、存储保护、数据校验、操作日志信息保存等手段，可有效地防止小软件故障对系统所造成的冲击，提高了软件系统的容错能力（即软件错误情况下的自愈能力）。
故障监视及处理
ZXVMAX-S具备自动检测与诊断系统软硬件故障的功能，可对故障硬件实施自动隔离、倒换、重新启动、重新加载等处理。
满足漏洞扫描
为了避免存在安全漏洞使在网设备受外部攻击，需要使系统满足主流安全漏洞扫描工具的要求。
ZXVMAX-S满足Nessus和CD工具安全漏洞扫描的要求。
7.3 数据可靠性
ZXVMAX-S产品在不同分层架构下的数据都可以确保数据可靠和可恢复。
- 采集层
采集层运行环境部署采用冗余磁盘阵列设计，在服务器损坏、单一硬盘故障等场景下，通过替换故障硬件后即可工作，数据不会丢失。同时，系统支持数据补充采集，确保系统数据连续性。
- 存储共享层
存储共享层的MPP数据库，根据需要都设置了多个冗余备份，单一节点的损坏不会影响整系统的运行，也不会导致数据丢失。在替换损坏硬件后，系统能自动进行数据冗余备份的补全和负载均衡。
存储共享层

===== 质量统计 =====
原始页数： 29
清洗后有效页数： 29
原始字符数： 18486
清洗后字符数： 17041
残留私有项目符号页数： 0
当前页元数据： {'producer': 'Apache FOP Version 2.6', 'creator': 'DITA Open Toolkit', 'creationdate': '2023-05-23T21:45:33+08:00', 'source': '../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf', 'file_path': '../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf', 'total_pages': 29, 'format': 'PDF 1.4', 'title': '目录', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20230523214533+08'00'", 'page': 20, 'source_name': 'ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf', 'page_number': 21, 'cleaned': True}
```

### 2.3 文档分割

由于单个文档的长度往往会超过模型支持的上下文，导致检索得到的知识太长超出模型的处理能力，因此，在构建向量知识库的过程中，我们往往需要对文档进行分割，将单个文档按长度或者按固定的规则分割成若干个 chunk，然后将每个 chunk 转化为词向量，存储到向量数据库中。

在检索时，我们会以 chunk 作为检索的元单位，也就是每一次检索到 k 个 chunk 作为模型可以参考来回答用户问题的知识，这个 k 是我们可以自由设定的。

Langchain 中文本分割器都根据 `chunk_size` (块大小)和 `chunk_overlap` (块与块之间的重叠大小)进行分割。

![image.png](../assets/rag-08.png)

* chunk_size 指每个块包含的字符或 Token （如单词、句子等）的数量

* chunk_overlap 指两个块之间共享的字符数量，用于保持上下文的连贯性，避免分割丢失上下文信息

**Langchain 提供多种文档分割方式，区别在怎么确定块与块之间的边界、块由哪些字符/token组成、以及如何测量块大小**

- RecursiveCharacterTextSplitter(): 按字符串分割文本，递归地尝试按不同的分隔符进行分割文本。
- CharacterTextSplitter(): 按字符来分割文本。
- MarkdownHeaderTextSplitter(): 基于指定的标题来分割markdown 文件。
- TokenTextSplitter(): 按token来分割文本。
- SentenceTransformersTokenTextSplitter(): 按token来分割文本
- Language(): 用于 CPP、Python、Ruby、Markdown 等。
- NLTKTextSplitter(): 使用 NLTK（自然语言工具包）按句子分割文本。
- SpacyTextSplitter(): 使用 Spacy按句子的切割文本。

In [ ]:
''' 
* RecursiveCharacterTextSplitter 递归字符文本分割
RecursiveCharacterTextSplitter 将按不同的字符递归地分割(按照这个优先级["\n\n", "\n", " ", ""])，
    这样就能尽量把所有和语义相关的内容尽可能长时间地保留在同一位置
RecursiveCharacterTextSplitter需要关注的是4个参数：

* separators - 分隔符字符串数组
* chunk_size - 每个文档的字符数量限制
* chunk_overlap - 两份文档重叠区域的长度
* length_function - 长度计算函数
'''
#导入文本分割器
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# 知识库中单段文本长度
CHUNK_SIZE = 512

# 知识库中相邻文本重合长度
OVERLAP_SIZE = 50

In [ ]:
# 第二阶段只负责切分过长章节；第一阶段的章节边界由下面的函数识别
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=OVERLAP_SIZE,
    separators=["\n\n", "\n", "。", "；", "！", "？", "，", ""],
)

SECTION_TITLE_PATTERN = re.compile(
    r"^(?:第[一二三四五六七八九十百]+章(?:\s+.+)?|\d+(?:\.\d+)+(?:\s+.+)|\d+\s+[^0-9].+)"
)
BUSINESS_TYPE_PATTERN = re.compile(r"（([^（）]+业务)）")


def is_section_title(line: str) -> bool:
    # 目录条目通常包含连续点线，不应当作为正文的章节边界
    return not re.search(r"[.．·…]{3,}", line) and bool(SECTION_TITLE_PATTERN.match(line))


def extract_business_type(source_name: str) -> str:
    match = BUSINESS_TYPE_PATTERN.search(source_name)
    return match.group(1) if match else "未知业务"


def build_section_documents(pages):
    """先按文档和页面识别章节；跨页时继承上一页最后识别到的章节。"""
    section_docs = []
    current_section_by_source = {}

    for page in pages:
        metadata = dict(page.metadata)
        source_name = metadata.get("source_name") or Path(metadata.get("source", "")).name
        current_section = current_section_by_source.get(source_name, "未识别章节")
        buffer = []

        def flush_buffer():
            if not buffer:
                return
            block_text = "\n".join(buffer).strip()
            # 页尾可能只有下一章节标题，正文会出现在下一页；避免产生标题空块
            if block_text == current_section:
                buffer.clear()
                return
            block_metadata = dict(metadata)
            block_metadata.update({
                "source_name": source_name,
                "business_type": extract_business_type(source_name),
                "section_title": current_section,
            })
            section_docs.append(Document(
                page_content=block_text,
                metadata=block_metadata,
            ))
            buffer.clear()

        for line in page.page_content.splitlines():
            if is_section_title(line):
                flush_buffer()
                current_section = line
                current_section_by_source[source_name] = current_section
            buffer.append(line)
        flush_buffer()

    return section_docs


def split_documents_by_section(pages):
    """章节优先、长度兜底切分，并为每个来源生成稳定的 chunk_index。"""
    from collections import defaultdict

    section_docs = build_section_documents(pages)
    chunks = text_splitter.split_documents(section_docs)
    source_counters = defaultdict(int)

    for chunk in chunks:
        source_name = chunk.metadata["source_name"]
        chunk.metadata["chunk_index"] = source_counters[source_name]
        source_counters[source_name] += 1

        # 标题也进入 embedding/BM25 文本，提升章节名称相关问题的召回率
        section_title = chunk.metadata["section_title"]
        if section_title != "未识别章节" and not chunk.page_content.startswith(section_title):
            chunk.page_content = f"{section_title}\n{chunk.page_content}"

    return chunks

In [ ]:
# 先按章节标题分段，再对超过 CHUNK_SIZE 的章节做二次切分
split_docs = split_documents_by_section(cleaned_pdf_pages)
print(f"章节感知切分后的文件数量：{len(split_docs)}")
print("metadata 示例：", {
    key: split_docs[0].metadata.get(key)
    for key in ["source_name", "page_number", "business_type", "section_title", "chunk_index"]
})

**预期输出：**

```text
章节感知切分后的文件数量：83
metadata 示例： {'source_name': 'ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf', 'page_number': 1, 'business_type': '上网日志业务', 'section_title': '未识别章节', 'chunk_index': 0}
```

In [ ]:
split_docs[50].page_content

**预期输出：**

```text
'5.1 典型组网应用\nZXVMAX-S典型组网如图5-1所示。\n图5-1 ZXVMAX IPDR典型组网应用1. 探针Prober要求部署在各个被采集单元所在的机房，便于采集设备的部署，减少流量负担。\n2. 防火墙日志处理单元要求在防火墙所在的机房，便于防火墙日志，减少流量负担。\n3. 数据合成单元、数据存储单元、应用服务器单元部署在同一个机房，一般部署在中心机房。'
```

In [ ]:
required_metadata = {"source_name", "page_number", "business_type", "section_title", "chunk_index"}
print(f"切分后的字符数：{sum(len(doc.page_content) for doc in split_docs)}")
print("metadata 是否完整：", all(
    required_metadata.issubset(doc.metadata) for doc in split_docs
))
print("误识别为标题的目录点线数：", sum(
    bool(re.search(r"[.．·…]{3,}", doc.metadata["section_title"]))
    for doc in split_docs
))

**预期输出：**

```text
切分后的字符数：17172
metadata 是否完整： True
误识别为标题的目录点线数： 0
```

注：文档分割往往决定检索系统的下限。本节采用两阶段策略：先按编号章节标题恢复语义边界，再用 `chunk_size` 和 `chunk_overlap` 对过长章节做二次切分。目录中的连续点线会被排除，跨页正文会继承上一页的章节标题。每个 chunk 同时保存来源、页码、业务类型、章节标题和文档内序号，便于过滤检索、引用溯源和问题定位。

## 3. PDF 转 Markdown 预处理

这一部分增加第三条数据预处理路线：`PDF → Markdown → 标题切分 → 过长章节二次切分`。它不会替换现有优化版，而是作为可量化对照实验。

### 3.1 版面感知的 PDF 转 Markdown

对应： `C02_数据准备/01_数据加载.md` 中的“文档加载器”和“结构化解析”。这里不只抽取纯文本，还利用字体大小、字体名称和页面坐标恢复标题、列表，并显式保存页码。

In [ ]:
import fitz

MARKDOWN_PDF_PATH = (
    "../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf"
)

def _join_pdf_wrapped_lines(lines):
    """合并 PDF 排版硬换行，保留句末形成的自然段。"""
    paragraphs = []
    buffer = ""
    for line in lines:
        buffer += line
        if line.endswith(("。", "！", "？", "；", "：")):
            paragraphs.append(buffer)
            buffer = ""
    if buffer:
        paragraphs.append(buffer)
    return paragraphs

def pdf_page_to_markdown(page, page_number):
    """根据字体与位置把一个 PDF 页面恢复成带页码标记的 Markdown。"""
    records = []
    for block in page.get_text("dict")["blocks"]:
        for line in block.get("lines", []):
            spans = line.get("spans", [])
            text = "".join(span["text"] for span in spans).replace("\u00a0", " " ).strip()
            if not text:
                continue
            y0 = line["bbox"][1]
            if y0 < 75 or y0 > page.rect.height - 80:
                continue
            records.append({
                "text": text,
                "size": max(span["size"] for span in spans),
                "font": " ".join(span["font"] for span in spans),
                "y": y0,
                "x": line["bbox"][0],
            })
    # 同一视觉行的项目符号与文字可能有 1-2pt 基线差，按 4pt 分组后再按 x 排序
    records.sort(key=lambda item: (round(item["y"] / 4), item["x"]))

    markdown_lines = [f"<!-- page_number: {page_number} -->", ""]
    body_lines = []
    pending_bullet = False

    def flush_body():
        if body_lines:
            markdown_lines.extend(_join_pdf_wrapped_lines(body_lines))
            markdown_lines.append("")
            body_lines.clear()

    for record in records:
        text = record["text"]
        if text in {"\uf06c", "•", "●", "▪"}:
            flush_body()
            pending_bullet = True
            continue
        if pending_bullet:
            flush_body()
            markdown_lines.extend([f"- **{text}**", ""])
            pending_bullet = False
            continue
        if record["size"] >= 15 and not re.search(r"[.．·…]{3,}", text):
            flush_body()
            level = 2 if re.match(r"^\d+(?:\.\d+)+\s+", text) else 1
            markdown_lines.extend([f"{'#' * level} {text}", ""])
        elif (
            record["size"] >= 11
            and "ZHONGCU" in record["font"].upper()
            and not re.search(r"[.．·…]{3,}", text)
        ):
            flush_body()
            markdown_lines.extend([f"### {text}", ""])
        else:
            body_lines.append(text)
    flush_body()
    return "\n".join(markdown_lines).strip()

def pdf_to_markdown_documents(pdf_path):
    pdf = fitz.open(pdf_path)
    source_name = Path(pdf_path).name
    docs = []
    for page_index, page in enumerate(pdf):
        docs.append(Document(
            page_content=pdf_page_to_markdown(page, page_index + 1),
            metadata={
                "source": pdf_path,
                "source_name": source_name,
                "page_number": page_index + 1,
                "business_type": extract_business_type(source_name),
                "content_format": "markdown",
            },
        ))
    return docs

markdown_pages = pdf_to_markdown_documents(MARKDOWN_PDF_PATH)
print(markdown_pages[20].page_content[:1500])

**预期输出：**

```text
<!-- page_number: 21 -->

### 容错能力

ZXVMAX-S通过对关键软件资源的定时检测、实时任务监控、存储保护、数据校验、操作日志信息保存等手段，可有效地防止小软件故障对系统所造成的冲击，提高了软件系统的容错能力（即软件错误情况下的自愈能力）。

### 故障监视及处理

ZXVMAX-S具备自动检测与诊断系统软硬件故障的功能，可对故障硬件实施自动隔离、倒换、重新启动、重新加载等处理。

### 满足漏洞扫描

为了避免存在安全漏洞使在网设备受外部攻击，需要使系统满足主流安全漏洞扫描工具的要求。
ZXVMAX-S满足Nessus和CD工具安全漏洞扫描的要求。

## 7.3 数据可靠性

ZXVMAX-S产品在不同分层架构下的数据都可以确保数据可靠和可恢复。

- **采集层**

采集层运行环境部署采用冗余磁盘阵列设计，在服务器损坏、单一硬盘故障等场景下，通过替换故障硬件后即可工作，数据不会丢失。同时，系统支持数据补充采集，确保系统数据连续性。

- **存储共享层**

存储共享层的MPP数据库，根据需要都设置了多个冗余备份，单一节点的损坏不会影响整系统的运行，也不会导致数据丢失。在替换损坏硬件后，系统能自动进行数据冗余备份的补全和负载均衡。
存储共享层的MPP数据库，设计运行在冗余磁盘阵列上，并且是具备双控制器的外置磁阵，在确保性能的同时避免了磁阵数据丢失。

- **应用分析层**

应用分析层操作系统和软件模块都部署在冗余磁盘阵列上，系统软件、配置数据等信息在单一磁盘损坏，主机故障等情形下都不会丢失。

## 7.4 组网可靠性

ZXVMAX-S产品根据需要，可以通过交换机冗余备份，各主机网络的以太网隧道冗余备份，实现对业务和设备的保护。
在单一交换机故障时，备用交换机会及时启用，避免业务中断。
在单一的网络传输故障时，备用的传输链路仍然生效，也能避免业务中断。
```

In [ ]:
from pathlib import Path


# ===============================
# 保存 Markdown
# ===============================

OUTPUT_DIR = Path("../knowledge_md/VMAX-S")

# 自动创建目录
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


pdf_path = Path(MARKDOWN_PDF_PATH)

md_path = (
    OUTPUT_DIR /
    pdf_path.with_suffix(".md").name
)


with open(
    md_path,
    "w",
    encoding="utf-8"
) as f:

    for doc in markdown_pages:

        f.write(
            doc.page_content
        )

        f.write(
            "\n\n"
        )


print(
    "Markdown保存完成:"
)

print(
    md_path
)

**预期输出：**

```text
Markdown保存完成:
../knowledge_md/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.md
```

### 3.2 按 Markdown 标题建立结构边界

**对应打基础：** `C02_数据准备/02_数据切块.md` 中的“基于文档结构的分块”。先用 `MarkdownHeaderTextSplitter` 按 `# / ## / ###` 切分，再对超过长度限制的章节做二次递归切分。

In [ ]:
from collections import defaultdict
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "chapter"),
        ("##", "section"),
        ("###", "subsection"),
    ],
    strip_headers=False,
)

markdown_section_docs = []
for page_doc in markdown_pages:
    for section_doc in markdown_header_splitter.split_text(page_doc.page_content):
        section_doc.metadata = {**page_doc.metadata, **section_doc.metadata}
        section_doc.metadata["section_title"] = (
            section_doc.metadata.get("subsection")
            or section_doc.metadata.get("section")
            or section_doc.metadata.get("chapter")
            or "未识别章节"
        )
        markdown_section_docs.append(section_doc)

markdown_split_docs = text_splitter.split_documents(markdown_section_docs)
markdown_chunk_counters = defaultdict(int)
for chunk in markdown_split_docs:
    source_name = chunk.metadata["source_name"]
    chunk.metadata["chunk_index"] = markdown_chunk_counters[source_name]
    markdown_chunk_counters[source_name] += 1

print(f"Markdown 页面数：{len(markdown_pages)}")
print(f"标题分段数：{len(markdown_section_docs)}")
print(f"二次切分后数据块数：{len(markdown_split_docs)}")

**预期输出：**

```text
Markdown 页面数：29
标题分段数：101
二次切分后数据块数：121
```

### 3.3 Markdown 与 metadata 质量检查

**对应打基础：** `C06_RAG系统评估/01_评估介绍.md` 的分层评估思想。入库前先检查转换质量，而不是等回答错误后再排查。

In [ ]:
markdown_required_metadata = {
    "source_name", "page_number", "business_type",
    "section_title", "chunk_index", "content_format",
}
assert all(markdown_required_metadata.issubset(doc.metadata) for doc in markdown_split_docs)
assert not any("\uf06c" in doc.page_content for doc in markdown_split_docs)
assert any(doc.metadata["section_title"] == "7.3 数据可靠性" for doc in markdown_split_docs)
assert not any(
    re.search(r"[.．·…]{3,}", doc.metadata["section_title"])
    for doc in markdown_split_docs
)
print("Markdown 预处理质量检查通过")
print("识别到的章节示例：", sorted({
    doc.metadata["section_title"] for doc in markdown_split_docs
    if doc.metadata["section_title"] != "未识别章节"
})[:12])

**预期输出：**

```text
Markdown 预处理质量检查通过
识别到的章节示例： ['1 产品定位与特点', '1.1 产品背景', '1.2 产品定位', '1.3 产品特点', '10 遵循标准与要求', '10.1 运维标准', '10.2 安全标准', '2 产品结构', '2.1 组网结构', '2.3 硬件结构', '2.4 探针', '3 操作维护系统']
```

## 4. Unstructured 文档元素预处理

### 4.1 解析并检查文档元素类型

**对应打基础：** `C02_数据准备/01_数据加载.md` 中的“Unstructured 文档处理库”和“支持的文档元素类型”。Unstructured 提供统一的 `Title / NarrativeText / ListItem / Table / Header / Footer` 元素模型。这里先统计类型，再决定如何切分。

当前中文 DITA PDF 在 `fast` 模式下会把较多短文本判断为 `Title`，因此本实验不直接使用 `chunk_by_title`，而是让 Unstructured 负责元素与页码抽取，再用可靠的编号标题规则确定章节边界。
>
> **可选依赖**：`unstructured[pdf]` 体积较大（会引入推理相关依赖），本项目默认环境未安装。未安装时下面几个单元会自动跳过，不影响其余章节运行。

In [ ]:
import os
import tempfile
from collections import Counter

# 避免受只读 HOME 缓存目录影响；必须在导入 Unstructured PDF 模块前设置
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "rag-mpl-cache"))
os.environ.setdefault("NUMBA_CACHE_DIR", str(Path(tempfile.gettempdir()) / "rag-numba-cache"))

try:
    from unstructured.partition.pdf import partition_pdf
except ImportError:
    partition_pdf = None

if partition_pdf is None:
    unstructured_elements = []
    print("未安装 Unstructured，本节跳过。需要体验时执行：pip install \"unstructured[pdf]\"")
else:
    unstructured_elements = partition_pdf(
        filename=MARKDOWN_PDF_PATH,
        strategy="fast",
        include_page_breaks=True,
        languages=["eng", "chi_sim"],
    )
    unstructured_type_counts = Counter(
        getattr(element, "category", type(element).__name__)
        for element in unstructured_elements
    )
    print(f"Unstructured 元素数：{len(unstructured_elements)}")
    print("元素类型统计：", dict(unstructured_type_counts))

**预期输出：**

```text
Thank you for using Unstructured Open Source! Try Unstructured's commercial offering for improved output quality. 10,000 free pages to start. Learn more: https://transform.unstructured.io/?utm_source=unstructured_oss Hide this notice: UNSTRUCTURED_DISABLE_NOTICE=1
Unstructured 元素数：741
元素类型统计： {'UncategorizedText': 225, 'Title': 441, 'PageBreak': 29, 'NarrativeText': 18, 'ListItem': 28}
```

### 4.2 元素清洗、章节组织与二次切分

**对应打基础：** `C02_数据准备/02_数据切块.md` 中“基于文档结构的分块”和 Unstructured 的元素级分块思想。保留元素类型和页码，过滤页脚、独立页码与页面分隔符，再按可靠章节标题组织内容。

In [ ]:
def unstructured_elements_to_section_docs(elements, source_path):
    source_name = Path(source_path).name
    section_docs = []
    current_section = "未识别章节"
    current_page = None
    buffer = []
    element_types = set()

    def flush_buffer():
        if not buffer or current_page is None:
            return
        paragraphs = []
        for text in buffer:
            if paragraphs and should_join(paragraphs[-1], text):
                paragraphs[-1] += text
            else:
                paragraphs.append(text)
        section_docs.append(Document(
            page_content="\n".join(paragraphs),
            metadata={
                "source": source_path,
                "source_name": source_name,
                "page_number": current_page,
                "business_type": extract_business_type(source_name),
                "section_title": current_section,
                "content_format": "unstructured",
                "parser_strategy": "fast",
                "element_types": sorted(element_types),
            },
        ))
        buffer.clear()
        element_types.clear()

    for element in elements:
        category = getattr(element, "category", type(element).__name__)
        if category == "PageBreak":
            continue
        metadata = element.metadata.to_dict()
        page_number = metadata.get("page_number")
        if page_number is None:
            continue
        text = normalize_line(str(element))
        text = BULLET_PATTERN.sub("- ", text)
        if not text or FOOTER_PATTERN.match(text) or PAGE_NUMBER_PATTERN.match(text):
            continue
        if REPEATED_HEADER_PATTERN.match(text):
            continue
        if current_page is not None and page_number != current_page:
            flush_buffer()
        current_page = page_number
        if is_section_title(text):
            flush_buffer()
            current_section = text
        buffer.append(text)
        element_types.add(category)
    flush_buffer()
    return section_docs

if unstructured_elements:
    unstructured_section_docs = unstructured_elements_to_section_docs(
        unstructured_elements, MARKDOWN_PDF_PATH
    )
    unstructured_split_docs = text_splitter.split_documents(unstructured_section_docs)
    unstructured_counters = defaultdict(int)
    for chunk in unstructured_split_docs:
        source_name = chunk.metadata["source_name"]
        chunk.metadata["chunk_index"] = unstructured_counters[source_name]
        unstructured_counters[source_name] += 1

    print(f"Unstructured 章节块数：{len(unstructured_section_docs)}")
    print(f"二次切分后数据块数：{len(unstructured_split_docs)}")
else:
    unstructured_section_docs = []
    unstructured_split_docs = []
    print("Unstructured 未安装，跳过元素清洗与二次切分。")

**预期输出：**

```text
Unstructured 章节块数：75
二次切分后数据块数：88
```

### 4.3 Unstructured 结果质量检查

对应：`C06_RAG系统评估/01_评估介绍.md`。检查元素类型、章节、页码和噪声残留，并把解析器策略写入 metadata，便于后续比较不同加载器。

In [ ]:
if unstructured_split_docs:
    unstructured_required_metadata = {
        "source_name", "page_number", "business_type", "section_title",
        "chunk_index", "content_format", "parser_strategy", "element_types",
    }
    assert all(
        unstructured_required_metadata.issubset(doc.metadata)
        for doc in unstructured_split_docs
    )
    assert not any("\uf06c" in doc.page_content for doc in unstructured_split_docs)
    assert any(
        doc.metadata["section_title"] == "7.3 数据可靠性"
        for doc in unstructured_split_docs
    )
    print("Unstructured 预处理质量检查通过")
else:
    print("Unstructured 未安装，跳过质量检查。")

**预期输出：**

```text
Unstructured 预处理质量检查通过
```

## 5. 四种预处理路线对比

下面固定比较同一份文档第 21 页中包含“7.3 数据可靠性”的数据块：简单版本按长度截取，优化版本用正则恢复章节，Markdown 版本根据 PDF 版面恢复结构，Unstructured 版本使用统一文档元素模型并保留元素类型。

In [ ]:
COMPARE_PAGE_NUMBER = 21
COMPARE_SECTION = "7.3 数据可靠性"

simple_example = next(
    doc for doc in simple_split_docs
    if doc.metadata.get("page_number") == COMPARE_PAGE_NUMBER
    and "数据可靠性" in doc.page_content
)
optimized_example = next(
    doc for doc in split_docs
    if doc.metadata.get("page_number") == COMPARE_PAGE_NUMBER
    and doc.metadata.get("section_title") == COMPARE_SECTION
)
markdown_example = next(
    doc for doc in markdown_split_docs
    if doc.metadata.get("page_number") == COMPARE_PAGE_NUMBER
    and doc.metadata.get("section_title") == COMPARE_SECTION
)
unstructured_example = None
if unstructured_split_docs:
    unstructured_example = next(
        (doc for doc in unstructured_split_docs
         if doc.metadata.get("page_number") == COMPARE_PAGE_NUMBER
         and doc.metadata.get("section_title") == COMPARE_SECTION),
        None,
    )

print("===== 简单预处理数据块 =====")
print(simple_example.page_content)
print("metadata:", simple_example.metadata)
print("\n===== 优化预处理数据块 =====")
print(optimized_example.page_content)
print("metadata:", {
    key: optimized_example.metadata.get(key)
    for key in ["source_name", "page_number", "business_type", "section_title", "chunk_index"]
})
if unstructured_example is not None:
    print("\n===== Unstructured 数据块 =====")
    print(unstructured_example.page_content)
    print("metadata:", {
        key: unstructured_example.metadata.get(key)
        for key in [
            "source_name", "page_number", "business_type", "section_title",
            "chunk_index", "content_format", "parser_strategy", "element_types",
        ]
    })
print("\n===== PDF 转 Markdown 数据块 =====")
print(markdown_example.page_content)
print("metadata:", {
    key: markdown_example.metadata.get(key)
    for key in [
        "source_name", "page_number", "business_type",
        "section_title", "chunk_index", "content_format",
    ]
})
print("\n===== 对比结论 =====")
print("简单版本字符数：", len(simple_example.page_content))
print("优化版本字符数：", len(optimized_example.page_content))
print("Markdown 版本字符数：", len(markdown_example.page_content))
print("优化版本章节标题：", optimized_example.metadata["section_title"])
print("Markdown 版本章节标题：", markdown_example.metadata["section_title"])
print("Markdown 版本保留列表：", "- **采集层**" in markdown_example.page_content)
if unstructured_example is not None:
    print("Unstructured 版本字符数：", len(unstructured_example.page_content))
    print("Unstructured 元素类型：", unstructured_example.metadata["element_types"])
else:
    print("Unstructured 未安装，本节仅对比其余三种预处理路线。")

**预期输出：**

```text
===== 简单预处理数据块 =====
7 可靠性设计 容错能力 ZXVMAX-S通过对关键软件资源的定时检测、实时任务监控、存储保护、数据校验、操作日志 信息保存等手段，可有效地防止小软件故障对系统所造成的冲击，提高了软件系统的容错能力 （即软件错误情况下的自愈能力）。 故障监视及处理 ZXVMAX-S具备自动检测与诊断系统软硬件故障的功能，可对故障硬件实施自动隔离、倒换、 重新启动、重新加载等处理。 满足漏洞扫描 为了避免存在安全漏洞使在网设备受外部攻击，需要使系统满足主流安全漏洞扫描工具的要求。 ZXVMAX-S满足Nessus和CD工具安全漏洞扫描的要求。 7.3 数据可靠性 ZXVMAX-S产品在不同分层架构下的数据都可以确保数据可靠和可恢复。  采集层 采集层运行环境部署采用冗余磁盘阵列设计，在服务器损坏、单一硬盘故障等场景下，通过 替换故障硬件后即可工作，数据不会丢失。同时，系统支持数据补充采集，确保系统数据连 续性。  存储共享层 存储共享层的MPP数据库，根据需要都设置了多个冗余备份，单一节点的损坏不会影响整系 统的运行，也不会导致数据丢失。在替换损坏硬件后，系统能自动进行数据冗余备份的补全 和负载均衡。
metadata: {'producer': 'Apache FOP Version 2.6', 'creator': 'DITA Open Toolkit', 'creationdate': '2023-05-23T21:45:33+08:00', 'source': '../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf', 'file_path': '../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf', 'total_pages': 29, 'format': 'PDF 1.4', 'title': '目录', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20230523214533+08'00'", 'page': 20, 'page_number': 21, 'preprocess_version': 'simple', 'chunk_index': 40}

===== 优化预处理数据块 =====
7.3 数据可靠性
ZXVMAX-S产品在不同分层架构下的数据都可以确保数据可靠和可恢复。
- 采集层
采集层运行环境部署采用冗余磁盘阵列设计，在服务器损坏、单一硬盘故障等场景下，通过替换故障硬件后即可工作，数据不会丢失。同时，系统支持数据补充采集，确保系统数据连续性。
- 存储共享层
存储共享层的MPP数据库，根据需要都设置了多个冗余备份，单一节点的损坏不会影响整系统的运行，也不会导致数据丢失。在替换损坏硬件后，系统能自动进行数据冗余备份的补全和负载均衡。
存储共享层的MPP数据库，设计运行在冗余磁盘阵列上，并且是具备双控制器的外置磁阵，在确保性能的同时避免了磁阵数据丢失。
- 应用分析层
应用分析层操作系统和软件模块都部署在冗余磁盘阵列上，系统软件、配置数据等信息在单一磁盘损坏，主机故障等情形下都不会丢失。
metadata: {'source_name': 'ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf', 'page_number': 21, 'business_type': '上网日志业务', 'section_title': '7.3 数据可靠性', 'chunk_index': 57}

===== Unstructured 数据块 =====
7.3 数据可靠性
ZXVMAX-S产品在不同分层架构下的数据都可以确保数据可靠和可恢复。
- 采集层
采集层运行环境部署采用冗余磁盘阵列设计，在服务器损坏、单一硬盘故障等场景下，通过替换故障硬件后即可工作，数据不会丢失。同时，系统支持数据补充采集，确保系统数据连续性。
- 存储共享层
存储共享层的MPP数据库，根据需要都设置了多个冗余备份，单一节点的损坏不会影响整系统的运行，也不会导致数据丢失。在替换损坏硬件后，系统能自动进行数据冗余备份的补全和负载均衡。
存储共享层的MPP数据库，设计运行在冗余磁盘阵列上，并且是具备双控制器的外置磁阵，在确保性能的同时避免了磁阵数据丢失。
- 应用分析层
应用分析层操作系统和软件模块都部署在冗余磁盘阵列上，系统软件、配置数据等信息在单一磁盘损坏，主机故障等情形下都不会丢失。
metadata: {'source_name': 'ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf', 'page_number': 21, 'business_type': '上网日志业务', 'section_title': '7.3 数据可靠性', 'chunk_index': 61, 'content_format': 'unstructured', 'parser_strategy': 'fast', 'element_types': ['NarrativeText', 'Title', 'UncategorizedText']}

===== PDF 转 Markdown 数据块 =====
## 7.3 数据可靠性  
ZXVMAX-S产品在不同分层架构下的数据都可以确保数据可靠和可恢复。  
- **采集层**  
采集层运行环境部署采用冗余磁盘阵列设计，在服务器损坏、单一硬盘故障等场景下，通过替换故障硬件后即可工作，数据不会丢失。同时，系统支持数据补充采集，确保系统数据连续性。  
- **存储共享层**  
存储共享层的MPP数据库，根据需要都设置了多个冗余备份，单一节点的损坏不会影响整系统的运行，也不会导致数据丢失。在替换损坏硬件后，系统能自动进行数据冗余备份的补全和负载均衡。
存储共享层的MPP数据库，设计运行在冗余磁盘阵列上，并且是具备双控制器的外置磁阵，在确保性能的同时避免了磁阵数据丢失。  
- **应用分析层**  
应用分析层操作系统和软件模块都部署在冗余磁盘阵列上，系统软件、配置数据等信息在单一磁盘损坏，主机故障等情形下都不会丢失。
metadata: {'source_name': 'ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf', 'page_number': 21, 'business_type': '上网日志业务', 'section_title': '7.3 数据可靠性', 'chunk_index': 84, 'content_format': 'markdown'}

===== 对比结论 =====
简单版本字符数： 501
优化版本字符数： 360
Markdown 版本字符数： 389
优化版本章节标题： 7.3 数据可靠性
Markdown 版本章节标题： 7.3 数据可靠性
Markdo

（本次输出共 3106 个字符，这里摘录前 3000 个；完整内容请在真实环境执行本单元查看。）
```

## 6. 对比结论与选型建议

综合当前 ZXVMAX-S 产品说明书的实际输出，四种方案的推荐顺序为：

1. **优化预处理版**：当前默认入库方案；
2. **PDF 转 Markdown 版**：标题、列表和表格结构明显时优先考虑；
3. **Unstructured 版**：适合 PDF、Word、HTML、Excel 等多格式统一处理；
4. **简单预处理版**：只保留为教学 Baseline，不建议用于正式入库。

| 方案 | 主要优点 | 主要问题 | 建议用途 |
|---|---|---|---|
| 简单预处理 | 实现简单、速度快 | 多主题混合，硬换行和私有符号残留，metadata 不完整 | 教学基线 |
| 优化预处理 | 章节纯度高、文本紧凑、速度快、依赖少、规则可控 | 规则针对当前文档格式，需要按新文档类型适配 | 当前默认入库方案 |
| PDF 转 Markdown | 标题层级、列表和表格结构表达最好 | 转换规则依赖字体和版面，Markdown 标记会增加少量文本 | 强结构 PDF、表格和列表较多的文档 |
| Unstructured | 统一元素模型、metadata 丰富、支持多种文件格式 | 依赖较重、速度较慢，`fast` 模式对当前中文 PDF 存在 Title 过度分类 | 多格式企业知识库 |

当前样例中，优化版与 Unstructured 版的正文质量基本一致，但优化版只包含 360 个字符，信息密度更高，也没有 Unstructured 元素误分类问题，因此更适合作为当前项目的默认路线。Markdown 版保留了 `##` 标题和加粗列表，结构表达最好，可作为复杂版面文档的补充路线。

推荐采用按文档类型路由的策略：

```text
DITA 产品说明书、纯文字 PDF → 优化预处理
标题、列表、表格结构明显的 PDF → PDF 转 Markdown
PDF、Word、HTML、Excel 混合知识库 → Unstructured
简单预处理 → 仅用于 Baseline 对比
```

> 当前结论来自同一页、同一章节的数据块质量比较。最终是否替换默认方案，还应使用同一评测集比较 Hit@K、MRR、章节命中率、页码准确率和回答忠实度。

## 7. 批量数据预处理

批量处理文件夹中所有文件

In [ ]:
import os

# 获取folder_path下所有文件路径，储存在file_paths里
file_paths = []
folder_path = '../knowledge_path/VMAX-S'
for root, dirs, files in os.walk(folder_path):
    for file in files:
        file_path = os.path.join(root, file)
        file_paths.append(file_path)
print(file_paths)

from langchain_community.document_loaders import PyMuPDFLoader
# from langchain_community.document_loaders import UnstructuredMarkdownLoader

# 遍历文件路径并把实例化的loader存放在loaders里
loaders = []

for file_path in file_paths:
    file_type = file_path.split('.')[-1]
    if file_type == 'pdf':
        loaders.append(PyMuPDFLoader(file_path))
    else:
        print(f"Unsupported file type: {file_type} for file {file_path}")


# 下载文件并存储到text
# 加载所有文档内容到 texts
texts = []
for loader in loaders:
    texts.extend(loader.load())  # 关键步骤：初始化 texts


# 统一复用前面定义的清洗函数，避免单文档与批量处理规则不一致
raw_texts = texts
texts = [clean_pdf_document(doc) for doc in raw_texts if doc.page_content.strip()]
print(f"原始页面数：{len(raw_texts)}，清洗后有效页面数：{len(texts)}")

**预期输出：**

```text
['../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf', '../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（语音业务）.pdf', '../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（数据业务）.pdf', '../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（端到端业务）.pdf', '../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf']
原始页面数：188，清洗后有效页面数：188
```

In [ ]:
''' 
* RecursiveCharacterTextSplitter 递归字符文本分割
RecursiveCharacterTextSplitter 将按不同的字符递归地分割(按照这个优先级["\n\n", "\n", " ", ""])，
    这样就能尽量把所有和语义相关的内容尽可能长时间地保留在同一位置
RecursiveCharacterTextSplitter需要关注的是4个参数：

* separators - 分隔符字符串数组
* chunk_size - 每个文档的字符数量限制
* chunk_overlap - 两份文档重叠区域的长度
* length_function - 长度计算函数
'''
#导入文本分割器
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# 知识库中单段文本长度
CHUNK_SIZE = 512

# 知识库中相邻文本重合长度
OVERLAP_SIZE = 50

In [ ]:
# 使用递归字符文本分割器
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=OVERLAP_SIZE,
    separators=["\n\n", "\n", "。", "；", "！", "？", "，", ""],
)
# text_splitter.split_text(text.page_content[0:1000])

In [ ]:
split_docs = split_documents_by_section(texts)
print(f"切分后的文件数量：{len(split_docs)}")

**预期输出：**

```text
切分后的文件数量：451
```

In [ ]:
print(f"切分后的字符数（可以用来大致评估 token 数）：{sum([len(doc.page_content) for doc in split_docs])}")

**预期输出：**

```text
切分后的字符数（可以用来大致评估 token 数）：101457
```

In [ ]:
split_docs[19].page_content

**预期输出：**

```text
'1.3 产品特点\n- 使用Gbase数据库或者分布式本地文件系统(HDFS)保存上网日志，支持保存上网日志1年时\n间。\n- 可通过硬采或软采方式采集网元信令，同时接收防火墙NAT日志，关联网元信令和NAT日志\n生成用户上网日志，保存在数据库中。\nà 硬采：通过分光或镜像，采集Gn口或S11口、S1-U口信令。\nà 软采：通过与GGSN/xGW的消息接口，采集网元信令。\n- 支持多种方式查询用户上网日志，包括：\nà 时间范围+公网IP\nà 时间范围+目的IP\nà 时间范围+IMSI\nà 时间范围+MSISDN\nà 时间范围+URL'
```

# 向量数据库的构建

## 1. 安装 Milvus 数据库

Milvus Standalone 是单机服务器部署。Milvus Standalone 的所有组件都打包到一个[Docker 镜像](https://milvus.io/docs/install_standalone-docker.md)中，部署起来非常方便。如果你有生产工作负载，但又不想使用 Kubernetes，那么在内存充足的单机上运行 Milvus Standalone 是一个不错的选择。此外，Milvus Standalone 通过主从复制支持高可用性。

**使用 Docker Compose 运行 Milvus (Linux)**

Milvus 在 Milvus 资源库中提供了 Docker Compose 配置文件。要使用 Docker Compose 安装 Milvus，只需运行

```shell
# Download the configuration file
$ wget https://github.com/milvus-io/milvus/releases/download/v3.0.3/milvus-standalone-docker-compose.yaml -O docker-compose.yaml


# Start Milvus
$ sudo docker compose -f docker-compose.yaml up -d

Creating milvus-etcd  ... done
Creating milvus-minio ... done
Creating milvus-standalone ... done
```

- 如果运行上述命令失败，请检查系统是否安装了 Docker Compose V1。如果是这种情况，建议你根据[本页](https://docs.docker.com/compose/)的说明迁移到 Docker Compose V2。
- 如果您在拉取镜像时遇到任何问题，请通过[community@zilliz.com](mailto:community@zilliz.com)联系我们，并提供有关问题的详细信息，我们将为您提供必要的支持。

启动 Milvus 后、

- 名为milvus-standalone、milvus-minio和milvus-etcd的容器启动。
  - **milvus-etcd**容器不向主机暴露任何端口，并将其数据映射到当前文件夹中的**volumes/etcd**。
  - **milvus-minio**容器使用默认身份验证凭据在本地为端口**9090**和**9091**提供服务，并将其数据映射到当前文件夹中的**volumes/minio**。
  - **Milvus-standalone**容器使用默认设置为本地**19530**端口提供服务，并将其数据映射到当前文件夹中的**volumes/milvus**。

你可以使用以下命令检查容器是否启动并运行：

```shell
$ sudo docker-compose ps

      Name                     Command                  State                            Ports
--------------------------------------------------------------------------------------------------------------------
milvus-etcd         etcd -advertise-client-url ...   Up             2379/tcp, 2380/tcp
milvus-minio        /usr/bin/docker-entrypoint ...   Up (healthy)   9000/tcp
milvus-standalone   /tini -- milvus run standalone   Up             0.0.0.0:19530->19530/tcp, 0.0.0.0:9091->9091/tcp
```

你还可以访问 Milvus WebUI，网址是`http://127.0.0.1:9091/webui/` ，了解有关 Milvus 实例的更多信息。详情请参阅[Milvus WebUI](https://milvus.io/docs/zh/milvus-webui.md)。

**停止和删除 Milvus**

您可以按以下步骤停止和删除此容器

```shell
# Stop Milvus
$ sudo docker compose down

# Delete service data
$ sudo rm -rf volumes
```

**Attu（Milvus 图形用户界面）**

https://github.com/zilliztech/attu

```shell
# Attu v3 通过 MILVUS_ADDRESS 指定 Milvus 地址；容器内访问宿主机用 host.docker.internal
docker run -d --name attu-container \
  -p 9092:3000 \
  -e MILVUS_ADDRESS=http://host.docker.internal:19530 \
  -v attu-data:/data \
  zilliz/attu:v3.0.0
```

启动后访问 `http://localhost:9092`，在登录页填入同一个 Milvus 地址即可，无需账号密码。

> 登录账号和密码不得写入 Notebook；如启用 Attu 鉴权，请通过部署环境变量或密码管理器配置。

**milvus使用指南**

- [数据库](https://milvus.io/docs/zh/manage_databases.md)
- Collections
- Schema 和数据字段
- 插入和删除
- 索引
- 搜索和 Rerankers

> 参考https://milvus.io/docs/zh/manage_databases.md

## 2. 前序数据预处理

In [ ]:
#!/usr/bin/env python
# coding: utf-8

# 批量处理文件夹中所有文件
import os

# 获取folder_path下所有文件路径，储存在file_paths里
file_paths = []
folder_path = '../knowledge_path/VMAX-S'
for root, dirs, files in os.walk(folder_path):
    for file in files:
        file_path = os.path.join(root, file)
        file_paths.append(file_path)
print(file_paths)

from langchain_community.document_loaders import PyMuPDFLoader
# from langchain_community.document_loaders import UnstructuredMarkdownLoader

# 遍历文件路径并把实例化的loader存放在loaders里
loaders = []

for file_path in file_paths:
    file_type = file_path.split('.')[-1]
    if file_type == 'pdf':
        loaders.append(PyMuPDFLoader(file_path))
    else:
        print(f"Unsupported file type: {file_type} for file {file_path}")

# 下载文件并存储到text
# 加载所有文档内容到 texts
texts = []
for loader in loaders:
    texts.extend(loader.load())  # 关键步骤：初始化 texts


# 统一复用前面定义的清洗函数
raw_texts = texts
texts = [clean_pdf_document(doc) for doc in raw_texts if doc.page_content.strip()]
print(f"原始页面数：{len(raw_texts)}，清洗后有效页面数：{len(texts)}")




#导入文本分割器
from langchain_text_splitters import RecursiveCharacterTextSplitter
''' 
* RecursiveCharacterTextSplitter 递归字符文本分割
RecursiveCharacterTextSplitter 将按不同的字符递归地分割(按照这个优先级["\n\n", "\n", " ", ""])，
    这样就能尽量把所有和语义相关的内容尽可能长时间地保留在同一位置
RecursiveCharacterTextSplitter需要关注的是4个参数：

* separators - 分隔符字符串数组
* chunk_size - 每个文档的字符数量限制
* chunk_overlap - 两份文档重叠区域的长度
* length_function - 长度计算函数
'''


# 知识库中单段文本长度
CHUNK_SIZE = 512

# 知识库中相邻文本重合长度
OVERLAP_SIZE = 50


# 使用递归字符文本分割器
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=OVERLAP_SIZE,
    separators=["\n\n", "\n", "。", "；", "！", "？", "，", ""],
)


split_docs = split_documents_by_section(texts)
print(f"切分后的文件数量：{len(split_docs)}")



print(f"切分后的字符数（可以用来大致评估 token 数）：{sum([len(doc.page_content) for doc in split_docs])}")



split_docs[100].page_content



**预期输出：**

```text
['../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf', '../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（语音业务）.pdf', '../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（数据业务）.pdf', '../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（端到端业务）.pdf', '../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf']
原始页面数：188，清洗后有效页面数：188
切分后的文件数量：451
切分后的字符数（可以用来大致评估 token 数）：101457
'1.1 产品背景\n语音业务是通过LTE/NR/WiFi/宽带网络作为业务接入、IMS网络实现业务控制的语音解决方案，对于运维带来的挑战主要体现在：\n- 多域联动，测量困难：语音业务涉及无线、5GC、EPC、IMS、CS多域协同互通，网络评价\n困难。\n- 问题定位困难：涉及网元众多，出现问题或用户投诉后，定位困难，通常要几个部门联动排\n查，没有好的定位手段，容易扯皮，效率低。\n- 语音质量保障困难：语音承载转移到以PS域及IMS域为主，并且需要经常在不同网元和网络\n覆盖区域切换。\n随着VoLTE/EPSFB/VoNR/VoWiFi/VoBB功能的部署，商用的需求推动力越来越大，用户对网络的语音服务质量要求不断提高，语音质量的好坏直接影响用户感知。传统的使用测试软件+MOS盒的评估过程相当繁琐、费时费力、效率低、评估范围有限。因此需要建立一套用于评估VoLTE/EPSFB/VoNR/VoWiFi/VoBB语音质量的指标体系，可以从整体网络统计角度，更灵活更全面地对网络语音服务质量进行评估、分析和优化。'
```

In [ ]:
#!/usr/bin/env python
# coding: utf-8


import os
from pathlib import Path

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter



# =====================================
# 目录配置
# =====================================

INPUT_DIR = Path("../knowledge_path/VMAX-S")

OUTPUT_DIR = Path("../knowledge_md/VMAX-S")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)



# =====================================
# 获取PDF文件
# =====================================

file_paths = []

for root, dirs, files in os.walk(INPUT_DIR):

    for file in files:

        if file.lower().endswith(".pdf"):

            file_paths.append(
                Path(root) / file
            )


print("PDF文件数量:", len(file_paths))



# =====================================
# PDF加载
# =====================================

all_documents = []


for pdf_file in file_paths:

    print("\n加载:", pdf_file.name)


    loader = PyMuPDFLoader(
        str(pdf_file)
    )


    docs = loader.load()


    # 添加来源信息
    for doc in docs:

        doc.metadata["source_file"] = pdf_file.name


    all_documents.extend(docs)



print(
    "PDF页面数量:",
    len(all_documents)
)



# =====================================
# 文本清洗
# =====================================

clean_docs = []


for doc in all_documents:


    text = doc.page_content


    if not text.strip():
        continue


    # 去除空行
    text = "\n".join(
        line.strip()
        for line in text.splitlines()
        if line.strip()
    )


    doc.page_content = text


    clean_docs.append(doc)



print(
    "清洗后页面:",
    len(clean_docs)
)



# =====================================
# 文本切分
# =====================================

text_splitter = RecursiveCharacterTextSplitter(

    chunk_size=800,

    chunk_overlap=100,


    separators=[

        "\n\n",

        "\n",

        "。",

        "！",

        "？",

        "；",

        "，",

        ""

    ]
)



split_docs = text_splitter.split_documents(
    clean_docs
)



print(
    "切分数量:",
    len(split_docs)
)



# =====================================
# 按PDF名称保存MD
# =====================================

from collections import defaultdict


pdf_groups = defaultdict(list)



for doc in split_docs:

    filename = doc.metadata["source_file"]

    pdf_groups[filename].append(doc)



# 一个PDF生成一个MD

for pdf_name, docs in pdf_groups.items():


    md_name = Path(pdf_name).stem + ".md"


    md_path = OUTPUT_DIR / md_name


    print(
        "生成:",
        md_path
    )


    with open(
        md_path,
        "w",
        encoding="utf-8"
    ) as f:


        f.write(
            f"# {Path(pdf_name).stem}\n\n"
        )


        for i, doc in enumerate(docs):


            f.write(
                f"## Chunk {i+1}\n\n"
            )


            f.write(
                doc.page_content
            )


            f.write(
                "\n\n"
            )



print("\n===== 全部完成 =====")

**预期输出：**

```text
PDF文件数量: 5

加载: ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf

加载: ZXVMAX-S（V6.23）产品描述（语音业务）.pdf

加载: ZXVMAX-S（V6.23）产品描述（数据业务）.pdf

加载:
 ZXVMAX-S（V6.23）产品描述（端到端业务）.pdf

加载: ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf
PDF页面数量: 188
清洗后页面: 188
切分数量: 245
生成: ../knowledge_md/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.md
生成: ../knowledge_md/VMAX-S/ZXVMAX-S（V6.23）产品描述（语音业务）.md
生成: ../knowledge_md/VMAX-S/ZXVMAX-S（V6.23）产品描述（数据业务）.md
生成: ../knowledge_md/VMAX-S/ZXVMAX-S（V6.23）产品描述（端到端业务）.md
生成: ../knowledge_md/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.md

===== 全部完成 =====
```

## 3. Embedding 模型准备

In [ ]:
import sys
print(sys.executable)

import langchain_core
print(langchain_core.__version__)

import langchain_core.messages as m
print(m.__file__)

**预期输出：**

```text
~/env_rag_uv/.venv/bin/python
1.7.0
~/env_rag_uv/.venv/lib/python3.12/site-packages/langchain_core/messages/__init__.py
```

（输出里的 `~` 代表运行机器上的用户主目录，每台机器不同。）

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from langchain_openai import OpenAIEmbeddings

# 初始化嵌入模型
my_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)


## 4. 构建 Milvus 向量库：简单检索版本


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from pymilvus import MilvusClient
from langchain_milvus import Milvus

# Milvus 连接参数。Quickstart 中使用 MilvusClient(uri=...) 连接数据库；这里连接远端 Milvus Standalone。
connection_args = {
    "uri": os.environ["MILVUS_URI"],
}

_milvus_client = MilvusClient(**connection_args)

# 简单检索集合只包含一个 dense 向量字段，相当于 Milvus Quickstart 中的单 vector 字段语义检索。
simple_collection_name = "ZXVMAXS"

# Dense：HNSW + COSINE —— 利用向量语义进行相似性检索。
dense_index_param = {
    "index_type": "HNSW",
    "metric_type": "COSINE",
    "params": {
        "M": 16,
        "efConstruction": 200,
    },
}

batch_size = 30

try:
    total_batches = (len(split_docs) + batch_size - 1) // batch_size
    vectordb = None

    for batch_num in range(total_batches):
        start_idx = batch_num * batch_size
        end_idx = min((batch_num + 1) * batch_size, len(split_docs))
        batch_docs = split_docs[start_idx:end_idx]

        print(f"正在处理第 {batch_num + 1}/{total_batches} 批文档 (文档 {start_idx}-{end_idx - 1})")

        if batch_num == 0:
            vectordb = Milvus.from_documents(
                documents=batch_docs,
                embedding=my_emb,
                collection_name=simple_collection_name,
                connection_args=connection_args,
                vector_field="dense",
                text_field="text",
                index_params=dense_index_param,
                drop_old=True,
            )
        else:
            vectordb.add_documents(batch_docs)

        print(f"第 {batch_num + 1} 批文档已成功导入 Milvus 简单检索集合")

    # 搜索时只需要返回文本和元数据字段，不需要返回 dense 向量原始值。
    vectordb.fields = [field for field in vectordb.fields if field not in {"dense", "sparse", "vector"}]
    print(f"所有文档已导入简单检索集合：{simple_collection_name}")
    print(f"向量字段：{vectordb.vector_fields}")
    print(f"返回字段：{vectordb.fields}")

except Exception as e:
    print(f"构建 Milvus 简单检索集合时发生错误: {e}")


**预期输出：**

```text
正在处理第 1/9 批文档 (文档 0-29)
第 1 批文档已成功导入 Milvus 简单检索集合
正在处理第 2/9 批文档 (文档 30-59)
第 2 批文档已成功导入 Milvus 简单检索集合
正在处理第 3/9 批文档 (文档 60-89)
第 3 批文档已成功导入 Milvus 简单检索集合
正在处理第 4/9 批文档 (文档 90-119)
第 4 批文档已成功导入 Milvus 简单检索集合
正在处理第 5/9 批文档 (文档 120-149)
第 5 批文档已成功导入 Milvus 简单检索集合
正在处理第 6/9 批文档 (文档 150-179)
第 6 批文档已成功导入 Milvus 简单检索集合
正在处理第 7/9 批文档 (文档 180-209)
第 7 批文档已成功导入 Milvus 简单检索集合
正在处理第 8/9 批文档 (文档 210-239)
第 8 批文档已成功导入 Milvus 简单检索集合
正在处理第 9/9 批文档 (文档 240-244)
第 9 批文档已成功导入 Milvus 简单检索集合
所有文档已导入简单检索集合：ZXVMAXS
向量字段：['dense']
返回字段：['text', 'pk', 'producer', 'creator', 'creationdate', 'source', 'file_path', 'total_pages', 'format', 'title', 'author', 'subject', 'keywords', 'moddate', 'trapped', 'modDate', 'creationDate', 'page', 'source_file']
```

## 5. 构建 Milvus 向量库：混合检索版本

参考 [Milvus Quickstart](https://milvus.io/docs/v2.6.x/quickstart.md) 的基本流程：连接 Milvus、创建 collection、写入向量和标量字段、搜索时只返回需要展示的文本与元数据字段。这里在 LangChain 封装上对应为：

- `dense`：由 embedding 模型生成，用于语义相似度检索；
- `sparse`：由 Milvus 服务端 BM25 内置函数根据原始文本生成，用于关键词检索；
- 检索时通过 `ranker_type="weighted"` 或 `ranker_type="rrf"` 融合 dense 与 sparse 两路召回结果；
- `sparse` 是 BM25 函数输出的稀疏向量字段，不能作为 raw data 返回，因此检索前要从 `fields` 中排除 `dense/sparse/vector`。

`BM25BuiltInFunction` 不需要客户端维护语料统计，文本分析、稀疏向量生成和 BM25 评分都在 Milvus 服务端完成。注意：Milvus Lite 暂不支持该内置全文检索能力，需要使用 Milvus Standalone、Distributed 或兼容服务。


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from pymilvus import MilvusClient
from langchain_milvus import Milvus, BM25BuiltInFunction

connection_args = {
    "uri": os.environ["MILVUS_URI"],
}

_milvus_client = MilvusClient(**connection_args)

# 混合检索集合使用独立名称，避免和简单检索集合的单向量字段 schema 冲突。
hybrid_collection_name = "ZXVMAXS_HYBRID"

# Dense：HNSW + COSINE —— 利用向量语义进行相似性检索。
# Sparse：AUTOINDEX + BM25 —— 利用关键词相关性进行检索。
# Hybrid Search —— 将两种检索结果融合，兼顾语义理解和关键词匹配。
dense_index_param = {
    "index_type": "HNSW",
    "metric_type": "COSINE",
    "params": {
        "M": 16,
        "efConstruction": 200,
    },
}
sparse_index_param = {
    "metric_type": "BM25",
    "index_type": "AUTOINDEX",
}

batch_size = 30

try:
    total_batches = (len(split_docs) + batch_size - 1) // batch_size
    vectordb_hybrid = None

    for batch_num in range(total_batches):
        start_idx = batch_num * batch_size
        end_idx = min((batch_num + 1) * batch_size, len(split_docs))
        batch_docs = split_docs[start_idx:end_idx]

        print(f"正在处理第 {batch_num + 1}/{total_batches} 批文档 (文档 {start_idx}-{end_idx - 1})")

        if batch_num == 0:
            vectordb_hybrid = Milvus.from_documents(
                documents=batch_docs,
                embedding=my_emb,
                builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
                vector_field=["dense", "sparse"],
                collection_name=hybrid_collection_name,
                connection_args=connection_args,
                consistency_level="Bounded",
                index_params=[dense_index_param, sparse_index_param],
                drop_old=True,
            )
        else:
            vectordb_hybrid.add_documents(batch_docs)

        print(f"第 {batch_num + 1} 批文档已成功导入 Milvus 混合检索集合")

    # 搜索结果只返回文本和元数据字段；Milvus 不允许返回 BM25 函数输出的 sparse raw data。
    vectordb_hybrid.fields = [
        field for field in vectordb_hybrid.fields
        if field not in {"dense", "sparse", "vector"}
    ]
    print(f"所有文档已导入混合检索集合：{hybrid_collection_name}")
    print(f"向量字段：{vectordb_hybrid.vector_fields}")
    print(f"返回字段：{vectordb_hybrid.fields}")

except Exception as e:
    print(f"构建 Milvus 混合检索集合时发生错误: {e}")


**预期输出：**

```text
正在处理第 1/9 批文档 (文档 0-29)
第 1 批文档已成功导入 Milvus 混合检索集合
正在处理第 2/9 批文档 (文档 30-59)
第 2 批文档已成功导入 Milvus 混合检索集合
正在处理第 3/9 批文档 (文档 60-89)
第 3 批文档已成功导入 Milvus 混合检索集合
正在处理第 4/9 批文档 (文档 90-119)
第 4 批文档已成功导入 Milvus 混合检索集合
正在处理第 5/9 批文档 (文档 120-149)
第 5 批文档已成功导入 Milvus 混合检索集合
正在处理第 6/9 批文档 (文档 150-179)
第 6 批文档已成功导入 Milvus 混合检索集合
正在处理第 7/9 批文档 (文档 180-209)
第 7 批文档已成功导入 Milvus 混合检索集合
正在处理第 8/9 批文档 (文档 210-239)
第 8 批文档已成功导入 Milvus 混合检索集合
正在处理第 9/9 批文档 (文档 240-244)
第 9 批文档已成功导入 Milvus 混合检索集合
所有文档已导入混合检索集合：ZXVMAXS_HYBRID
向量字段：['dense', 'sparse']
返回字段：['text', 'pk', 'producer', 'creator', 'creationdate', 'source', 'file_path', 'total_pages', 'format', 'title', 'author', 'subject', 'keywords', 'moddate', 'trapped', 'modDate', 'creationDate', 'page', 'source_file']
```

下面用同一个 Milvus 集合做一次快速检索验证。`weights` 的顺序与 `vector_field=["dense", "sparse"]` 对应：第一个权重控制语义向量检索，第二个权重控制 BM25 关键词检索。

In [ ]:
query = "ZXVMAX-S上网日志有哪些功能？"

# Milvus 不允许把 sparse/dense 向量字段作为 raw data 返回；混合检索前只保留普通字段。
vectordb_hybrid.fields = [
    field for field in vectordb_hybrid.fields
    if field not in {"dense", "sparse", "vector"}
]

hybrid_docs = vectordb_hybrid.similarity_search(
    query,
    k=5,
    ranker_type="weighted",
    ranker_params={"weights": [0.6, 0.4]},
)

for i, doc in enumerate(hybrid_docs, 1):
    source = doc.metadata.get("source") or doc.metadata.get("file_path") or "未知来源"
    page = doc.metadata.get("page", "-")
    print(f"===== 混合检索结果 {i} | source={source} | page={page} =====")
    print(doc.page_content[:500])
    print()


**预期输出：**

```text
===== 混合检索结果 1 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=13 =====
4 功能
本章包含如下主题：
 上网日志保存
10
 上网日志查询
10
 上网日志批量导入查询
11
 日志管理
11
 账号管理
11
 角色管理
12
 资源监控
12
 告警管理
12
 省级网关对接
12
 拨测结果自动比对功能
12
 NAT日志入库功能
13
 北向接口
13
 上网日志历史查询
13
 云化上网日志XDR查询
13
以下介绍ZXVMAX-S的主要的功能。
4.1 上网日志保存

支持使用Gbase数据库或HDFS保存上网日志。

上网日志保存时间可配置，最短保存7天时间，最长可保存一年时间。

支持自动清理超过保存时间的上网日志。
4.2 上网日志查询
可通过web界面指定查询条件，查询用户上网日志。
支持的查询条件：

时间范围+公网IP

时间范围+目的IP

时间范围+MSISDN

时间范围+IMSI

时间范围+URL

支持组合以上五种基础条件，或在基础条件上增加指定公网端口、目的端口，做更精确的查
询。
10
SJ-20220623151803-017 | 2023-03-30（R1.0）

===== 混合检索结果 2 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=4 =====
1 产品定位与特点
本章包含如下主题：
 产品背景
1
 产品定位
1
 产品特点
1
1.1 产品背景
随着移动通信网络的不断发展，移动智能终端用户数突飞猛进，越来越多的用户开始使用移动终
端上网。为了网络安全考虑，政府要求移动运营商可以追溯用户的上网行为，这就要求移动运营
商能记录下所有用户的上网日志。
1.2 产品定位
ZXVMAX-S系统可以长时间的保存用户的上网日志，并可通过公网IP、目的IP、IMSI、
MSISDN、URL等参数查询用户上网日志，满足运营商追溯用户上网行为的需求。
1.3 产品特点

使用Gbase数据库或者分布式本地文件系统(HDFS)保存上网日志，支持保存上网日志1年时
间。

可通过硬采或软采方式采集网元信令，同时接收防火墙NAT日志，关联网元信令和NAT日志
生成用户上网日志，保存在数据库中。
à 硬采：通过分光或镜像，采集Gn口或S11口、S1-U口信令。
à 软采：通过与GGSN/xGW的消息接口，采集网元信令。

支持多种方式查询用户上网日志，包括：
à 时间范围+公网IP
à 时间范围+目的IP
à 时间范围+IMSI
à 时间

===== 混合检索结果 3 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=16 =====
4 功能
4.11 NAT日志入库功能
ZXVMAX-S支持将NAT日志保存到GP数据库，可通过公网IP或目的IP查询NAT日志。
4.12 北向接口
支持中移规范北向接口，中移集团查询平台下发查询命令，上网日志系统将查询结果以文件方
式上传至查询平台。包括云化核心网的4G/5G上网日志查询、4G/5G控制面用户面XDR查询、
4G/5G软采XDR查询、VoLTE XDR查询、5G消息XDR查询，以及原始码流查询功能。
4.13 上网日志历史查询
当现网使用两个库分别存储2个月数据和6个月数据时，使用历史数据查询操作查询6个月库。
4.14 云化上网日志XDR查询
上网日志系统支持云化上网日志XDR查询，包括云化核心网的4G/5G控制面用户面XDR查询、
4G/5G软采XDR查询、VoLTE XDR查询、5G消息XDR查询，以及原始码流查询功能。
SJ-20220623151803-017 | 2023-03-30（R1.0）
13

===== 混合检索结果 4 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=7 =====
ZXVMAX-S 多维价值分析系统 产品描述（上网日志业务）
图2-3 ZXVMAX逻辑结构图
模块功能

采集层：
à Probe：探针，支持硬采和软采，负责进行标准接口的信令和Packet抓包。
à Firewall log Process：防火墙日志处理单元，负责接收防火墙日志。
à Data Merge Process：数据合成单元，负责关联网元信令和防火墙日志，生成上网日志
数据。

存储共享层：
à 分布式存储集群：采用分布式存储集群，留存上网日志记录。支持MPP数据库Gbase。
à 本地分布式文件系统：支持HDFS。
à Saturn：负责接收和解析数据合成单元发送的上网日志实时流数据，将上网日志记录存入
MPP数据库。

应用分析层：
à 客户端：ZXVMAX-S用户访问接口，通过Windows应用程序、Browser等形式提供用户
界面和操作接口。
à REST接口：以RESTful接口形式提供服务器端的服务访问能力。
à 服务器端：实现ZXVMAX-S系统应用分析功能的一系列服务端逻辑、算法，服务器端各
模块分别对应不同的Feature。
2.3 硬件结构


===== 混合检索结果 5 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（数据业务）.pdf | page=5 =====
ZXVMAX-S 多维价值分析系统 产品描述（数据业务）
1.2 产品定位
ZXVMAX-S系统是是面向用户的端到端的网络运维和运营分析产品，立足于从客户角度去感知
和分析网络和业务信息，通过对海量数据灵活的挖掘和分析，从网元、用户、终端、业务等多个
维度，对业务使用过程中的质量和特征进行全方位的挖掘。ZXVMAX-S支持XDR查询、实时保
障、多维分析、会话跟踪等功能，为移动通讯网络的运维和运营提供全面支撑。
ZXVMAX-S系统为运营商的网络建设、运维优化、客户服务等多个环节提供支撑。主要服务对
象是运营商网优和运维部门，支撑网络运维；具体用户是网优工程师和维护工程师，聚焦客户是
CTO和运维负责人。
ZXVMAX-S系统（黄色部分）在网络中的位置如图1-1所示。
图1-1 ZXVMAX-S系统在网络中的位置
1.3 产品特点

分层架构，可分可合
ZXVMAX-S系统分为数据采集层、数据共享层和应用分析层，可以根据客户的需求，支持某
一层独立部署，也支持多层组合部署。ZXVMAX-S系统架构如图1-2所示。
2
SJ-20230406152852-014 | 2023-03-3
```

In [ ]:
# 也可以使用 RRF（Reciprocal Rank Fusion）融合两路召回结果，适合不想手动调权重的场景。
# Milvus 不允许把 sparse/dense 向量字段作为 raw data 返回；混合检索前只保留普通字段。
vectordb_hybrid.fields = [
    field for field in vectordb_hybrid.fields
    if field not in {"dense", "sparse", "vector"}
]

rrf_docs = vectordb_hybrid.similarity_search(
    query,
    k=5,
    ranker_type="rrf",
    ranker_params={"k": 100},
)

for i, doc in enumerate(rrf_docs, 1):
    source = doc.metadata.get("source") or doc.metadata.get("file_path") or "未知来源"
    page = doc.metadata.get("page", "-")
    print(f"===== RRF混合检索结果 {i} | source={source} | page={page} =====")
    print(doc.page_content[:500])
    print()


**预期输出：**

```text
===== RRF混合检索结果 1 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=13 =====
4 功能
本章包含如下主题：
 上网日志保存
10
 上网日志查询
10
 上网日志批量导入查询
11
 日志管理
11
 账号管理
11
 角色管理
12
 资源监控
12
 告警管理
12
 省级网关对接
12
 拨测结果自动比对功能
12
 NAT日志入库功能
13
 北向接口
13
 上网日志历史查询
13
 云化上网日志XDR查询
13
以下介绍ZXVMAX-S的主要的功能。
4.1 上网日志保存

支持使用Gbase数据库或HDFS保存上网日志。

上网日志保存时间可配置，最短保存7天时间，最长可保存一年时间。

支持自动清理超过保存时间的上网日志。
4.2 上网日志查询
可通过web界面指定查询条件，查询用户上网日志。
支持的查询条件：

时间范围+公网IP

时间范围+目的IP

时间范围+MSISDN

时间范围+IMSI

时间范围+URL

支持组合以上五种基础条件，或在基础条件上增加指定公网端口、目的端口，做更精确的查
询。
10
SJ-20220623151803-017 | 2023-03-30（R1.0）

===== RRF混合检索结果 2 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（数据业务）.pdf | page=5 =====
ZXVMAX-S 多维价值分析系统 产品描述（数据业务）
1.2 产品定位
ZXVMAX-S系统是是面向用户的端到端的网络运维和运营分析产品，立足于从客户角度去感知
和分析网络和业务信息，通过对海量数据灵活的挖掘和分析，从网元、用户、终端、业务等多个
维度，对业务使用过程中的质量和特征进行全方位的挖掘。ZXVMAX-S支持XDR查询、实时保
障、多维分析、会话跟踪等功能，为移动通讯网络的运维和运营提供全面支撑。
ZXVMAX-S系统为运营商的网络建设、运维优化、客户服务等多个环节提供支撑。主要服务对
象是运营商网优和运维部门，支撑网络运维；具体用户是网优工程师和维护工程师，聚焦客户是
CTO和运维负责人。
ZXVMAX-S系统（黄色部分）在网络中的位置如图1-1所示。
图1-1 ZXVMAX-S系统在网络中的位置
1.3 产品特点

分层架构，可分可合
ZXVMAX-S系统分为数据采集层、数据共享层和应用分析层，可以根据客户的需求，支持某
一层独立部署，也支持多层组合部署。ZXVMAX-S系统架构如图1-2所示。
2
SJ-20230406152852-014 | 2023-03-3

===== RRF混合检索结果 3 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=4 =====
1 产品定位与特点
本章包含如下主题：
 产品背景
1
 产品定位
1
 产品特点
1
1.1 产品背景
随着移动通信网络的不断发展，移动智能终端用户数突飞猛进，越来越多的用户开始使用移动终
端上网。为了网络安全考虑，政府要求移动运营商可以追溯用户的上网行为，这就要求移动运营
商能记录下所有用户的上网日志。
1.2 产品定位
ZXVMAX-S系统可以长时间的保存用户的上网日志，并可通过公网IP、目的IP、IMSI、
MSISDN、URL等参数查询用户上网日志，满足运营商追溯用户上网行为的需求。
1.3 产品特点

使用Gbase数据库或者分布式本地文件系统(HDFS)保存上网日志，支持保存上网日志1年时
间。

可通过硬采或软采方式采集网元信令，同时接收防火墙NAT日志，关联网元信令和NAT日志
生成用户上网日志，保存在数据库中。
à 硬采：通过分光或镜像，采集Gn口或S11口、S1-U口信令。
à 软采：通过与GGSN/xGW的消息接口，采集网元信令。

支持多种方式查询用户上网日志，包括：
à 时间范围+公网IP
à 时间范围+目的IP
à 时间范围+IMSI
à 时间

===== RRF混合检索结果 4 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（端到端业务）.pdf | page=6 =====
1 产品定位与特点
图1-2 ZXVMAX-S 端到端系统架构

ZXVMAX-S端到端提供4大主要功能：用户分析、网络分析、专题分析和评估优化。

综合网络分析，无线核心网综合解决方案。
ZXVMAX-S端到端系统支持对无线数据、核心网数据进行关联，支持问题定界，可以更加准
确的定位问题点，确定网络的性能瓶颈，更有针对性的指导优化。

多样化处理模式，实时分析、周期任务与指定任务驱动，满足不同用户的需求。
ZXVMAX-S端到端系统支持对网络性能KPI、端到端质量等进行实时分析，仪表盘呈现，对
超出门限指标进行预警，指导运维人员快速响应解决问题。
ZXVMAX-S端到端系统支持定制周期任务，利用空闲时段完成运算，让用户第一时间了解关
注信息。
ZXVMAX-S端到端系统支持根据用户需求进行数据自定义的查询或分析，满足不同场景下用
户分析解决问题的需求。

信息呈现与问题定位并存，满足不同层次用户的需求。
ZXVMAX-S端到端系统支持事件详细信息的呈现，也支持基于经验算法的问题定位，提升分
析解决问题的效率，满足不同层次用户的需求。

多种应用，自由选择。
ZXVMAX-S

===== RRF混合检索结果 5 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=16 =====
4 功能
4.11 NAT日志入库功能
ZXVMAX-S支持将NAT日志保存到GP数据库，可通过公网IP或目的IP查询NAT日志。
4.12 北向接口
支持中移规范北向接口，中移集团查询平台下发查询命令，上网日志系统将查询结果以文件方
式上传至查询平台。包括云化核心网的4G/5G上网日志查询、4G/5G控制面用户面XDR查询、
4G/5G软采XDR查询、VoLTE XDR查询、5G消息XDR查询，以及原始码流查询功能。
4.13 上网日志历史查询
当现网使用两个库分别存储2个月数据和6个月数据时，使用历史数据查询操作查询6个月库。
4.14 云化上网日志XDR查询
上网日志系统支持云化上网日志XDR查询，包括云化核心网的4G/5G控制面用户面XDR查询、
4G/5G软采XDR查询、VoLTE XDR查询、5G消息XDR查询，以及原始码流查询功能。
SJ-20220623151803-017 | 2023-03-30（R1.0）
13
```

## 6. 连接数据库进行测试


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from pymilvus import MilvusClient

_ = load_dotenv(find_dotenv())

def list_all_collections(uri: str):
    """获取所有 collection 列表。"""
    try:
        client = MilvusClient(uri=uri)
        collections = client.list_collections()
        print("所有Collection列表:")
        for i, coll_name in enumerate(collections, 1):
            print(f"{i}. {coll_name}")
        return collections
    except Exception as e:
        print("获取collection列表失败:", e)
        return []

# 执行
collections = list_all_collections(os.environ["MILVUS_URI"])
print(f"总共找到 {len(collections)} 个collection")

**预期输出：**

```text
所有Collection列表:
1. ZXVMAXS
2. ZXVMAXS_HYBRID
3. rag_lab_demo
总共找到 3 个collection
```

## 7. 简单检索与混合检索结果对比

参考 [Milvus Quickstart](https://milvus.io/docs/v2.6.x/quickstart.md) 中的检索流程：先把文档向量和元数据写入 collection，再用同一个 query 发起 `search` 并查看 Top-K 命中的文本与元数据。这里沿用同一思路，对比前面创建的两个 Milvus collection：

- **简单检索版本 `ZXVMAXS`**：只使用 `dense` 向量字段，主要依赖语义相似度召回；
- **混合检索版本 `ZXVMAXS_HYBRID`**：同时使用 `dense` 语义向量和 `sparse` BM25 关键词向量，再通过 weighted ranker 融合结果。

如果 query 中包含产品名、告警码、功能名等关键词，混合检索通常更容易把关键词精确命中的片段排到前面；如果 query 更偏自然语言语义表达，简单向量检索和混合检索的结果可能会有较多重合。


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from pymilvus import MilvusClient
from langchain_milvus import Milvus, BM25BuiltInFunction
from langchain_openai import OpenAIEmbeddings

my_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)

connection_args = {
    "uri": os.environ["MILVUS_URI"],
}

_milvus_client = MilvusClient(**connection_args)

vectordb = Milvus(
    embedding_function=my_emb,
    collection_name="ZXVMAXS",  # 简单检索集合：只有 dense 向量字段
    connection_args=connection_args,
    vector_field="dense",
    text_field="text",
)

vectordb_hybrid = Milvus(
    embedding_function=my_emb,
    builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
    collection_name="ZXVMAXS_HYBRID",  # 混合检索集合：dense + sparse 两个向量字段
    connection_args=connection_args,
    vector_field=["dense", "sparse"],
    text_field="text",
    consistency_level="Bounded",
)

# Milvus 不允许把向量字段作为 raw data 返回，尤其是 BM25 生成的 sparse 字段。
# 这里显式删除 dense/sparse/vector，比只依赖 vector_fields 更稳。
for vectorstore in [vectordb, vectordb_hybrid]:
    vectorstore.fields = [
        field for field in vectorstore.fields
        if field not in {"dense", "sparse", "vector"}
    ]


In [ ]:
comparison_query = "ZXVMAX-S上网日志有哪些功能？"
k = 5

# Milvus 不允许把 sparse/dense 向量字段作为 raw data 返回；检索前只保留普通字段。
for vectorstore in [vectordb, vectordb_hybrid]:
    vectorstore.fields = [
        field for field in vectorstore.fields
        if field not in {"dense", "sparse", "vector"}
    ]

print("简单检索返回字段：", vectordb.fields)
print("混合检索返回字段：", vectordb_hybrid.fields)

simple_docs = vectordb.similarity_search(comparison_query, k=k)
hybrid_docs = vectordb_hybrid.similarity_search(
    comparison_query,
    k=k,
    ranker_type="weighted",
    ranker_params={"weights": [0.6, 0.4]},  # dense 权重 0.6，BM25 sparse 权重 0.4
)

def doc_key(doc):
    source = str(doc.metadata.get("source") or doc.metadata.get("file_path") or "未知来源").strip()
    page = str(doc.metadata.get("page", "-")).strip()
    return source, page

def print_results(title, docs):
    print(f"\n===== {title} =====")
    displayed = set()
    row = 1
    for doc in docs:
        source = str(doc.metadata.get("source") or doc.metadata.get("file_path") or "未知来源").strip()
        page = str(doc.metadata.get("page", "-")).strip()
        display_key = (source, page)
        if display_key in displayed:
            continue
        displayed.add(display_key)
        preview = doc.page_content[:180].replace("\n", " ")
        print(f"{row}. source={source} | page={page}")
        print(f"   {preview}")
        row += 1

print(f"Query: {comparison_query}")
print_results("简单检索 Top-K（dense semantic search）", simple_docs)
print_results("混合检索 Top-K（dense + BM25 sparse weighted fusion）", hybrid_docs)

simple_keys = {doc_key(doc): i + 1 for i, doc in enumerate(simple_docs)}
hybrid_keys = {doc_key(doc): i + 1 for i, doc in enumerate(hybrid_docs)}
common_keys = set(simple_keys) & set(hybrid_keys)
only_simple = set(simple_keys) - set(hybrid_keys)
only_hybrid = set(hybrid_keys) - set(simple_keys)

print("\n===== Top-K 差异统计 =====")
print(f"共同召回数量：{len(common_keys)}/{k}")
print(f"仅简单检索命中：{len(only_simple)}")
print(f"仅混合检索命中：{len(only_hybrid)}")

if only_hybrid:
    print("\n混合检索独有结果示例：")
    hybrid_doc_by_key = {doc_key(doc): doc for doc in hybrid_docs}
    for key in list(only_hybrid)[:3]:
        doc = hybrid_doc_by_key[key]
        source, page = key
        preview = doc.page_content[:120].replace("\n", " ")
        print(f"- source={source} | page={page} | {preview}")

print("\n对比结论：")
print("- 简单检索只看 dense 向量语义相似度，适合语义改写较多的问题。")
print("- 混合检索同时融合 dense 语义召回和 BM25 关键词召回，更适合包含产品名、模块名、告警码、专有名词的问题。")
print("- 如果共同召回数量较低，说明关键词通道显著改变了排序或召回集合；如果共同召回数量较高，说明该 query 的语义召回已经比较稳定。")


**预期输出：**

```text
简单检索返回字段： ['text', 'pk', 'producer', 'creator', 'creationdate', 'source', 'file_path', 'total_pages', 'format', 'title', 'author', 'subject', 'keywords', 'moddate', 'trapped', 'modDate', 'creationDate', 'page', 'source_file']
混合检索返回字段： ['text', 'pk', 'producer', 'creator', 'creationdate', 'source', 'file_path', 'total_pages', 'format', 'title', 'author', 'subject', 'keywords', 'moddate', 'trapped', 'modDate', 'creationDate', 'page', 'source_file']
Query: ZXVMAX-S上网日志有哪些功能？

===== 简单检索 Top-K（dense semantic search） =====
1. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=13
   4 功能 本章包含如下主题：  上网日志保存 10  上网日志查询 10  上网日志批量导入查询 11  日志管理 11  账号管理 11  角色管理 12  资源监控 12  告警管理 12  省级网关对接 12  拨测结果自动比对功能 12  NAT日志入库功能 13  北向接口 13  上网日志历史查询 13  云化上网日志
2. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=4
   1 产品定位与特点 本章包含如下主题：  产品背景 1  产品定位 1  产品特点 1 1.1 产品背景 随着移动通信网络的不断发展，移动智能终端用户数突飞猛进，越来越多的用户开始使用移动终 端上网。为了网络安全考虑，政府要求移动运营商可以追溯用户的上网行为，这就要求移动运营 商能记录下所有用户的上网日志。 1.2 产品定位 ZXVMAX-S系统可以长
3. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=16
   4 功能 4.11 NAT日志入库功能 ZXVMAX-S支持将NAT日志保存到GP数据库，可通过公网IP或目的IP查询NAT日志。 4.12 北向接口 支持中移规范北向接口，中移集团查询平台下发查询命令，上网日志系统将查询结果以文件方 式上传至查询平台。包括云化核心网的4G/5G上网日志查询、4G/5G控制面用户面XDR查询、 4G/5G软采XDR查询、Vo
4. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=7
   ZXVMAX-S 多维价值分析系统 产品描述（上网日志业务） 图2-3 ZXVMAX逻辑结构图 模块功能  采集层： à Probe：探针，支持硬采和软采，负责进行标准接口的信令和Packet抓包。 à Firewall log Process：防火墙日志处理单元，负责接收防火墙日志。 à Data Merge Process：数据合成单元，负责关联网元信
5. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=0
   ZXVMAX-S 多维价值分析系统 产品描述（上网日志业务） 产品版本：V6.23 中兴通讯股份有限公司 地址：深圳市南山区高新技术产业园科技南路中兴通讯大厦 邮编：518057 电话：0755-26770800 400-8301118 800-8301118（座机） 技术支持网站：http://support.zte.com.cn 电子邮件：800@zte

===== 混合检索 Top-K（dense + BM25 sparse weighted fusion） =====
1. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=13
   4 功能 本章包含如下主题：  上网日志保存 10  上网日志查询 10  上网日志批量导入查询 11  日志管理 11  账号管理 11  角色管理 12  资源监控 12  告警管理 12  省级网关对接 12  拨测结果自动比对功能 12  NAT日志入库功能 13  北向接口 13  上网日志历史查询 13  云化上网日志
2. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=4
   1 产品定位与特点 本章包含如下主题：  产品背景 1  产品定位 1  产品特点 1 1.1 产品背景 随着移动通信网络的不断发展，移动智能终端用户数突飞猛进，越来越多的用户开始使用移动终 端上网。为了网络安全考虑，政府要求移动运营商可以追溯用户的上网行为，这就要求移动运营 商能记录下所有用户的上网日志。 1.2 产品定位 ZXVMAX-S系统可以长
3. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=16
   4 功能 4.11 NAT日志入库功能 ZXVMAX-S支持将NAT日志保存到GP数据库，可通过公网IP或目的IP查询NAT日志。 4.12 北向接口 支持中移规范北向接口，中移集团查询平台下发查询命令，上网日志系统将查询结果以文件方 式上传至查询平台。包括云化核心网的4G/5G上网日志查询、4G/5G控制面用户面XDR查询、 4G/5G软采XDR查询、Vo
4. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=7
   ZXVMAX-S 多维价值分析系统 产品描述（上网日志业务） 图2-3 ZXVMAX逻辑结构图 模块功能  采集层： à Probe：探针，支持硬采和软采，负责进行标准接口的信令和Packet抓包。 à Firewall log Process：防火墙日志处理单元，负责接收防火墙日志。 à Data Merge Process：数据合成单元，负责关联网元信
5. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（数据业务）.pdf | pag

（本次输出共 3621 个字符，这里摘录前 3000 个；完整内容请在真实环境执行本单元查看。）
```

### 7.1 对比结果解读

本次真实环境中，`ZXVMAXS` 和 `ZXVMAXS_HYBRID` 都使用同一批 `split_docs` 构建，因此两者的 Top-K 结果有较高重合是正常现象。查询 `ZXVMAX-S上网日志有哪些功能？` 同时包含产品名和功能类关键词，`dense` 语义检索已经能稳定召回“上网日志业务”相关页面，所以简单检索和混合检索共同命中 4 条。

混合检索相比简单检索额外命中了“端到端业务”文档中的功能概述页，说明 BM25 关键词通道会强化 `ZXVMAX-S`、`功能` 等字面匹配，再与 dense 语义召回结果融合排序。这个结果可以说明两者差异：

- 简单检索更依赖语义相似度，适合自然语言改写后的问题；
- 混合检索同时考虑语义相似度和关键词命中，适合包含产品名、模块名、告警码、接口名等专有词的问题；
- 检索时只返回 `text/source/page` 等普通字段，不返回 `dense/sparse` 向量字段，符合 Milvus Quickstart 中通过 `output_fields` 控制返回字段的做法，也避免 `not allowed to retrieve raw data of field sparse` 错误。


# RAG基础：构建基础检索问答链

本章基于上一章构建好的 **简单检索向量库 `ZXVMAXS`** 搭建 RAG 问答链。流程可以拆成四步：

1. 连接 Milvus，加载只包含 `dense` 向量字段的简单检索 collection；
2. 将用户问题转成向量，从 Milvus 中召回 Top-K 相关文档片段；
3. 把召回片段拼接进 Prompt，约束大模型只基于上下文回答；
4. 对比“有知识库”和“无知识库”回答，观察 RAG 对专业问题和幻觉抑制的作用。

这一章暂不使用 BM25 sparse 混合检索，目的是先把最基础的 dense 向量检索问答链讲清楚。


## 1. 加载向量数据库 Milvus

这里加载上一章创建的简单检索集合 `ZXVMAXS`。需要注意三点：

- `embedding_function` 必须和构建向量库时使用的 embedding 模型一致；
- collection 的向量字段名是 `dense`，文本字段名是 `text`，不能使用 LangChain 默认的 `vector`；
- 检索结果只需要返回文本和元数据字段，不需要返回原始向量字段，所以后面会从 `fields` 中排除 `dense/sparse/vector`。


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from pymilvus import MilvusClient
from langchain_milvus import Milvus
from langchain_openai import OpenAIEmbeddings

my_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)

connection_args = {
    "uri": os.environ["MILVUS_URI"],
}

_milvus_client = MilvusClient(**connection_args)

vectordb = Milvus(
    embedding_function=my_emb,
    collection_name="ZXVMAXS",
    connection_args=connection_args,
    vector_field="dense",
    text_field="text",
)

# 搜索结果只需要返回文本和元数据字段，不返回 dense 向量原始值。
vectordb.fields = [
    field for field in vectordb.fields
    if field not in {"dense", "sparse", "vector"}
]

print("向量字段：", vectordb.vector_fields)
print("返回字段：", vectordb.fields)


**预期输出：**

```text
向量字段： ['dense']
返回字段： ['text', 'pk', 'producer', 'creator', 'creationdate', 'source', 'file_path', 'total_pages', 'format', 'title', 'author', 'subject', 'keywords', 'moddate', 'trapped', 'modDate', 'creationDate', 'page', 'source_file']
```

In [ ]:
results = vectordb.similarity_search(query="什么是vmax的上网日志系统？", k=5)

for i, doc in enumerate(results, 1):
    source = doc.metadata.get("source") or doc.metadata.get("file_path") or "未知来源"
    page = doc.metadata.get("page", "-")
    preview = doc.page_content[:220].replace("\n", " ")
    print(f"===== 召回结果 {i} | source={source} | page={page} =====")
    print(preview)
    print()


**预期输出：**

```text
===== 召回结果 1 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=13 =====
4 功能 本章包含如下主题：  上网日志保存 10  上网日志查询 10  上网日志批量导入查询 11  日志管理 11  账号管理 11  角色管理 12  资源监控 12  告警管理 12  省级网关对接 12  拨测结果自动比对功能 12  NAT日志入库功能 13  北向接口 13  上网日志历史查询 13  云化上网日志XDR查询 13 以下介绍ZXVMAX-S的主要的功能。 4.1 上网日志保存 

===== 召回结果 2 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=4 =====
1 产品定位与特点 本章包含如下主题：  产品背景 1  产品定位 1  产品特点 1 1.1 产品背景 随着移动通信网络的不断发展，移动智能终端用户数突飞猛进，越来越多的用户开始使用移动终 端上网。为了网络安全考虑，政府要求移动运营商可以追溯用户的上网行为，这就要求移动运营 商能记录下所有用户的上网日志。 1.2 产品定位 ZXVMAX-S系统可以长时间的保存用户的上网日志，并可通过公网IP、目的IP、IMSI、 MSISDN、

===== 召回结果 3 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=7 =====
ZXVMAX-S 多维价值分析系统 产品描述（上网日志业务） 图2-3 ZXVMAX逻辑结构图 模块功能  采集层： à Probe：探针，支持硬采和软采，负责进行标准接口的信令和Packet抓包。 à Firewall log Process：防火墙日志处理单元，负责接收防火墙日志。 à Data Merge Process：数据合成单元，负责关联网元信令和防火墙日志，生成上网日志 数据。  存储共享层： à 分布式存储集群：采用

===== 召回结果 4 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=16 =====
4 功能 4.11 NAT日志入库功能 ZXVMAX-S支持将NAT日志保存到GP数据库，可通过公网IP或目的IP查询NAT日志。 4.12 北向接口 支持中移规范北向接口，中移集团查询平台下发查询命令，上网日志系统将查询结果以文件方 式上传至查询平台。包括云化核心网的4G/5G上网日志查询、4G/5G控制面用户面XDR查询、 4G/5G软采XDR查询、VoLTE XDR查询、5G消息XDR查询，以及原始码流查询功能。 4.13 上网日

===== 召回结果 5 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=9 =====
ZXVMAX-S 多维价值分析系统 产品描述（上网日志业务） 图2-4 ZXVMAX系统服务器机柜 6 SJ-20220623151803-017 | 2023-03-30（R1.0）
```

## 2. 创建大语言模型

这里使用 vLLM 部署的 Qwen3（OpenAI 兼容接口）初始化一个聊天模型。教学时可以把它理解为“生成器”：检索器负责从知识库找上下文，LLM 负责根据上下文组织最终答案。

运行前需要确认仓库根目录 `.env` 中已配置 `vLLM_QWEN_BASE_URL`、`vLLM_QWEN_API_KEY`、`vLLM_QWEN_MODEL_ID`。

**vllm版本**

In [ ]:
import os
import httpx
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model

# ── 1. 大语言模型配置 ─────────────────────────────
_ = load_dotenv(find_dotenv())

my_llm = init_chat_model(
    model=os.getenv("vLLM_QWEN_MODEL_ID"),
    model_provider="openai",
    base_url=os.getenv("vLLM_QWEN_BASE_URL"),
    api_key=os.getenv("vLLM_QWEN_API_KEY"),
    http_client=httpx.Client(trust_env=False),
    temperature=0.3,
    max_tokens=10000,
)

**备选：改用云端 OpenAI 兼容 API**

In [ ]:
# 备选写法：把大模型换成任意云端 OpenAI 兼容服务（占位符，勿写入真实密钥）
# import os
# import httpx
# from dotenv import load_dotenv, find_dotenv
# from langchain.chat_models import init_chat_model
#
# _ = load_dotenv(find_dotenv())
# cloud_api_key = os.getenv("CLOUD_LLM_API_KEY")   # 自行在 .env 中配置
# if not cloud_api_key:
#     raise ValueError("未配置 CLOUD_LLM_API_KEY，请先在仓库根目录 .env 中填写。")
#
# my_llm = init_chat_model(
#     model="替换为云端模型名",
#     model_provider="openai",
#     base_url="https://替换为服务端点/v1",
#     api_key=cloud_api_key,
#     http_client=httpx.Client(trust_env=False),
#     temperature=0.3,
#     max_tokens=3000,
# )
# print(my_llm.invoke("你是谁？").content)

## 3. 构建检索问答链

Prompt 是 RAG 问答链的关键控制点。这里的模板强调三件事：

- 只能基于召回上下文回答，减少脱离知识库的发挥；
- 如果上下文没有答案，明确回答“我不知道”；
- 输出保持简洁、连贯，适合教学演示。


构建系统提示词

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """你是 VMAX 运维助手，请严格根据以下上下文信息回答问题。

上下文：
{context}

问题：{question}

回答要求：
1. 只基于上下文回答，不要编造上下文之外的信息；
2. 如果上下文没有提供答案，请回答“我不知道”；
3. 回答要简洁、连贯，不要重复相同信息；
4. 最后说“谢谢你的提问！”。

回答："""

QA_CHAIN_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=template,
)


创建一个基于模板的LCEL检索链，LCEL（LangChain Expression Language）可以把“检索文档 → 格式化上下文 → 填充 Prompt → 调用 LLM → 解析输出”串成一条可组合的链。

In [ ]:
from langchain_classic.chains import RetrievalQA

# 基础版先使用普通相似度检索，便于理解：问题向量 -> Milvus Top-K 文档片段。
# 如果希望增加结果多样性，可以把 search_type 改为 "mmr"，并设置 fetch_k/lambda_mult。
base_retriever = vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)


In [ ]:
def format_docs(docs):
    """把召回文档拼成 prompt 中的 context，并保留简单来源标记，便于观察依据。"""
    formatted = []
    for i, doc in enumerate(docs, 1):
        source = (doc.metadata.get("source_name") or doc.metadata.get("source")
                  or doc.metadata.get("file_path") or "未知来源")
        page = doc.metadata.get("page_number", doc.metadata.get("page", "-"))
        section = doc.metadata.get("section_title", "未识别章节")
        formatted.append(
            f"[文档{i} | source={source} | page={page} | section={section}]\n{doc.page_content}"
        )
    return "\n\n".join(formatted)


In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

answer_chain = (
    {
        "context": lambda x: format_docs(x["source_documents"]),
        "question": lambda x: x["question"],
    }
    | QA_CHAIN_PROMPT
    | my_llm
    | StrOutputParser()
)

# 先保留原始召回文档，再基于这些文档生成答案。
# 最终统一返回 question、source_documents 和 result，便于展示答案来源。
rag_chain = (
    RunnableParallel(
        question=RunnablePassthrough(),
        source_documents=base_retriever,
    )
    | RunnablePassthrough.assign(result=answer_chain)
)

In [ ]:
question_1 = "什么是ZXVMAX-S的5GC业务？"
result = rag_chain.invoke(question_1)
print("LCEL检索链回答 question_1 的结果：")
print(result["result"])

**预期输出：**

```text
LCEL检索链回答 question_1 的结果：


我不知道。谢谢你的提问！
```

## 4. 检索问答链效果测试

下面用两个问题测试基础版 RAG 链：

- `question_1` 是知识库内的专业问题，用来观察 RAG 是否能从 VMAX 文档中召回依据并生成答案；
- `question_2` 是知识库外的常识问题，用来观察 Prompt 是否能约束模型在缺少上下文时回答“我不知道”。

> **读数提示**：本项目的 Prompt 明确要求「上下文不足时只回答“我不知道”」，而 vLLM 上的 Qwen3 比较保守。当召回头部是目录页、图表页这类信息密度低的片段时，RAG 链会直接拒答，无知识库基线反而写得更长。这不是链路故障，而是**检索质量 + 严格约束型 Prompt** 的共同结果。教学时请结合打印出来的来源文件名和页码判断是否命中正文，再看评估环节：RAG 的优势体现在**忠实度（不编造）**，而不是回答长度。想要更“敢答”的效果，可以增大 `k`、改用混合检索，或放宽 Prompt 里的拒答约束。

### 4.1 知识库内问题：专业问题回答

这个问题会经过“问题向量化 → Milvus 相似度检索 → 召回片段填入 Prompt → LLM 生成答案”的完整 RAG 流程。代码同时打印前 3 个来源文档，帮助学习者理解答案依据来自哪里。


In [ ]:
question_1 = "什么是ZXVMAX-S的5GC业务？"
result = rag_chain.invoke(question_1)

print("大模型+知识库后回答 question_1 的结果：")
print(result["result"])
print("\n来源文档数量：", len(result["source_documents"]))
for i, doc in enumerate(result["source_documents"][:3], 1):
    source = (doc.metadata.get("source_name") or doc.metadata.get("source")
              or doc.metadata.get("file_path") or "未知来源")
    page = doc.metadata.get("page_number", doc.metadata.get("page", "-"))
    section = doc.metadata.get("section_title", "未识别章节")
    print(f"来源 {i}: source={source} | page={page} | section={section}")


**预期输出：**

```text
大模型+知识库后回答 question_1 的结果：


我不知道。谢谢你的提问！

来源文档数量： 5
来源 1: source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=0 | section=未识别章节
来源 2: source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=29 | section=未识别章节
来源 3: source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（语音业务）.pdf | page=0 | section=未识别章节
```

### 4.2 知识库外问题：拒答能力测试

这个问题与 VMAX 知识库无关。理想情况下，检索到的上下文无法支持回答，Prompt 会约束模型输出“我不知道”，而不是根据通用知识自由发挥。


In [ ]:
question_2 = "周杰伦是谁？"
result = rag_chain.invoke(question_2)

print("大模型+知识库后回答 question_2 的结果：")
print(result["result"])


**预期输出：**

```text
大模型+知识库后回答 question_2 的结果：


我不知道。谢谢你的提问！
```

### 4.3 无知识库大模型回答对比

为了体现 RAG 的作用，下面绕过向量数据库，直接让大模型回答同样的问题。对比时重点观察：

- 专业问题是否容易出现泛化、不准确或脱离文档的回答；
- 知识库外问题是否会被直接回答，而不是拒答。


In [ ]:
prompt_template = f"请回答下列问题:\n{question_1}"

raw_result_1 = my_llm.invoke(prompt_template)
print("无知识库回答 question_1 的结果：")
print(raw_result_1.content)


**预期输出：**

```text
无知识库回答 question_1 的结果：


ZXVMAX-S 的 5GC 业务，通常是指 **ZXVMAX-S 产品/平台所承载的 5G 核心网（5GC，5G Core Network）业务与能力**。

简单来说，它就是在 5G SA 组网中，由 ZXVMAX-S 提供或支撑的核心网功能，负责完成终端接入、移动性管理、会话建立、用户面数据转发、鉴权、策略控制、计费、网络切片、开放能力等核心网业务。

主要包括以下几类 5GC 业务/网元能力：

1. **接入与移动性管理业务**
   - 对应 AMF 网元。
   - 负责终端接入 5G 网络、注册、鉴权协调、移动性管理、寻呼、NAS 信令处理等。

2. **会话管理业务**
   - 对应 SMF 网元。
   - 负责 PDU 会话建立、修改、释放，IP 地址分配，UPF 选择，会话策略控制等。

3. **用户面转发业务**
   - 对应 UPF 网元。
   - 负责用户数据包的转发、路由、QoS 处理、流量统计、本地分流、边缘计算接入等。

4. **用户数据与签约管理业务**
   - 对应 UDM/UDR 网元。
   - 负责用户签约数据、鉴权数据、会话数据、位置信息等的管理和存储。

5. **鉴权与安全业务**
   - 对应 AUSF 网元。
   - 负责 5G 用户身份认证、密钥协商、安全鉴权等。

6. **策略控制业务**
   - 对应 PCF 网元。
   - 负责制定用户策略、会话策略、QoS 策略、切片策略、计费策略等。

7. **网络切片相关业务**
   - 对应 NSSF 等网元。
   - 负责切片选择、切片实例管理、切片策略协调等，支持不同行业应用提供差异化网络服务。

8. **计费与开放能力业务**
   - 对应 CHF、NEF 等网元。
   - 负责在线计费、离线计费、能力开放、第三方应用调用网络能力等。

9. **服务化架构相关能力**
   - 5GC 采用 SBA 服务化架构，ZXVMAX-S 的 5GC 业务通常也基于服务化接口，如 N1、N2、N4、N6、N7、N8、N10、N11、N12、N15、N16 等接口完成网元间交互。

一句话概括：

> **ZXVMAX-S 的 5GC 业务，就是基于 ZXVMAX-S 平台提供的 5G 核心网服务，包括 AMF、SMF、UPF、UDM、AUSF、PCF、NRF、NSSF、CHF、NEF 等核心网功能，用于支撑 5G 终端接入、会话管理、数据转发、鉴权、策略、计费和网络切片等业务。**
```

In [ ]:
prompt_template = f"请回答下列问题:\n{question_2}"

raw_result_2 = my_llm.invoke(prompt_template)
print("无知识库回答 question_2 的结果：")
print(raw_result_2.content)


**预期输出：**

```text
无知识库回答 question_2 的结果：


周杰伦（Jay Chou）是华语流行音乐歌手、词曲作者、音乐制作人、演员和导演，1979年出生于台湾台北。他以融合流行、R&B、嘻哈、摇滚与中国传统音乐元素的风格著称，代表作包括《双截棍》《青花瓷》《稻香》《晴天》《七里香》等。

他不仅推动了华语流行音乐的发展，也参与电影演出与制作，如《头文字D》《不能说的秘密》等，是华语乐坛极具影响力的音乐人之一。
```

### 4.4 效果小结

通过以上两个问题可以看到：接入知识库后，模型会优先依据本地文档回答 VMAX 专业问题；当问题与知识库无关时，Prompt 会约束模型回答“我不知道”。不接入知识库时，模型只能依赖自身参数知识，容易给出泛化甚至不准确的回答。这正是 RAG 缓解幻觉、补充私有知识的核心价值。


## 5. 添加历史对话的记忆功能

基础检索问答链每次只处理当前问题。如果用户提出“继续整理成表格”“上面结果再简化一下”这类依赖上一轮上下文的问题，就需要加入对话历史。

下面使用 `ConversationalRetrievalChain` 演示最简单的多轮问答。注意：这里使用的是 `langchain_classic` 的教学 API，当前版本会有 deprecation warning，但不影响本课程示例运行。


In [ ]:
from langchain_classic.memory import ConversationBufferMemory

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
)


**预期输出：**

```text
…/1922486086.py:3: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(
~/env_rag_uv/.venv/lib/python3.12/site-packages/pydantic/main.py:280: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
```

（输出里的 `~` 代表运行机器上的用户主目录，每台机器不同。）

（输出里的 `…` 是 Jupyter 内核本次运行的临时目录，每台机器、每次运行都不同；告警本身可以忽略。）

### 5.1 对话检索链（ConversationalRetrievalChain）

对话检索链会先结合历史对话改写或理解当前问题，再进行检索和回答。它适合演示“第二轮问题依赖第一轮答案”的场景。


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import ConversationalRetrievalChain

template = """你是 VMAX 运维助手，请参考以下对话历史和上下文回答问题。

对话历史：
{chat_history}

相关上下文：
{context}

问题：{question}

回答要求：
1. 只基于上下文回答，不要编造上下文之外的信息；
2. 如果上下文没有提供答案，请回答“我不知道”；
3. 最后说“谢谢你的提问！”。

回答："""

QA_PROMPT = PromptTemplate(
    input_variables=["chat_history", "context", "question"],
    template=template,
)

conversation_qa_chain = ConversationalRetrievalChain.from_llm(
    llm=my_llm,
    retriever=base_retriever,
    memory=memory,
    combine_docs_chain_kwargs={"prompt": QA_PROMPT},
    chain_type="stuff",
)


In [ ]:
questions = [
    "什么是ZXVMAX-S的5GC业务？包含哪些功能？/no_think",
    "包含哪些功能？对每个功能做下阐述/no_think",
    "将以上结果整理成markdown表格形式/no_think",
]

for question in questions:
    result = conversation_qa_chain.invoke({"question": question})
    print(f"问题：{question}")
    print(f"回答：{result['answer']}")
    print("\n" + "=" * 50 + "\n")


**预期输出：**

```text
问题：什么是ZXVMAX-S的5GC业务？包含哪些功能？/no_think
回答：

上下文未明确定义“ZXVMAX-S的5GC业务”具体是什么。根据上下文，ZXVMAX-S是面向用户的网络运维和运营分析产品，支持实时分析和事后分析；其应用分析产品可实现性能优化、事件处理、用户分析、终端分析、业务优化等业务功能，并支持性能分析、事件分析、终端分析等多种应用。

谢谢你的提问！

==================================================
问题：包含哪些功能？对每个功能做下阐述/no_think
回答：

我不知道。谢谢你的提问！

==================================================
问题：将以上结果整理成markdown表格形式/no_think
回答：

上下文未明确定义“ZXVMAX-S的5GC业务”具体是什么，仅可基于上下文整理出以下相关功能/能力：

| 类别 | 功能/能力 | 上下文阐述 |
|---|---|---|
| 应用分析产品业务功能 | 性能优化 | 应用分析产品实现的业务功能之一。 |
| 应用分析产品业务功能 | 事件处理 | 应用分析产品实现的业务功能之一。 |
| 应用分析产品业务功能 | 用户分析 | 应用分析产品实现的业务功能之一。 |
| 应用分析产品业务功能 | 终端分析 | 应用分析产品实现的业务功能之一；ZXVMAX-S支持的应用之一。 |
| 应用分析产品业务功能 | 业务优化 | 应用分析产品实现的业务功能之一。 |
| 应用分析产品特性 | 独立发布/部署 | 能针对Feature独立发布，基于服务器+客户端方式部署。 |
| 系统支持应用 | 性能分析 | ZXVMAX-S支持的应用之一，可根据目标用户需求开启对应权限、部署相应模块。 |
| 系统支持应用 | 事件分析 | ZXVMAX-S支持的应用之一，可根据目标用户需求开启对应权限、部署相应模块。 |
| 系统支持应用 | 自定义查询/分析 | 根据用户需求进行数据自定义的查询或分析，满足不同场景下用户分析解决问题的需求。 |
| 分析模式 | 实时分析 | 支持网络质量、数据业务质量等实时分析。 |
| 分析模式 | 事后分析 | 支持用户投诉、问题分析等事后分析。 |
| 系统组件功能 | 存储共享 | 基于Hadoop软件的服务器集群，实现海量数据的存储、预处理、共享。 |
| 系统组件功能 | 采集层产品-IDS | 面向核心网信令抓包，通常需要在各标准接口进行端口映射或分光。 |
| 5GC业务相关图示 | 会话跟踪 | 上下文仅出现“会话跟踪组网逻辑图”，未提供功能阐述。 |

谢谢你的提问！

==================================================
```

### 5.2 对话链结果解读

第一轮问题会从 Milvus 中检索 `ZXVMAXS` 简单向量库，生成关于 5GC 业务的回答；后两轮问题没有重复完整背景，只说“以上结果”，对话链会借助 `memory` 中保存的历史问题和回答继续生成表格。这说明 memory 解决的是“多轮上下文连续性”问题，而向量数据库解决的是“外部知识检索”问题，两者职责不同。


# RAG进阶：查询理解与上下文优化

基础版 RAG 采用“原始问题直接检索、召回结果直接进入 LLM”的流程。本章分别优化检索前和检索后两个阶段：

1. **查询理解**：规范用户问题，识别业务类型，并决定采用 Dense 还是 Hybrid 检索；
2. **上下文优化**：整理召回结果，减少重复内容，补齐相邻语义，并控制输入 LLM 的上下文长度。

整体链路如下：

`用户问题 → 查询规范化 → 条件识别与检索路由 → 文档召回 → 去重与邻接扩展 → 上下文预算控制 → LLM`

> 本章先准备两个可复用的优化模块；后面的“混合检索”和“Agentic RAG”章节再把它们接入完整链路。

## 1. 查询理解

### 1.1 目标与处理策略

查询理解发生在文档召回之前，目标是把用户的自然表达转换为稳定、可执行的检索条件。本节依次完成：

- **查询规范化**：统一产品别名和缩写，例如把 `vmax` 转换为 `ZXVMAX-S`；
- **Metadata Filter 条件识别**：从问题中提取 `business_type`，为后续元数据过滤做准备；
- **检索路由**：包含告警码、接口名或精确术语时选择 Hybrid，普通语义问题选择 Dense。

**对应打基础：** `C04_检索优化/02_查询构建.md` 的元数据过滤，以及 `C04_检索优化/04_查询重构与分发.md` 的查询改写与路由。

本节输出统一的查询策略字典：`original_query`、`normalized_query`、`business_type` 和 `route`。

### 1.2 查询理解实现

下面将查询规范化、业务类型识别和检索路由封装为统一的 `prepare_retrieval_query()`。

In [ ]:
import re

QUERY_ALIASES = {
    "vmax": "ZXVMAX-S",
    "zxvmax": "ZXVMAX-S",
    "5g核心网": "5GC",
}
BUSINESS_TYPES = ["上网日志业务", "语音业务", "数据业务", "端到端业务", "5GC业务"]
BUSINESS_KEYWORDS = {
    "上网日志": "上网日志业务",
    "语音": "语音业务",
    "端到端": "端到端业务",
    "5GC": "5GC业务",
}

def normalize_query(question: str) -> str:
    normalized = re.sub(r"\s+", " ", question).strip()
    for alias in sorted(QUERY_ALIASES, key=len, reverse=True):
        canonical = QUERY_ALIASES[alias]
        if re.fullmatch(r"[A-Za-z0-9-]+", alias):
            pattern = rf"(?<![A-Za-z0-9-]){re.escape(alias)}(?![A-Za-z0-9-])"
        else:
            pattern = re.escape(alias)
        normalized = re.sub(pattern, canonical, normalized, flags=re.IGNORECASE)
    return normalized

def extract_query_business_type(question: str):
    compact = question.replace(" " , "")
    for business_type in BUSINESS_TYPES:
        if business_type.replace(" " , "") in compact:
            return business_type
    for keyword, business_type in BUSINESS_KEYWORDS.items():
        if keyword.upper() in question.upper():
            return business_type
    if "5GC" in question.upper() or "5G核心网" in question:
        return "5GC业务"
    return None

def choose_retrieval_route(question: str) -> str:
    keyword_pattern = r"(?:\d{6,}|[A-Z]{2,}[A-Z0-9_-]*|接口|字段|告警|模块|日志)"
    return "hybrid" if re.search(keyword_pattern, question) else "dense"

def prepare_retrieval_query(question: str) -> dict:
    normalized = normalize_query(question)
    return {
        "original_query": question,
        "normalized_query": normalized,
        "business_type": extract_query_business_type(normalized),
        "route": choose_retrieval_route(normalized),
    }

query_strategy_examples = [
    prepare_retrieval_query("vmax 上网日志有哪些功能？"),
    prepare_retrieval_query("114020039 告警怎么处理？"),
    prepare_retrieval_query("为什么需要做业务质量定界？"),
]
query_strategy_examples

**预期输出：**

```text
[{'original_query': 'vmax 上网日志有哪些功能？',
  'normalized_query': 'ZXVMAX-S 上网日志有哪些功能？',
  'business_type': '上网日志业务',
  'route': 'hybrid'},
 {'original_query': '114020039 告警怎么处理？',
  'normalized_query': '114020039 告警怎么处理？',
  'business_type': None,
  'route': 'hybrid'},
 {'original_query': '为什么需要做业务质量定界？',
  'normalized_query': '为什么需要做业务质量定界？',
  'business_type': None,
  'route': 'dense'}]
```

### 1.3 运行结果解读

三个示例分别展示了别名规范化、精确标识符路由和普通语义路由。这里先生成检索策略，暂不执行检索；`business_type` 可用于 Metadata Filter，`route` 会在后续章节决定调用 Dense 还是 Hybrid 检索器。

## 2. 上下文优化

### 2.1 目标与处理策略

上下文优化发生在文档召回之后、调用 LLM 之前。本节按以下顺序处理候选文档：

1. **去重**：删除重复或高度重合的数据块；
2. **邻接扩展**：依据 `source_name + chunk_index` 补充命中块前后的内容，减少语义断裂；
3. **预算控制**：在 `max_chars` 限制内选择文档，避免上下文无限增长。

**对应打基础：** `C03_索引构建/05_索引优化.md` 的上下文扩展，以及 `C04_检索优化/05_检索进阶技术.md` 的上下文压缩思想。

> 邻接扩展依赖优化预处理阶段写入的 `source_name` 和 `chunk_index` metadata。

### 2.2 上下文优化实现

下面分别实现文档去重、邻接数据块扩展和上下文预算控制。

In [ ]:
def document_identity(doc):
    return (
        doc.metadata.get("source_name") or doc.metadata.get("source"),
        doc.metadata.get("page_number", doc.metadata.get("page")),
        doc.metadata.get("section_title"),
        re.sub(r"\s+", "", doc.page_content)[:160],
    )

def deduplicate_documents(docs):
    seen = set()
    result = []
    for doc in docs:
        identity = document_identity(doc)
        if identity not in seen:
            seen.add(identity)
            result.append(doc)
    return result

def build_chunk_lookup(all_docs):
    return {
        (doc.metadata.get("source_name"), doc.metadata.get("chunk_index")): doc
        for doc in all_docs
        if doc.metadata.get("source_name") is not None
        and doc.metadata.get("chunk_index") is not None
    }

def expand_adjacent_chunks(docs, all_docs, window=1):
    lookup = build_chunk_lookup(all_docs)
    expanded = []
    for doc in docs:
        source_name = doc.metadata.get("source_name")
        chunk_index = doc.metadata.get("chunk_index")
        if source_name is None or chunk_index is None:
            expanded.append(doc)
            continue
        for neighbor_index in range(max(0, chunk_index - window), chunk_index + window + 1):
            neighbor = lookup.get((source_name, neighbor_index))
            if neighbor is not None:
                expanded.append(neighbor)
    return deduplicate_documents(expanded)

def select_context_with_budget(docs, max_chars=6000):
    selected, used = [], 0
    for doc in deduplicate_documents(docs):
        remaining = max_chars - used
        if remaining <= 0:
            break
        if len(doc.page_content) <= remaining:
            selected.append(doc)
            used += len(doc.page_content)
    return selected

print("上下文优化函数准备完成：去重 → 邻接扩展 → 预算控制")

**预期输出：**

```text
上下文优化函数准备完成：去重 → 邻接扩展 → 预算控制
```

### 2.3 上下文优化效果演示

下面用同一个真实问题直观展示：`原始召回 → 去重 → 邻接扩展 → 预算控制`。为模拟 Dense、BM25 等多路召回的常见情况，代码会主动复制第一条结果作为重复块。

> 演示依赖前面已经创建的 `base_retriever` 和优化预处理产生的 `split_docs`。

In [ ]:
context_demo_query = "ZXVMAX-S的数据可靠性如何保障？"

# 取 3 条真实召回结果，再复制第 1 条，模拟多路召回产生重复块。
retrieved_docs = base_retriever.invoke(context_demo_query)[:3]
raw_demo_docs = retrieved_docs + retrieved_docs[:1]
deduplicated_demo_docs = deduplicate_documents(raw_demo_docs)

# 优先使用召回结果中带完整 metadata 的块；旧向量库缺字段时回退到 split_docs。
expand_seed_docs = [
    doc for doc in deduplicated_demo_docs
    if doc.metadata.get("source_name") is not None
    and doc.metadata.get("chunk_index") is not None
][:2]
if not expand_seed_docs:
    expand_seed_docs = [
        doc for doc in split_docs
        if "数据可靠性" in doc.metadata.get("section_title", "")
    ][:1]

expanded_demo_docs = expand_adjacent_chunks(expand_seed_docs, split_docs, window=1)
final_demo_docs = select_context_with_budget(expanded_demo_docs, max_chars=1800)

def summarize_context_stage(stage_name, docs):
    total_chars = sum(len(doc.page_content) for doc in docs)
    print(f"\n===== {stage_name}：{len(docs)} 块 / {total_chars} 字符 =====")
    for i, doc in enumerate(docs, 1):
        source = (doc.metadata.get("source_name") or doc.metadata.get("source")
                  or doc.metadata.get("file_path") or "未知来源")
        page = doc.metadata.get("page_number", doc.metadata.get("page", "-"))
        section = doc.metadata.get("section_title", "未识别章节")
        chunk_index = doc.metadata.get("chunk_index", "-")
        preview = re.sub(r"\s+", " ", doc.page_content)[:90]
        print(f"{i}. chunk={chunk_index} | page={page} | section={section}")
        print(f"   source={source}")
        print(f"   {preview}...")

print("演示问题：", context_demo_query)
summarize_context_stage("优化前（包含模拟重复块）", raw_demo_docs)
summarize_context_stage("步骤一：去重后", deduplicated_demo_docs)
summarize_context_stage("步骤二：邻接扩展后", expanded_demo_docs)
summarize_context_stage("步骤三：1800 字符预算后", final_demo_docs)

print("\n===== 前后对比 =====")
print(f"重复块删除：{len(raw_demo_docs)} → {len(deduplicated_demo_docs)}")
print(f"邻接扩展：{len(expand_seed_docs)} → {len(expanded_demo_docs)}")
print(f"预算控制：{sum(len(d.page_content) for d in expanded_demo_docs)} → "
      f"{sum(len(d.page_content) for d in final_demo_docs)} 字符")

**预期输出：**

```text
演示问题： ZXVMAX-S的数据可靠性如何保障？

===== 优化前（包含模拟重复块）：4 块 / 2107 字符 =====
1. chunk=- | page=22 | section=未识别章节
   source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf
   8 技术指标 类别 参数 具体指标 时间同步精度 100 ms 时间同步接口 FE 接口数量 1 8.5 可靠性指标 ZXVMAX-S的可靠性指标参见表8-3。 表8-3 可靠性指...
2. chunk=- | page=20 | section=未识别章节
   source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf
   7 可靠性设计 容错能力 ZXVMAX-S通过对关键软件资源的定时检测、实时任务监控、存储保护、数据校验、操作日志 信息保存等手段，可有效地防止小软件故障对系统所造成的冲击，提高了...
3. chunk=- | page=32 | section=未识别章节
   source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（数据业务）.pdf
   7 可靠性设计 容错能力 ZXVMAX-S通过对关键软件资源的定时检测、实时任务监控、存储保护、数据校验、操作日志 信息保存等手段，可有效地防止小软件故障对系统所造成的冲击，提高了...
4. chunk=- | page=22 | section=未识别章节
   source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf
   8 技术指标 类别 参数 具体指标 时间同步精度 100 ms 时间同步接口 FE 接口数量 1 8.5 可靠性指标 ZXVMAX-S的可靠性指标参见表8-3。 表8-3 可靠性指...

===== 步骤一：去重后：3 块 / 1832 字符 =====
1. chunk=- | page=22 | section=未识别章节
   source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf
   8 技术指标 类别 参数 具体指标 时间同步精度 100 ms 时间同步接口 FE 接口数量 1 8.5 可靠性指标 ZXVMAX-S的可靠性指标参见表8-3。 表8-3 可靠性指...
2. chunk=- | page=20 | section=未识别章节
   source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf
   7 可靠性设计 容错能力 ZXVMAX-S通过对关键软件资源的定时检测、实时任务监控、存储保护、数据校验、操作日志 信息保存等手段，可有效地防止小软件故障对系统所造成的冲击，提高了...
3. chunk=- | page=32 | section=未识别章节
   source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（数据业务）.pdf
   7 可靠性设计 容错能力 ZXVMAX-S通过对关键软件资源的定时检测、实时任务监控、存储保护、数据校验、操作日志 信息保存等手段，可有效地防止小软件故障对系统所造成的冲击，提高了...

===== 步骤二：邻接扩展后：0 块 / 0 字符 =====

===== 步骤三：1800 字符预算后：0 块 / 0 字符 =====

===== 前后对比 =====
重复块删除：4 → 3
邻接扩展：0 → 0
预算控制：0 → 0 字符
```

### 2.4 如何解读演示结果

运行时重点观察：

- **去重后文档数下降**：重复块不再占用 LLM 上下文；
- **邻接扩展后出现连续的 `chunk_index`**：命中块前后的语义得到补齐；
- **预算控制后不超过 1800 字符**：最终上下文长度和调用成本可控。

邻接扩展导致文档数暂时增加是预期行为：它负责补全语义，最后的预算控制负责限制长度。

## 3. 查询理解与上下文优化的组合

查询理解和上下文优化不是两套独立的 RAG，而是完整检索链前后两个位置的增强模块：

| 阶段 | 输入 | 核心处理 | 输出 |
| --- | --- | --- | --- |
| 查询理解 | 用户原始问题 | 规范化、条件识别、路由 | 查询策略 |
| 检索执行 | 查询策略 | Dense / Hybrid 召回、Metadata Filter | 候选文档 |
| 上下文优化 | 候选文档 | 去重、邻接扩展、预算控制 | 最终上下文 |
| 答案生成 | 最终上下文 + 原始问题 | Prompt + LLM | 带知识依据的答案 |

后续章节的衔接关系：

- **混合检索**：消费查询理解产生的 `route`，并对召回结果执行上下文优化；
- **Agentic RAG**：由 Agent 决定是否调用检索工具，工具内部仍可复用查询理解与上下文优化；
- **基础版 RAG**：保留为基线，便于对比加入这些优化前后的效果。

# RAG进阶：混合检索

基础版 RAG 只使用 `dense` 语义向量检索。它适合自然语言问题，但对产品名、告警码、接口名、缩写等“字面关键词”不一定稳定。本章使用上一章构建的 `ZXVMAXS_HYBRID` 集合，演示 **dense 语义检索 + sparse BM25 关键词检索** 的混合检索，并进一步加入 Rerank 精排。

本章仍然围绕 VMAX 文档做教学实验，重点观察三件事：

1. 混合检索如何融合语义召回和关键词召回；
2. weighted 与 RRF 两种融合策略的差异；
3. Rerank 如何从召回候选中重新筛选更适合输入 LLM 的上下文。


## 1. 混合检索原理

混合检索通常包含两条召回路径：

- **Dense 向量检索**：将 query 和文档 chunk 编码为 dense embedding，按语义相似度召回，适合自然语言表达、同义改写、模糊问题；
- **Sparse / BM25 检索**：基于关键词、词频和逆文档频率召回，适合产品名、模块名、告警码、接口名、缩写、数字编号等精确匹配。

Milvus 2.6 支持通过 `BM25BuiltInFunction` 在服务端生成 sparse 向量。也就是说，客户端只需要写入 `text` 和 `dense` embedding，Milvus 会根据 `text` 自动生成 `sparse` 字段并参与混合检索。

注意：`sparse` 是 BM25 函数输出的稀疏向量字段，只能用于检索打分，不能作为 raw data 返回。因此使用 LangChain Milvus 检索前，需要从 `vectorstore.fields` 中排除 `dense/sparse/vector`。


## 2. 教学版混合检索实现

在直接调用 Milvus 内置混合检索之前，先用一段教学代码把“混合”这件事拆开看清楚：

1. **Dense 路径**：从简单向量库 `ZXVMAXS` 做语义向量检索；
2. **BM25 路径**：从同一批文档文本构建本地 BM25 关键词检索；
3. **分数归一化**：把两路分数都压到 0-1；
4. **加权融合**：`hybrid_score = dense_weight * dense_score + bm25_weight * bm25_score`；
5. **排序输出**：观察不同权重下结果如何变化。

这段代码不是最终工程推荐做法，目的是让学生直观看到混合检索的底层逻辑。理解之后，再使用 Milvus 服务端内置 BM25 和 ranker，会更自然。


### 2.1 准备 Dense 与 BM25 两路召回

这里使用 `ZXVMAXS` 简单检索集合中的文本字段构建 BM25。为了课堂运行速度，最多读取 500 条文本；当前示例数据约 322 条，可以完整读取。


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from pymilvus import MilvusClient
from langchain_milvus import Milvus
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from rank_bm25 import BM25Okapi
import re

my_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)
connection_args = {
    "uri": os.environ["MILVUS_URI"],
}

_milvus_client = MilvusClient(**connection_args)

# Dense 语义检索路径：使用简单检索集合 ZXVMAXS。
vectordb_dense = Milvus(
    embedding_function=my_emb,
    collection_name="ZXVMAXS",
    connection_args=connection_args,
    vector_field="dense",
    text_field="text",
)
vectordb_dense.fields = [field for field in vectordb_dense.fields if field not in {"dense", "sparse", "vector"}]

# BM25 关键词路径：从同一个 collection 读取 text/source/page，构建本地 BM25 索引。
raw_rows = _milvus_client.query(
    collection_name="ZXVMAXS",
    filter="pk >= 0",
    output_fields=["text", "source", "file_path", "page"],
    limit=500,
)

bm25_docs = [
    Document(
        page_content=row["text"],
        metadata={
            "source": row.get("source") or row.get("file_path") or "未知来源",
            "page": row.get("page", "-"),
        },
    )
    for row in raw_rows
]

def tokenize_zh_en(text: str):
    """tokenizer：英文/数字按词切，中文按 bigram 切，足够演示 BM25。"""
    text = text.lower()
    tokens = re.findall(r"[a-z0-9][a-z0-9_-]+", text)
    chinese = "".join(re.findall(r"[\u4e00-\u9fff]+", text))
    tokens.extend(chinese[i:i + 2] for i in range(max(len(chinese) - 1, 0)))
    return [token for token in tokens if len(token) >= 2]

bm25_corpus = [tokenize_zh_en(doc.page_content) for doc in bm25_docs]
bm25_index = BM25Okapi(bm25_corpus)

print(f"Dense collection: ZXVMAXS")
print(f"BM25 文档数：{len(bm25_docs)}")


**预期输出：**

```text
Dense collection: ZXVMAXS
BM25 文档数：245
```

### 2.2 固定权重融合

固定权重适合讲解“混合检索分数如何合成”。下面手动设置 `dense_weight=0.6`、`bm25_weight=0.4`，分别代表语义检索和关键词检索的重要性。


In [ ]:
def normalize_scores(score_map):
    if not score_map:
        return {}
    values = list(score_map.values())
    min_score, max_score = min(values), max(values)
    if max_score == min_score:
        return {key: 1.0 for key in score_map}
    return {key: (value - min_score) / (max_score - min_score) for key, value in score_map.items()}

def doc_key(doc):
    source = str(doc.metadata.get("source") or doc.metadata.get("file_path") or "未知来源").strip()
    page = str(doc.metadata.get("page", "-")).strip()
    return source, page, doc.page_content[:80]

def manual_hybrid_search(query, dense_weight=0.6, bm25_weight=0.4, dense_k=8, bm25_k=8, final_k=5):
    # Dense 路径：使用带分数检索。Milvus 返回的分数方向与距离策略有关，教学中做 min-max 归一化后用于相对排序观察。
    dense_pairs = vectordb_dense.similarity_search_with_score(query, k=dense_k)
    dense_docs = [doc for doc, score in dense_pairs]
    dense_raw = {doc_key(doc): float(score) for doc, score in dense_pairs}
    doc_by_key = {doc_key(doc): doc for doc in dense_docs}

    # BM25 路径：本地关键词检索，分数越高越相关。
    tokenized_query = tokenize_zh_en(query)
    bm25_scores = bm25_index.get_scores(tokenized_query)
    top_bm25_indices = sorted(range(len(bm25_scores)), key=lambda idx: bm25_scores[idx], reverse=True)[:bm25_k]
    bm25_raw = {}
    for idx in top_bm25_indices:
        doc = bm25_docs[idx]
        key = doc_key(doc)
        bm25_raw[key] = float(bm25_scores[idx])
        doc_by_key.setdefault(key, doc)

    dense_norm = normalize_scores(dense_raw)
    bm25_norm = normalize_scores(bm25_raw)

    fused = []
    for key, doc in doc_by_key.items():
        dense_score = dense_norm.get(key, 0.0)
        bm25_score = bm25_norm.get(key, 0.0)
        hybrid_score = dense_weight * dense_score + bm25_weight * bm25_score
        fused.append((hybrid_score, dense_score, bm25_score, doc))

    fused.sort(key=lambda item: item[0], reverse=True)
    return fused[:final_k]

def print_manual_hybrid_results(title, results):
    print(f"\n===== {title} =====")
    for i, (hybrid_score, dense_score, bm25_score, doc) in enumerate(results, 1):
        source = doc.metadata.get("source") or doc.metadata.get("file_path") or "未知来源"
        page = doc.metadata.get("page", "-")
        preview = doc.page_content[:160].replace("\n", " ")
        print(f"{i}. hybrid={hybrid_score:.3f} | dense={dense_score:.3f} | bm25={bm25_score:.3f} | source={source} | page={page}")
        print(f"   {preview}")

manual_query = "ZXVMAX-S上网日志有哪些功能？"
fixed_results = manual_hybrid_search(manual_query, dense_weight=0.6, bm25_weight=0.4)
print_manual_hybrid_results("教学版固定权重混合检索", fixed_results)


**预期输出：**

```text

===== 教学版固定权重混合检索 =====
1. hybrid=1.000 | dense=1.000 | bm25=1.000 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=13
   4 功能 本章包含如下主题：  上网日志保存 10  上网日志查询 10  上网日志批量导入查询 11  日志管理 11  账号管理 11  角色管理 12  资源监控 12  告警管理 12  省级网关对接 12  拨测结果自动比对功能 12  NAT日志入库功能 13  北向接口 13  
2. hybrid=0.606 | dense=0.519 | bm25=0.737 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=16
   4 功能 4.11 NAT日志入库功能 ZXVMAX-S支持将NAT日志保存到GP数据库，可通过公网IP或目的IP查询NAT日志。 4.12 北向接口 支持中移规范北向接口，中移集团查询平台下发查询命令，上网日志系统将查询结果以文件方 式上传至查询平台。包括云化核心网的4G/5G上网日志查询、4G/5G控制面用户面XD
3. hybrid=0.544 | dense=0.648 | bm25=0.387 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=4
   1 产品定位与特点 本章包含如下主题：  产品背景 1  产品定位 1  产品特点 1 1.1 产品背景 随着移动通信网络的不断发展，移动智能终端用户数突飞猛进，越来越多的用户开始使用移动终 端上网。为了网络安全考虑，政府要求移动运营商可以追溯用户的上网行为，这就要求移动运营 商能记录下所有用户的上网日志。 1.
4. hybrid=0.417 | dense=0.380 | bm25=0.472 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=7
   ZXVMAX-S 多维价值分析系统 产品描述（上网日志业务） 图2-3 ZXVMAX逻辑结构图 模块功能  采集层： à Probe：探针，支持硬采和软采，负责进行标准接口的信令和Packet抓包。 à Firewall log Process：防火墙日志处理单元，负责接收防火墙日志。 à Data Merge Pr
5. hybrid=0.218 | dense=0.363 | bm25=0.000 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=0
   ZXVMAX-S 多维价值分析系统 产品描述（上网日志业务） 产品版本：V6.23 中兴通讯股份有限公司 地址：深圳市南山区高新技术产业园科技南路中兴通讯大厦 邮编：518057 电话：0755-26770800 400-8301118 800-8301118（座机） 技术支持网站：http://support.zte
```

### 2.3 动态权重融合

动态权重是在固定权重的基础上，根据 query 特征自动调节 dense 与 BM25 的比例：

- 产品名、告警码、接口名、字段名等关键词明显的问题，提高 BM25 权重；
- 原因分析、优化建议、影响说明等语义问题，提高 dense 权重；
- 其他问题保持均衡。


In [ ]:
def choose_manual_hybrid_weights(question: str):
    keyword_patterns = [
        r"\d{4,}",
        r"ZXVMAX-S",
        r"接口|字段|告警|日志|NAT|IMSI|MSISDN|URL",
        r"有哪些|包括哪些|列出|表格",
    ]
    semantic_patterns = [
        r"为什么|原因|如何优化|怎么解决|影响|优缺点|解释",
    ]
    if any(re.search(pattern, question, re.IGNORECASE) for pattern in keyword_patterns):
        return 0.4, 0.6, "keyword_first"
    if any(re.search(pattern, question, re.IGNORECASE) for pattern in semantic_patterns):
        return 0.7, 0.3, "semantic_first"
    return 0.5, 0.5, "balanced"

for test_query in [
    "ZXVMAX-S上网日志有哪些功能？",
    "为什么需要做5GC业务质量定界？",
    "114020039告警如何处理？",
]:
    dense_weight, bm25_weight, query_type = choose_manual_hybrid_weights(test_query)
    results = manual_hybrid_search(
        test_query,
        dense_weight=dense_weight,
        bm25_weight=bm25_weight,
        final_k=3,
    )
    print(f"\nQuery: {test_query}")
    print(f"query_type={query_type}, dense_weight={dense_weight}, bm25_weight={bm25_weight}")
    print_manual_hybrid_results("教学版动态权重混合检索", results)


**预期输出：**

```text

Query: ZXVMAX-S上网日志有哪些功能？
query_type=keyword_first, dense_weight=0.4, bm25_weight=0.6

===== 教学版动态权重混合检索 =====
1. hybrid=1.000 | dense=1.000 | bm25=1.000 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=13
   4 功能 本章包含如下主题：  上网日志保存 10  上网日志查询 10  上网日志批量导入查询 11  日志管理 11  账号管理 11  角色管理 12  资源监控 12  告警管理 12  省级网关对接 12  拨测结果自动比对功能 12  NAT日志入库功能 13  北向接口 13  
2. hybrid=0.650 | dense=0.519 | bm25=0.737 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=16
   4 功能 4.11 NAT日志入库功能 ZXVMAX-S支持将NAT日志保存到GP数据库，可通过公网IP或目的IP查询NAT日志。 4.12 北向接口 支持中移规范北向接口，中移集团查询平台下发查询命令，上网日志系统将查询结果以文件方 式上传至查询平台。包括云化核心网的4G/5G上网日志查询、4G/5G控制面用户面XD
3. hybrid=0.491 | dense=0.648 | bm25=0.387 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=4
   1 产品定位与特点 本章包含如下主题：  产品背景 1  产品定位 1  产品特点 1 1.1 产品背景 随着移动通信网络的不断发展，移动智能终端用户数突飞猛进，越来越多的用户开始使用移动终 端上网。为了网络安全考虑，政府要求移动运营商可以追溯用户的上网行为，这就要求移动运营 商能记录下所有用户的上网日志。 1.

Query: 为什么需要做5GC业务质量定界？
query_type=semantic_first, dense_weight=0.7, bm25_weight=0.3

===== 教学版动态权重混合检索 =====
1. hybrid=0.700 | dense=1.000 | bm25=0.000 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=18
   4 功能 会话分析 该功能针对PDU会话创建、修改、释放的失败流程中的全流程首次失败的失败码进行分析。当 流程失败时，通过多接口的关联分析，找出最先失败的接口的流程，针对该流程的失败原因进行 定界。同时，对PDU会话创建、修改、释放的失败码组合进行聚类分析，用于进行问题的定位处 理。将PDU会话创建、修改、释放失败次数
2. hybrid=0.380 | dense=0.543 | bm25=0.000 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（语音业务）.pdf | page=18
   4 功能与业务 专题名称 专题说明 EPS FB分析专题 支持五大视图（分析视图/概览视图/全局视图/时延视图/接通视图）差异化切入剖析EPS FB网络。其中，分析视图以四大阶段主题页（起呼-回落-接通-返回）的形式提供5G语音 回落的全程分析，直观呈现每个阶段的感知KQI性能指标、关联接口KPI性能指标的发展趋 势，
3. hybrid=0.300 | dense=0.000 | bm25=1.000 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（语音业务）.pdf | page=7
   ZXVMAX-S 多维价值分析系统 产品描述（语音业务） ZXVMAX-S语音业务系统支持事件详细信息的呈现，也支持基于经验算法的问题定位，提升 分析解决问题的效率，满足不同层次用户的需求。  多种应用，自由选择。 ZXVMAX-S语音业务支持信令回溯、记录查询、呼叫分析、网络质量分析、业务质量定界、 时延分析等多种

Query: 114020039告警如何处理？
query_type=keyword_first, dense_weight=0.4, bm25_weight=0.6

===== 教学版动态权重混合检索 =====
1. hybrid=0.600 | dense=0.000 | bm25=1.000 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=15
   ZXVMAX-S 多维价值分析系统 产品描述（上网日志业务） 4.6 角色管理 角色是一组命令的集合，用户被授权的单位，具备某角色的用户可以执行此角色所能操作的任何 命令。角色用来定义用户的权限，包括操作权限和资源权限。 角色管理提供角色对应的操作权限管理，便于系统安全管理。通过将角色分配给用户，实现对用 户的权限设置
2. hybrid=0.404 | dense=0.000 | bm25=0.674 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=2
   4.8 告警管理.................................................................................................................... 12 4.9 省级网关对接......................
3. hybrid=0.400 | dense=1.000 | bm25=0.000 | source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=17
   网元和流程维度下的分布，并最终下钻到XDR错误详单以及全流程信令配合。 注册分析 该功能针对注册失败流程中的全流程首次失败的失败码进行分析。当注册流程失败时，通过多接 口的关联分析，找出最先失败的接口的流程，针对该流程的失败原因进行定界。同时，对注册流 程的失败码组合进行聚类分析，用于进行问题的定位处理。将注册失败次数
```

## 3. Milvus 内置混合检索

上面的手写版本用于理解原理，但工程上不推荐把 BM25 索引拉到本地维护。Milvus 2.6 可以在服务端通过 `BM25BuiltInFunction` 生成 sparse 字段，并在搜索时直接做 weighted 或 RRF 融合。

下面开始使用上一章构建好的 `ZXVMAXS_HYBRID`，这才是推荐的实际调用方式。


### 3.1 加载混合检索向量库

这里加载上一章创建的 `ZXVMAXS_HYBRID` collection。它包含两个向量字段：

- `dense`：语义 embedding 字段；
- `sparse`：Milvus BM25 内置函数生成的关键词检索字段。


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from pymilvus import MilvusClient
from langchain_milvus import Milvus, BM25BuiltInFunction
from langchain_openai import OpenAIEmbeddings

my_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)

connection_args = {
    "uri": os.environ["MILVUS_URI"],
}

_milvus_client = MilvusClient(**connection_args)

vectordb_hybrid = Milvus(
    embedding_function=my_emb,
    builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
    collection_name="ZXVMAXS_HYBRID",
    connection_args=connection_args,
    vector_field=["dense", "sparse"],
    text_field="text",
    consistency_level="Bounded",
)

# 检索结果只返回文本和元数据字段，不返回 dense/sparse 向量原始值。
vectordb_hybrid.fields = [
    field for field in vectordb_hybrid.fields
    if field not in {"dense", "sparse", "vector"}
]

print("向量字段：", vectordb_hybrid.vector_fields)
print("返回字段：", vectordb_hybrid.fields)


**预期输出：**

```text
向量字段： ['dense', 'sparse']
返回字段： ['text', 'pk', 'producer', 'creator', 'creationdate', 'source', 'file_path', 'total_pages', 'format', 'title', 'author', 'subject', 'keywords', 'moddate', 'trapped', 'modDate', 'creationDate', 'page', 'source_file']
```

### 3.2 Weighted 融合

Weighted Ranker 会把 dense 与 sparse 两路分数按权重融合。下面设置 `weights=[0.6, 0.4]`：

- `0.6` 给 dense 语义检索；
- `0.4` 给 sparse BM25 关键词检索。

当问题更偏自然语言时，可以提高 dense 权重；当问题包含告警码、产品名、接口名等关键词时，可以提高 sparse 权重。


In [ ]:
def print_docs(title, docs, preview_chars=220):
    print(f"\n===== {title} =====")
    displayed = set()
    row = 1
    for doc in docs:
        source = str(doc.metadata.get("source") or doc.metadata.get("file_path") or "未知来源").strip()
        page = str(doc.metadata.get("page", "-")).strip()
        key = (source, page, doc.page_content[:80])
        if key in displayed:
            continue
        displayed.add(key)
        preview = doc.page_content[:preview_chars].replace("\n", " ")
        score = doc.metadata.get("rerank_score")
        score_text = f" | rerank_score={score:.2f}" if isinstance(score, (int, float)) else ""
        print(f"{row}. source={source} | page={page}{score_text}")
        print(f"   {preview}")
        row += 1

query = "ZXVMAX-S上网日志有哪些功能？"

weighted_docs = vectordb_hybrid.similarity_search(
    query,
    k=5,
    ranker_type="weighted",
    ranker_params={"weights": [0.6, 0.4]},
)

print_docs("Weighted 混合检索结果", weighted_docs)


**预期输出：**

```text

===== Weighted 混合检索结果 =====
1. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=13
   4 功能 本章包含如下主题：  上网日志保存 10  上网日志查询 10  上网日志批量导入查询 11  日志管理 11  账号管理 11  角色管理 12  资源监控 12  告警管理 12  省级网关对接 12  拨测结果自动比对功能 12  NAT日志入库功能 13  北向接口 13  上网日志历史查询 13  云化上网日志XDR查询 13 以下介绍ZXVMAX-S的主要的功能。 4.1 上网日志保存 
2. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=4
   1 产品定位与特点 本章包含如下主题：  产品背景 1  产品定位 1  产品特点 1 1.1 产品背景 随着移动通信网络的不断发展，移动智能终端用户数突飞猛进，越来越多的用户开始使用移动终 端上网。为了网络安全考虑，政府要求移动运营商可以追溯用户的上网行为，这就要求移动运营 商能记录下所有用户的上网日志。 1.2 产品定位 ZXVMAX-S系统可以长时间的保存用户的上网日志，并可通过公网IP、目的IP、IMSI、 MSISDN、
3. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=16
   4 功能 4.11 NAT日志入库功能 ZXVMAX-S支持将NAT日志保存到GP数据库，可通过公网IP或目的IP查询NAT日志。 4.12 北向接口 支持中移规范北向接口，中移集团查询平台下发查询命令，上网日志系统将查询结果以文件方 式上传至查询平台。包括云化核心网的4G/5G上网日志查询、4G/5G控制面用户面XDR查询、 4G/5G软采XDR查询、VoLTE XDR查询、5G消息XDR查询，以及原始码流查询功能。 4.13 上网日
4. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=7
   ZXVMAX-S 多维价值分析系统 产品描述（上网日志业务） 图2-3 ZXVMAX逻辑结构图 模块功能  采集层： à Probe：探针，支持硬采和软采，负责进行标准接口的信令和Packet抓包。 à Firewall log Process：防火墙日志处理单元，负责接收防火墙日志。 à Data Merge Process：数据合成单元，负责关联网元信令和防火墙日志，生成上网日志 数据。  存储共享层： à 分布式存储集群：采用
5. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（数据业务）.pdf | page=5
   ZXVMAX-S 多维价值分析系统 产品描述（数据业务） 1.2 产品定位 ZXVMAX-S系统是是面向用户的端到端的网络运维和运营分析产品，立足于从客户角度去感知 和分析网络和业务信息，通过对海量数据灵活的挖掘和分析，从网元、用户、终端、业务等多个 维度，对业务使用过程中的质量和特征进行全方位的挖掘。ZXVMAX-S支持XDR查询、实时保 障、多维分析、会话跟踪等功能，为移动通讯网络的运维和运营提供全面支撑。 ZXVMAX-S系统为运
```

### 3.3 RRF 融合

RRF（Reciprocal Rank Fusion）不需要手动设置 dense/sparse 的权重，而是根据多个召回列表中的排序位置进行融合。它适合教学和初始实验：当你不确定权重如何设置时，可以先用 RRF 作为稳健基线。


In [ ]:
rrf_docs = vectordb_hybrid.similarity_search(
    query,
    k=5,
    ranker_type="rrf",
    ranker_params={"k": 100},
)

print_docs("RRF 混合检索结果", rrf_docs)


**预期输出：**

```text

===== RRF 混合检索结果 =====
1. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=13
   4 功能 本章包含如下主题：  上网日志保存 10  上网日志查询 10  上网日志批量导入查询 11  日志管理 11  账号管理 11  角色管理 12  资源监控 12  告警管理 12  省级网关对接 12  拨测结果自动比对功能 12  NAT日志入库功能 13  北向接口 13  上网日志历史查询 13  云化上网日志XDR查询 13 以下介绍ZXVMAX-S的主要的功能。 4.1 上网日志保存 
2. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（数据业务）.pdf | page=5
   ZXVMAX-S 多维价值分析系统 产品描述（数据业务） 1.2 产品定位 ZXVMAX-S系统是是面向用户的端到端的网络运维和运营分析产品，立足于从客户角度去感知 和分析网络和业务信息，通过对海量数据灵活的挖掘和分析，从网元、用户、终端、业务等多个 维度，对业务使用过程中的质量和特征进行全方位的挖掘。ZXVMAX-S支持XDR查询、实时保 障、多维分析、会话跟踪等功能，为移动通讯网络的运维和运营提供全面支撑。 ZXVMAX-S系统为运
3. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=4
   1 产品定位与特点 本章包含如下主题：  产品背景 1  产品定位 1  产品特点 1 1.1 产品背景 随着移动通信网络的不断发展，移动智能终端用户数突飞猛进，越来越多的用户开始使用移动终 端上网。为了网络安全考虑，政府要求移动运营商可以追溯用户的上网行为，这就要求移动运营 商能记录下所有用户的上网日志。 1.2 产品定位 ZXVMAX-S系统可以长时间的保存用户的上网日志，并可通过公网IP、目的IP、IMSI、 MSISDN、
4. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（端到端业务）.pdf | page=6
   1 产品定位与特点 图1-2 ZXVMAX-S 端到端系统架构  ZXVMAX-S端到端提供4大主要功能：用户分析、网络分析、专题分析和评估优化。  综合网络分析，无线核心网综合解决方案。 ZXVMAX-S端到端系统支持对无线数据、核心网数据进行关联，支持问题定界，可以更加准 确的定位问题点，确定网络的性能瓶颈，更有针对性的指导优化。  多样化处理模式，实时分析、周期任务与指定任务驱动，满足不同用户的需求。 ZXVMAX-S端到端
5. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=16
   4 功能 4.11 NAT日志入库功能 ZXVMAX-S支持将NAT日志保存到GP数据库，可通过公网IP或目的IP查询NAT日志。 4.12 北向接口 支持中移规范北向接口，中移集团查询平台下发查询命令，上网日志系统将查询结果以文件方 式上传至查询平台。包括云化核心网的4G/5G上网日志查询、4G/5G控制面用户面XDR查询、 4G/5G软采XDR查询、VoLTE XDR查询、5G消息XDR查询，以及原始码流查询功能。 4.13 上网日
```

### 3.4 动态权重策略

实际业务中，不同问题适合不同权重：

- 包含“告警码、接口名、产品名、字段名、编号”等关键词的问题，适合提高 BM25 sparse 权重；
- 表达更自然、需要语义理解的问题，适合提高 dense 权重；
- 不确定时使用均衡权重，或者直接使用 RRF。

下面用一个简单规则演示动态权重。这个规则不是最终工程方案，但适合帮助学生理解权重调节的意义。


In [ ]:
import re

def choose_hybrid_weights(question: str):
    keyword_patterns = [
        r"\d{4,}",          # 长数字、告警码、编号
        r"ZXVMAX-S",          # 产品名
        r"接口|字段|告警|日志|NAT|IMSI|MSISDN|URL",
        r"有哪些|包括哪些|列出|表格",
    ]
    semantic_patterns = [
        r"为什么|原因|如何优化|怎么解决|影响|优缺点|解释",
    ]

    if any(re.search(pattern, question, re.IGNORECASE) for pattern in keyword_patterns):
        return [0.4, 0.6], "keyword_first"
    if any(re.search(pattern, question, re.IGNORECASE) for pattern in semantic_patterns):
        return [0.7, 0.3], "semantic_first"
    return [0.5, 0.5], "balanced"

for test_query in [
    "ZXVMAX-S上网日志有哪些功能？",
    "为什么需要做5GC业务质量定界？",
    "114020039告警如何处理？",
]:
    weights, query_type = choose_hybrid_weights(test_query)
    docs = vectordb_hybrid.similarity_search(
        test_query,
        k=3,
        ranker_type="weighted",
        ranker_params={"weights": weights},
    )
    print(f"\nQuery: {test_query}")
    print(f"query_type={query_type}, weights={weights}")
    print_docs("动态权重检索结果", docs, preview_chars=120)


**预期输出：**

```text

Query: ZXVMAX-S上网日志有哪些功能？
query_type=keyword_first, weights=[0.4, 0.6]

===== 动态权重检索结果 =====
1. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=13
   4 功能 本章包含如下主题：  上网日志保存 10  上网日志查询 10  上网日志批量导入查询 11  日志管理 11  账号管理 11  角色管理 12  资源监控 12  告警管理 12  省级网关对接 12  拨
2. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=4
   1 产品定位与特点 本章包含如下主题：  产品背景 1  产品定位 1  产品特点 1 1.1 产品背景 随着移动通信网络的不断发展，移动智能终端用户数突飞猛进，越来越多的用户开始使用移动终 端上网。为了网络安全考虑，政府要求移动运营
3. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=16
   4 功能 4.11 NAT日志入库功能 ZXVMAX-S支持将NAT日志保存到GP数据库，可通过公网IP或目的IP查询NAT日志。 4.12 北向接口 支持中移规范北向接口，中移集团查询平台下发查询命令，上网日志系统将查询结果以文件方 式上

Query: 为什么需要做5GC业务质量定界？
query_type=semantic_first, weights=[0.7, 0.3]

===== 动态权重检索结果 =====
1. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=18
   4 功能 会话分析 该功能针对PDU会话创建、修改、释放的失败流程中的全流程首次失败的失败码进行分析。当 流程失败时，通过多接口的关联分析，找出最先失败的接口的流程，针对该流程的失败原因进行 定界。同时，对PDU会话创建、修改、释放的失败码
2. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（语音业务）.pdf | page=18
   4 功能与业务 专题名称 专题说明 EPS FB分析专题 支持五大视图（分析视图/概览视图/全局视图/时延视图/接通视图）差异化切入剖析EPS FB网络。其中，分析视图以四大阶段主题页（起呼-回落-接通-返回）的形式提供5G语音 回落的全程
3. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=18
   4G高价值小区。  接入诊断：检查项目包括注册、鉴权、业务请求、PDU会话建立、N2切入信令流程，并对失 败进行定界。  驻留诊断：检查项目包括市网协同、规划建设、网络质量调优、网络感知调优。  体验诊断：针对热点业务进行感知评估和质

Query: 114020039告警如何处理？
query_type=keyword_first, weights=[0.4, 0.6]

===== 动态权重检索结果 =====
1. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=17
   网元和流程维度下的分布，并最终下钻到XDR错误详单以及全流程信令配合。 注册分析 该功能针对注册失败流程中的全流程首次失败的失败码进行分析。当注册流程失败时，通过多接 口的关联分析，找出最先失败的接口的流程，针对该流程的失败原因进行定界。同
2. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=18
   4G高价值小区。  接入诊断：检查项目包括注册、鉴权、业务请求、PDU会话建立、N2切入信令流程，并对失 败进行定界。  驻留诊断：检查项目包括市网协同、规划建设、网络质量调优、网络感知调优。  体验诊断：针对热点业务进行感知评估和质
3. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=13
   节点（具体的业务协议处理模块）和合并输出节点（处理结果合并输出模块），一个机房部署一 套Prober，Prober通过合并输出节点与ZXVMAX-S相连。 Prober探针 系统 Prober探针系统支持目前主流移动通信网络的数据采集处理，
```

## 4. Rerank 精排

混合检索解决的是“尽量召回更多可能相关的候选文档”，但候选列表里仍可能有冗余或弱相关内容。Rerank 的作用是对 query 与候选文档逐对打分，再重新排序，通常用于：

1. 先用 hybrid search 召回较多候选，例如 top 10 或 top 20；
2. 再用 reranker 精排，保留 top 3 到 top 5；
3. 最后把精排后的高质量上下文交给 LLM。

本节对比三种 Rerank 方式，并使用 Xinference 部署的 Qwen3 Reranker 完成后续实测：

- **Cohere Rerank**：云端 API，效果成熟，需要 `COHERE_API_KEY`；
- **Jina Reranker**：云端 HTTP API，支持多语言，需要 `JINA_API_KEY`；
- **Xinference Qwen3 Reranker**：通过 Xinference 的 OpenAI 兼容 `{XINFERENCE_OPENAI_BASE_URL}/rerank` 接口调用 `my_qwen3_reranker_0.6b`，直接返回排序索引和相关性分数，适合内网、私有化和教学环境。


### 4.1 Cohere Rerank（仅示例）

如果环境中配置了 `COHERE_API_KEY`，可以使用 `langchain_cohere.CohereRerank`。没有 API Key 时跳过该单元即可。


In [ ]:
# import os

# if os.getenv("COHERE_API_KEY"):
#     from langchain_cohere import CohereRerank

#     cohere_compressor = CohereRerank(
#         model="rerank-multilingual-v3.0",
#         top_n=3,
#     )
#     cohere_reranked_docs = cohere_compressor.compress_documents(weighted_docs, query=query)
#     print_docs("Cohere Rerank 结果", cohere_reranked_docs)
# else:
#     print("未配置 COHERE_API_KEY，跳过 Cohere Rerank 示例。")


### 4.2 Jina Reranker（仅示例）

Jina Reranker 可以直接通过 HTTP API 调用。下面给出通用模板：如果环境中配置了 `JINA_API_KEY`，就会实际调用；否则只打印跳过信息。


In [ ]:
# import os
# import httpx
# from langchain_core.documents import Document

# def jina_rerank(query, documents, top_n=3, model="jina-reranker-v2-base-multilingual"):
#     api_key = os.getenv("JINA_API_KEY")
#     if not api_key:
#         print("未配置 JINA_API_KEY，跳过 Jina Rerank 示例。")
#         return []

#     payload = {
#         "model": model,
#         "query": query,
#         "top_n": top_n,
#         "documents": [doc.page_content for doc in documents],
#     }
#     headers = {
#         "Authorization": f"Bearer {api_key}",
#         "Content-Type": "application/json",
#     }
#     response = httpx.post(
#         "https://api.jina.ai/v1/rerank",
#         headers=headers,
#         json=payload,
#         timeout=60,
#         trust_env=False,
#     )
#     response.raise_for_status()
#     results = response.json().get("results", [])

#     reranked = []
#     for item in results:
#         idx = item["index"]
#         score = item.get("relevance_score", 0.0)
#         doc = documents[idx]
#         reranked.append(Document(
#             page_content=doc.page_content,
#             metadata={**doc.metadata, "rerank_score": float(score), "rerank_model": model},
#         ))
#     return reranked

# jina_reranked_docs = jina_rerank(query, weighted_docs, top_n=3)
# if jina_reranked_docs:
#     print_docs("Jina Rerank 结果", jina_reranked_docs)


### 4.3 Xinference Qwen3 Reranker（推荐）

Xinference 把 reranker 模型发布为 OpenAI 兼容风格的相关性接口，路径是 `{XINFERENCE_OPENAI_BASE_URL}/rerank`，
请求体与 Cohere 的 `/v2/rerank` 基本一致：`model` / `query` / `documents` / `top_n`，返回 `results[].index` 与 `results[].relevance_score`。

与用生成接口“让模型口算一个分数”相比，专用 rerank 接口一次请求就能对整批候选打分，延迟低、分数可比较，返回的下标也不会与原文档错位。

实现时要注意两点：

1. 传给服务的是 `Document.page_content` 列表，返回结果中的 `index` 用于映射回原始 LangChain `Document`；
2. 原文档 metadata 需要保留，同时增加 `rerank_score` 和 `rerank_model`，便于后续调试、展示和评估。

> **读数提示**：Qwen3 Reranker 输出的相关性分数会明显向 `1.0` 饱和，教学时请关注**排序是否合理**，不要拿绝对分值当阈值标准。

In [ ]:
import os
import httpx
from dotenv import load_dotenv, find_dotenv
from langchain_core.documents import Document

_ = load_dotenv(find_dotenv())

# 优先复用统一配置单元里的 reranker 配置，缺省时直接读 .env
RERANKER_BASE_URL = globals().get("RERANKER_BASE_URL") or os.environ["XINFERENCE_OPENAI_BASE_URL"]
RERANKER_MODEL = globals().get("RERANKER_MODEL") or os.environ["XINFERENCE_RERANKER_MODEL_ID"]


def xinference_qwen3_rerank(query, documents, top_n=3):
    """调用 Xinference 的 /v1/rerank 对候选文档精排，返回按相关性降序的文档。"""
    if not documents or top_n <= 0:
        return []

    response = httpx.post(
        RERANKER_BASE_URL.rstrip("/") + "/rerank",
        headers={"Authorization": "Bearer " + os.environ["XINFERENCE_API_KEY"]},
        json={
            "model": RERANKER_MODEL,
            "query": query,
            "documents": [doc.page_content for doc in documents],
            "top_n": min(top_n, len(documents)),
        },
        timeout=60,
        trust_env=False,
    )
    response.raise_for_status()

    reranked_docs = []
    for item in response.json().get("results", []):
        doc = documents[item["index"]]
        reranked_docs.append(Document(
            page_content=doc.page_content,
            metadata={
                **doc.metadata,
                "rerank_score": float(item.get("relevance_score", 0.0)),
                "rerank_model": RERANKER_MODEL,
            },
        ))

    reranked_docs.sort(key=lambda item: item.metadata.get("rerank_score", 0.0), reverse=True)
    return reranked_docs

## 5. 混合检索与 Xinference Rerank 实测

下面先用 weighted 混合检索召回 8 条候选，再用 Xinference 上的 Qwen3 Reranker 精排保留 5 条。教学时可以对比 rerank 前后的来源页和内容片段，观察精排是否把更贴近问题的上下文排到前面。


In [ ]:
rerank_query = "什么是ZXVMAX-S的5GC业务？包含哪些功能？"

candidate_docs = vectordb_hybrid.similarity_search(
    rerank_query,
    k=8,
    ranker_type="weighted",
    ranker_params={"weights": [0.6, 0.4]},
)

print_docs("Rerank 前：混合检索候选", candidate_docs, preview_chars=140)

reranked_docs = xinference_qwen3_rerank(rerank_query, candidate_docs, top_n=5)
print_docs("Rerank 后：Xinference Qwen3 精排结果", reranked_docs, preview_chars=180)


**预期输出：**

```text

===== Rerank 前：混合检索候选 =====
1. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=29
   ZXVMAX-S 多维价值分析系统 产品描述（5GC业务） 图5-8 会话跟踪组网逻辑图 26 SJ-20230406152852-016 | 2023-03-30（R1.0）
2. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=0
   ZXVMAX-S 多维价值分析系统 产品描述（5GC业务） 产品版本：V6.23 中兴通讯股份有限公司 地址：深圳市南山区高新技术产业园科技南路中兴通讯大厦 邮编：518057 电话：0755-26770800 400-8301118 800-8301118（座机） 技术支持网站
3. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=9
   ZXVMAX-S 多维价值分析系统 产品描述（5GC业务） 说明： 上图随各个现场的实际情况有所不同。例如现场是否接入了IMS系统，是否接入了2G系统、3G系统、LTE 系统、5G系统。 ZXVMAX-S系统组件组网说明参见表2-1。 表2-1 ZXVMAX-S系统组件组网说明 
4. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=5
   ZXVMAX-S 多维价值分析系统 产品描述（5GC业务） 1.2 产品定位 ZXVMAX-S系统是面向用户的网络运维和运营分析产品，立足于从客户角度去感知和分析网络 和业务信息，通过对海量数据灵活的挖掘和分析，从网元、用户、终端、业务等多个维度，对业 务使用过程中的质量和特征进

===== Rerank 后：Xinference Qwen3 精排结果 =====
1. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=5 | rerank_score=1.00
   ZXVMAX-S 多维价值分析系统 产品描述（5GC业务） 1.2 产品定位 ZXVMAX-S系统是面向用户的网络运维和运营分析产品，立足于从客户角度去感知和分析网络 和业务信息，通过对海量数据灵活的挖掘和分析，从网元、用户、终端、业务等多个维度，对业 务使用过程中的质量和特征进行全方位的挖掘。ZXVMAX-S支持实时分析（网络质量、数据业 务质量等）和事后
2. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=9 | rerank_score=1.00
   ZXVMAX-S 多维价值分析系统 产品描述（5GC业务） 说明： 上图随各个现场的实际情况有所不同。例如现场是否接入了IMS系统，是否接入了2G系统、3G系统、LTE 系统、5G系统。 ZXVMAX-S系统组件组网说明参见表2-1。 表2-1 ZXVMAX-S系统组件组网说明 ZXVMAX-S系统 组件 产品（组件）说明 应用分析产品 实现性能优化、事件处
3. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=0 | rerank_score=1.00
   ZXVMAX-S 多维价值分析系统 产品描述（5GC业务） 产品版本：V6.23 中兴通讯股份有限公司 地址：深圳市南山区高新技术产业园科技南路中兴通讯大厦 邮编：518057 电话：0755-26770800 400-8301118 800-8301118（座机） 技术支持网站：http://support.zte.com.cn 电子邮件：800@zte.
4. source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=29 | rerank_score=1.00
   ZXVMAX-S 多维价值分析系统 产品描述（5GC业务） 图5-8 会话跟踪组网逻辑图 26 SJ-20230406152852-016 | 2023-03-30（R1.0）
```

## 6. 混合检索小结

本章的推荐实践路径是：

1. **基础版**：先用 dense 向量检索搭建可工作的 RAG；
2. **进阶版**：引入 Milvus BM25 sparse 字段，用 weighted 或 RRF 做混合检索；
3. **质量优化**：当召回结果多而杂时，在混合检索后加入 Rerank，把更相关的上下文排到前面；
4. **模型选择**：Cohere/Jina 适合云端 API 场景，Xinference Qwen3 Reranker 适合内网、私有化和教学环境。

需要特别注意：Milvus 的 `sparse` 字段是检索字段，不是展示字段。无论是否加入 Rerank，最终返回给 LangChain 文档对象的都应该是 `text/source/page` 等普通字段。


## 7. 完整混合检索问答链（含记忆）

前面的混合检索和 Rerank 解决的是“召回和排序”的问题，但一个完整的 RAG 教学案例还需要把它们接成真正的问答链。

这一小节把前面准备的能力真正接成一条完整链路：

1. **查询理解**：多轮改写后重新规范化问题、识别业务类型并选择 Dense / Hybrid 路线；
2. **检索与精排**：按路线调用 `ZXVMAXS` 或 `ZXVMAXS_HYBRID`，执行 Metadata Filter 和可选 Rerank；
3. **上下文优化**：对召回结果去重、邻接扩展并控制上下文预算；
4. **生成与记忆**：用统一模板生成答案，并通过 `RunnableWithMessageHistory` 支持多轮对话。

### 7.1 完整链路设计

和前面的单轮检索问答相比，这里多了一个“会话状态”：

- 第一轮问“ZXVMAX-S 上网日志有哪些功能？”，系统先检索，再生成答案；
- 第二轮问“那它支持保存多久？”，系统会把前一轮对话带入当前回答，避免把“它”理解错；
- 如果课堂上要展示内部过程，可以直接打印历史和改写后的检索问题。

这类写法更接近真实业务里的连续问答助手。实际执行路径为：

`历史问题改写 → 查询规范化 → 业务类型识别 → Dense / Hybrid 路由 → Metadata Filter → Rerank → 去重 → 邻接扩展 → 预算控制 → 生成答案`

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# 复用前面章节里已经创建好的检索库；缺失时按统一配置重建。
qa_connection_args = globals().get("connection_args", {"uri": os.environ["MILVUS_URI"]})
qa_hybrid_db = globals().get("vectordb_hybrid")
if qa_hybrid_db is None:
    from pymilvus import MilvusClient
    from langchain_milvus import Milvus, BM25BuiltInFunction
    from langchain_openai import OpenAIEmbeddings

    qa_emb = OpenAIEmbeddings(
        base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
        api_key=os.environ["XINFERENCE_API_KEY"],
        model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
    )
    _qa_milvus_client = MilvusClient(**qa_connection_args)

    qa_hybrid_db = Milvus(
        embedding_function=qa_emb,
        builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
        collection_name="ZXVMAXS_HYBRID",
        connection_args=qa_connection_args,
        vector_field=["dense", "sparse"],
        text_field="text",
        consistency_level="Bounded",
    )

# 复用已经建好的混合检索库；重新过滤一次字段，避免 sparse/dense 被当作原始数据返回。
qa_hybrid_db.fields = [
    field for field in qa_hybrid_db.fields
    if field not in {"dense", "sparse", "vector"}
]

# Dense 路线复用简单向量库，供查询路由真正选择单一检索。
qa_dense_db = globals().get("vectordb_dense")
if qa_dense_db is None:
    qa_dense_db = globals().get("vectordb")
if qa_dense_db is None:
    from langchain_milvus import Milvus
    from langchain_openai import OpenAIEmbeddings
    qa_dense_emb = globals().get("my_emb")
    if qa_dense_emb is None:
        qa_dense_emb = OpenAIEmbeddings(
            base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
            api_key=os.environ["XINFERENCE_API_KEY"],
            model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
        )
    qa_dense_db = Milvus(
        embedding_function=qa_dense_emb,
        collection_name="ZXVMAXS",
        connection_args=qa_connection_args,
        vector_field="dense",
        text_field="text",
        consistency_level="Bounded",
    )
qa_dense_db.fields = [
    field for field in qa_dense_db.fields
    if field not in {"dense", "sparse", "vector"}
]

# 复用前面章节里已经创建好的 LLM；如果当前会话里没有，再按统一模型配置重建一次。
qa_llm = globals().get("my_llm")
if qa_llm is None:
    import os
    import httpx
    from dotenv import load_dotenv, find_dotenv
    from langchain.chat_models import init_chat_model

    _ = load_dotenv(find_dotenv())
    if not os.getenv("vLLM_QWEN_API_KEY"):
        raise RuntimeError("未配置 vLLM_QWEN_API_KEY，无法运行检索问答链示例。")

    qa_llm = init_chat_model(
        model=os.environ["vLLM_QWEN_MODEL_ID"],
        model_provider="openai",
        base_url=os.environ["vLLM_QWEN_BASE_URL"],
        api_key=os.environ["vLLM_QWEN_API_KEY"],
        http_client=httpx.Client(trust_env=False),
        temperature=0.3,
        max_tokens=3000,
    )

rewrite_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "你是一个检索问题改写器。请结合对话历史，把当前问题改写成一个独立、完整、适合检索的问题。只输出改写后的问题，不要解释。",
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

answer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "你是 VMAX 运维助手。只基于给定上下文回答，不要编造。"
        "如果上下文没有提供答案，明确回答“我不知道”。"
        "回答末尾补充引用来源和页码。",
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    (
        "human",
        "原始问题：{question}\n\n"
        "改写后的检索问题：{standalone_question}\n\n"
        "相关上下文：\n{context}\n\n"
        "请给出简洁、准确的中文回答。",
    ),
])

question_rewriter = rewrite_prompt | qa_llm | StrOutputParser()


def _format_retrieval_docs(docs, preview_chars=260):
    lines = []
    for i, doc in enumerate(docs, 1):
        source = str(doc.metadata.get("source_name") or doc.metadata.get("source") or doc.metadata.get("file_path") or "未知来源")
        page = str(doc.metadata.get("page_number", doc.metadata.get("page", "-")))
        section = str(doc.metadata.get("section_title", "未识别章节"))
        preview = doc.page_content[:preview_chars].replace("\n", " ")
        lines.append(f"[{i}] source={source} | page={page} | section={section}\n{preview}")
    return "\n\n".join(lines)


def _retrieve_by_route(question: str, route: str, k=12):
    if route == "hybrid":
        return qa_hybrid_db.similarity_search(
            question,
            k=k,
            ranker_type="weighted",
            ranker_params={"weights": [0.6, 0.4]},
        )
    return qa_dense_db.similarity_search(question, k=k)


def build_hybrid_qa_inputs(inputs: dict):
    question = inputs["question"]
    chat_history = inputs.get("chat_history", [])
    standalone_question = question
    if chat_history:
        standalone_question = question_rewriter.invoke({
            "question": question,
            "chat_history": chat_history,
        }).strip()

    # 问题改写后必须重新规范化和识别条件，避免沿用含糊追问的旧策略。
    query_strategy = prepare_retrieval_query(standalone_question)
    standalone_question = query_strategy["normalized_query"]
    retrieval_route = query_strategy["route"]
    docs = _retrieve_by_route(standalone_question, retrieval_route, k=12)
    business_type = query_strategy["business_type"]
    if business_type:
        filtered_docs = [
            doc for doc in docs
            if doc.metadata.get("business_type") == business_type
        ]
        if filtered_docs:
            docs = filtered_docs

    if "xinference_qwen3_rerank" in globals():
        docs = xinference_qwen3_rerank(standalone_question, docs, top_n=5)

    docs = deduplicate_documents(docs)
    adjacent_source_docs = globals().get("split_docs", [])
    adjacent_expansion_applied = bool(adjacent_source_docs)
    if adjacent_expansion_applied:
        docs = expand_adjacent_chunks(docs[:3], adjacent_source_docs, window=1)
    docs = select_context_with_budget(docs, max_chars=6000)

    return {
        "question": question,
        "standalone_question": standalone_question,
        "retrieval_route": retrieval_route,
        "business_type": business_type,
        "adjacent_expansion_applied": adjacent_expansion_applied,
        "source_documents": docs,
        "chat_history": chat_history,
        "context": _format_retrieval_docs(docs),
    }

hybrid_qa_chain = (
    RunnableLambda(build_hybrid_qa_inputs)
    | answer_prompt
    | qa_llm
    | StrOutputParser()
)

conversation_store = {}


def get_hybrid_qa_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in conversation_store:
        conversation_store[session_id] = ChatMessageHistory()
    return conversation_store[session_id]

hybrid_qa_with_memory = RunnableWithMessageHistory(
    hybrid_qa_chain,
    get_hybrid_qa_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

print("混合检索问答链已准备完成。")
print("- 支持查询规范化与 Metadata Filter")
print("- 支持 Dense / Hybrid 实际路由")
print("- 支持去重、邻接扩展与预算控制")
print("- 支持可选 Xinference Rerank")
print("- 支持会话记忆")


**预期输出：**

```text
混合检索问答链已准备完成。
- 支持查询规范化与 Metadata Filter
- 支持 Dense / Hybrid 实际路由
- 支持去重、邻接扩展与预算控制
- 支持可选 Xinference Rerank
- 支持会话记忆
~/env_rag_uv/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
```

（输出里的 `~` 代表运行机器上的用户主目录，每台机器不同。）

### 7.2 单轮问答效果

先跑一轮常见的功能类问题，确认完整问答链可以直接工作。

In [ ]:
qa_session_id = "hybrid_qa_demo"
first_question = "ZXVMAX-S上网日志有哪些功能？"
first_answer = hybrid_qa_with_memory.invoke(
    {"question": first_question},
    config={"configurable": {"session_id": qa_session_id}},
)

print("用户问题：", first_question)
print("\n模型回答：\n")
print(first_answer)

**预期输出：**

```text
用户问题： ZXVMAX-S上网日志有哪些功能？

模型回答：



我不知道。现有上下文未提及 ZXVMAX-S 上网日志的功能。

来源：../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（语音业务）.pdf，page=7；../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf，page=7；../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（端到端业务）.pdf，page=6。
```

### 7.3 带记忆的多轮追问

第二轮问题故意不把主语说完整，观察系统是否会利用前一轮对话把问题补全后再检索。

In [ ]:
follow_up_question = "那它支持保存多久？"

# 先把当前会话里的历史取出来，顺便展示改写后的检索问题。
current_history = get_hybrid_qa_history(qa_session_id).messages
preview_inputs = {
    "question": follow_up_question,
    "chat_history": current_history,
}
preview_result = build_hybrid_qa_inputs(preview_inputs)

print("用户追问：", follow_up_question)
print("改写后的检索问题：", preview_result["standalone_question"])
print("实际检索路径：", preview_result["retrieval_route"])
print("识别业务类型：", preview_result["business_type"] or "未限定")
print("邻接扩展：", "已执行" if preview_result["adjacent_expansion_applied"] else "未执行（当前会话无 split_docs）")
print("\n检索上下文预览：\n")
print(preview_result["context"][:1400])

second_answer = hybrid_qa_with_memory.invoke(
    {"question": follow_up_question},
    config={"configurable": {"session_id": qa_session_id}},
)

print("\n模型回答：\n")
print(second_answer)

**预期输出：**

```text
用户追问： 那它支持保存多久？
改写后的检索问题： ZXVMAX-S 上网日志支持保存多久？
实际检索路径： hybrid
识别业务类型： 上网日志业务
邻接扩展： 已执行

检索上下文预览：

[1] source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=4 | section=未识别章节
1 产品定位与特点 本章包含如下主题：  产品背景 1  产品定位 1  产品特点 1 1.1 产品背景 随着移动通信网络的不断发展，移动智能终端用户数突飞猛进，越来越多的用户开始使用移动终 端上网。为了网络安全考虑，政府要求移动运营商可以追溯用户的上网行为，这就要求移动运营 商能记录下所有用户的上网日志。 1.2 产品定位 ZXVMAX-S系统可以长时间的保存用户的上网日志，并可通过公网IP、目的IP、IMSI、 MSISDN、URL等参数查询用户上网日志，满足运营商追溯用户上网行为的需求。 1.3 产品特

[2] source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=13 | section=未识别章节
4 功能 本章包含如下主题：  上网日志保存 10  上网日志查询 10  上网日志批量导入查询 11  日志管理 11  账号管理 11  角色管理 12  资源监控 12  告警管理 12  省级网关对接 12  拨测结果自动比对功能 12  NAT日志入库功能 13  北向接口 13  上网日志历史查询 13  云化上网日志XDR查询 13 以下介绍ZXVMAX-S的主要的功能。 4.1 上网日志保存  支持使用Gbase数据库或HDFS保存上网日志。  上网日志保存时间可配置

[3] source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=16 | section=未识别章节
4 功能 4.11 NAT日志入库功能 ZXVMAX-S支持将NAT日志保存到GP数据库，可通过公网IP或目的IP查询NAT日志。 4.12 北向接口 支持中移规范北向接口，中移集团查询平台下发查询命令，上网日志系统将查询结果以文件方 式上传至查询平台。包括云化核心网的4G/5G上网日志查询、4G/5G控制面用户面XDR查询、 4G/5G软采XDR查询、VoLTE XDR查询、5G消息XDR查询，以及原始码流查询功能。 4.13 上网日志历史查询 当现网使用两个库分别存储2个月数据和6个月数据时，使用历史数据查询操

模型回答：



ZXVMAX-S 上网日志支持**长时间保存**，且**保存时间可配置**。在现网示例中，可使用两个库分别存储 **2 个月**和 **6 个月**的数据。

来源：../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf，page=4、13、16。
```

### 7.4 完整链路小结

这一节把“混合检索”补成了真正可对话的 RAG 链：

- 改写层负责把追问变成独立问题，并在改写后重新执行查询规范化；
- 路由层负责识别业务类型，并真正选择 Dense 或 Hybrid 检索；
- 检索层负责 Metadata Filter、候选召回和可选 Rerank；
- 上下文层负责去重、邻接扩展和预算控制；
- 记忆层负责保留上下文；
- 生成层负责把检索内容组织成稳定答案。

到这里，混合检索章节就不是只有检索对比，而是完整覆盖了“召回 - 精排 - 生成 - 记忆”的闭环。

# RAG进阶：Agentic RAG

前面的基础版 RAG 和混合检索 RAG 都是“开发者固定流程”：用户问题一定会进入某一个检索链，然后由 LLM 基于检索结果回答。

Agentic RAG 的思路是把检索能力包装成工具，让 Agent 根据问题自己判断：

```text
用户问题 -> Agent 判断任务类型 -> 选择工具 -> 观察工具结果 -> 生成最终答案
```

这和 `03_LangChain制作智能体.ipynb` 中的 Agent 思路一致，只是这里把普通工具升级成 RAG 工具：

- **简单检索工具**：基于 `ZXVMAXS`，只做 dense 语义检索，适合语义型问题；
- **混合检索工具**：基于 `ZXVMAXS_HYBRID`，融合 dense + BM25，适合产品名、模块名、告警码、专有名词较多的问题；
- **计算工具**：用于演示 Agent 不一定只会查知识库，也可以选择其他工具完成任务。

本章重点观察三条路径：

1. **单一检索**：语义描述、原因解释等问题调用 `simple_rag_search`；
2. **混合检索**：产品名、告警码、接口名等精确关键词问题调用 `hybrid_rag_search`；
3. **不走检索**：问候、改写、通用能力等不依赖企业知识的问题由 Agent 直接回答，不调用检索工具。

教学时可以把它理解为：RAG 是一种能力，Agent 是调度这些能力的决策层。

## 1. Agentic RAG 原理与工具设计

工具设计的关键不是“把所有逻辑都塞给 Agent”，而是把每个工具的边界写清楚：

| 场景 | Agent 行为 | 适合问题 | 底层能力 |
|---|---|---|---|
| 单一检索 | 调用 `simple_rag_search` | 语义描述、原因解释、概念说明 | Milvus Dense 检索 |
| 混合检索 | 调用 `hybrid_rag_search` | 产品名、模块名、告警码、接口名、字段名 | Milvus Dense + BM25，可接 Rerank |
| 不走检索 | 不调用检索工具，直接回答 | 问候、文本改写、通用能力问题 | LLM 自身能力 |
| 非检索工具 | 按需调用 `calculate` | 数学表达式 | Python AST 白名单计算 |

工具的 docstring 会进入模型上下文，直接影响 Agent 是否会选对工具。因此工具说明要具体、克制、可区分。

## 2. Agentic RAG 工具实现

下面的代码会重新连接两个已经构建好的 Milvus 集合，保证这一节可以单独运行：

- `ZXVMAXS`：简单 dense 检索集合；
- `ZXVMAXS_HYBRID`：混合检索集合，包含 `dense` 和 `sparse` 两个向量字段。

注意：Milvus 的向量字段只用于检索，不应该作为原始字段返回，所以这里仍然会过滤 `dense/sparse/vector` 字段。

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from pymilvus import MilvusClient
from langchain_milvus import Milvus, BM25BuiltInFunction
from langchain_openai import OpenAIEmbeddings
from langchain_core.tools import tool

# 这一节独立重建连接，方便从 Agentic RAG 章节开始单独运行。
agent_connection_args = {
    "uri": os.environ["MILVUS_URI"],
}

agent_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)

_agent_milvus_client = MilvusClient(**agent_connection_args)

agent_simple_db = Milvus(
    embedding_function=agent_emb,
    collection_name="ZXVMAXS",
    connection_args=agent_connection_args,
    vector_field="dense",
    text_field="text",
    consistency_level="Bounded",
)
agent_simple_db.fields = [
    field for field in agent_simple_db.fields
    if field not in {"dense", "sparse", "vector"}
]

agent_hybrid_db = Milvus(
    embedding_function=agent_emb,
    builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
    collection_name="ZXVMAXS_HYBRID",
    connection_args=agent_connection_args,
    vector_field=["dense", "sparse"],
    text_field="text",
    consistency_level="Bounded",
)
agent_hybrid_db.fields = [
    field for field in agent_hybrid_db.fields
    if field not in {"dense", "sparse", "vector"}
]

def _format_tool_docs(docs, max_chars=320):
    """把检索结果压缩成适合工具返回给 Agent 的文本。"""
    lines = []
    for i, doc in enumerate(docs, 1):
        source = (doc.metadata.get("source_name") or doc.metadata.get("source")
                  or doc.metadata.get("file_path") or "未知来源")
        page = doc.metadata.get("page_number", doc.metadata.get("page", "-"))
        section = doc.metadata.get("section_title", "未识别章节")
        content = doc.page_content[:max_chars].replace("\n", " ")
        lines.append(f"[{i}] source={source} | page={page} | section={section}\n{content}")
    return "\n\n".join(lines)

def _filter_and_optimize_tool_docs(question, docs, use_rerank=False):
    strategy = prepare_retrieval_query(question)
    business_type = strategy["business_type"]
    if business_type:
        filtered = [doc for doc in docs if doc.metadata.get("business_type") == business_type]
        if filtered:
            docs = filtered
    if use_rerank and "xinference_qwen3_rerank" in globals():
        docs = xinference_qwen3_rerank(strategy["normalized_query"], docs, top_n=4)
    docs = deduplicate_documents(docs)
    adjacent_source_docs = globals().get("split_docs", [])
    if adjacent_source_docs:
        docs = expand_adjacent_chunks(docs[:3], adjacent_source_docs, window=1)
    return strategy, select_context_with_budget(docs, max_chars=5000)

@tool
def simple_rag_search(question: str) -> str:
    """使用简单 dense 向量检索查询 ZXVMAX-S 知识库。适合语义型、描述型、概念解释类问题。"""
    strategy = prepare_retrieval_query(question)
    docs = agent_simple_db.similarity_search(strategy["normalized_query"], k=8)
    strategy, docs = _filter_and_optimize_tool_docs(question, docs)
    if not docs:
        return "简单检索没有找到相关资料。"
    return "检索路径：single_dense\n" + _format_tool_docs(docs)

@tool
def hybrid_rag_search(question: str) -> str:
    """使用 Milvus 混合检索查询 ZXVMAX-S 知识库。适合包含产品名、模块名、告警码、接口名、字段名等关键词的问题。"""
    strategy = prepare_retrieval_query(question)
    normalized_question = strategy["normalized_query"]
    docs = agent_hybrid_db.similarity_search(
        normalized_question,
        k=8,
        ranker_type="weighted",
        ranker_params={"weights": [0.5, 0.5]},
    )
    strategy, docs = _filter_and_optimize_tool_docs(question, docs, use_rerank=True)
    if not docs:
        return "混合检索没有找到相关资料。"
    return "检索路径：hybrid\n" + _format_tool_docs(docs)

@tool
def calculate(expression: str) -> str:
    """安全计算数学表达式。适合四则运算、乘方和括号表达式。不要用于知识库问答。"""
    import ast
    import operator

    operators = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.Pow: operator.pow,
        ast.USub: operator.neg,
    }

    def safe_eval(node):
        if isinstance(node, ast.Expression):
            return safe_eval(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in operators:
            return operators[type(node.op)](safe_eval(node.left), safe_eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in operators:
            return operators[type(node.op)](safe_eval(node.operand))
        raise ValueError(f"不支持的表达式：{ast.dump(node)}")

    try:
        tree = ast.parse(expression, mode="eval")
        return f"计算结果：{expression} = {safe_eval(tree)}"
    except Exception as exc:
        return f"计算错误：{exc}"

agentic_rag_tools = [simple_rag_search, hybrid_rag_search, calculate]

print("Agentic RAG 工具准备完成：")
for item in agentic_rag_tools:
    print("-", item.name, ":", item.description)

**预期输出：**

```text
Agentic RAG 工具准备完成：
- simple_rag_search : 使用简单 dense 向量检索查询 ZXVMAX-S 知识库。适合语义型、描述型、概念解释类问题。
- hybrid_rag_search : 使用 Milvus 混合检索查询 ZXVMAX-S 知识库。适合包含产品名、模块名、告警码、接口名、字段名等关键词的问题。
- calculate : 安全计算数学表达式。适合四则运算、乘方和括号表达式。不要用于知识库问答。
```

### 2.1 独立测试检索工具

在正式交给 Agent 调度之前，先直接调用工具本身。这样做有两个好处：

1. 确认工具和外部系统连接正常；
2. 让学生看到“Agent 调用工具”之前，工具返回的原始内容是什么。

如果工具本身不可用，Agent 再聪明也无法得到正确结果。

In [ ]:
tool_test_questions = [
    (simple_rag_search, {"question": "为什么需要做5GC业务质量定界？"}),
    (hybrid_rag_search, {"question": "ZXVMAX-S上网日志有哪些功能？"}),
    (calculate, {"expression": "(15 * 37 + 228) / 3"}),
]

for tool_item, args in tool_test_questions:
    print("=" * 80)
    print("工具：", tool_item.name)
    print("输入：", args)
    print(tool_item.invoke(args)[:1200])

**预期输出：**

```text
================================================================================
工具： simple_rag_search
输入： {'question': '为什么需要做5GC业务质量定界？'}
检索路径：single_dense
[1] source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=18 | section=未识别章节
4 功能 会话分析 该功能针对PDU会话创建、修改、释放的失败流程中的全流程首次失败的失败码进行分析。当 流程失败时，通过多接口的关联分析，找出最先失败的接口的流程，针对该流程的失败原因进行 定界。同时，对PDU会话创建、修改、释放的失败码组合进行聚类分析，用于进行问题的定位处 理。将PDU会话创建、修改、释放失败次数在小区、网元、终端等维度进行TOP分析，用于辅助 会话流程失败的定界。 切换分析 该功能针对5GC内切换流程中的首次失败的失败码进行分析。当流程失败时，通过多接口的关 联分析，找出最先失败的接口的流程，针对该流程的失败原因进行定界。同时，对切换的失败码 组合进行聚类分析，用于进行问题的定位处理。将切换失败次数在小区、网

[2] source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（语音业务）.pdf | page=18 | section=未识别章节
4 功能与业务 专题名称 专题说明 EPS FB分析专题 支持五大视图（分析视图/概览视图/全局视图/时延视图/接通视图）差异化切入剖析EPS FB网络。其中，分析视图以四大阶段主题页（起呼-回落-接通-返回）的形式提供5G语音 回落的全程分析，直观呈现每个阶段的感知KQI性能指标、关联接口KPI性能指标的发展趋 势，并支持跳转到定界分析+多维分析，为5G网络语音发展初中期的业务发展和用户感知提 供数据分析。 融合语音分析专 题 以多网对比（VoNR-VoLTE-VoBB-VoWiFi）+六个阶段（承载-注册-接续-切换-保持- 语音质量，分接入方式）的形式，提供在IMS融合核心网下各种语音接入方式的全景式分析 及对比呈现，实现从一

[3] source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=18 | section=未识别章节
4G高价值小区。  接入诊断：检查项目包括注册、鉴权、业务请求、PDU会话建立、N2切入信令流程，并对失 败进行定界。  驻留诊断：检查项目包括市网协同、规划建设、网络质量调优、网络感知调优。  体验诊断：针对热点业务进行感知评估和质差定界、定位。 4.2 维护保障 SJ-20230406152852-016 | 2023-03-30（R1.0） 15
================================================================================
工具： hybrid_rag_search
输入： {'question': 'ZXVMAX-S上网日志有哪些功能？'}
检索路径：hybrid
[1] source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（语音业务）.pdf | page=7 | section=未识别章节
ZXVMAX-S 多维价值分析系统 产品描述（语音业务） ZXVMAX-S语音业务系统支持事件详细信息的呈现，也支持基于经验算法的问题定位，提升 分析解决问题的效率，满足不同层次用户的需求。  多种应用，自由选择。 ZXVMAX-S语音业务支持信令回溯、记录查询、呼叫分析、网络质量分析、业务质量定界、 时延分析等多种应用，可以根据目标用户的需求，开启对应权限，部署相应模块。 4 SJ-20230406152852-013 | 2023-03-30（R1.0）

[2] source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=7 | section=未识别章节
ZXVMAX-S 多维价值分析系统 产品描述（5GC业务） ZXVMAX-S系统支持 满足用户需求 根据用户需求进行数据自定义的查询或分析。 满足不同场景下用户分析解决问题的需求。  多种应用，自由选择。 ZXVMAX-S支持性能分析、事件分析、终端分析等多种应用，可以根据目标用户的需求，开 启对应权限，部署相应模块。 4 SJ-20230406152852-016 | 2023-03-30（R1.0）

[3] source=../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（端到端业务）.pdf | page=6 | section=未识别章节
1 产品定位与特点 图1-2 ZXVMAX-S 端到端系统架构  ZXVMAX-S端到端提供4大主要功能：用户分析、网络分析、专题分析和评估优化。  综合网络分析，无线核心网综合解决方案。 ZXVMAX-S端到端系统支持对无线数据、核心网数据进行关联，支持问题定界，可以更加准 确的定位问题点，确定网络的性能瓶颈，更有针对性的指导优化。  多样化处理模式，实时分析、周期任务与指定任务驱动，满足不同用户的需求。 ZXVMAX-S端到端系统支持对网络性能KPI、端到端质量等进行实时分析，仪表盘呈现，对 超出门限指标进行预警，指导运维人员快速响应解决问题。 ZXVMAX-S端到端系统支持定制周期任务，利用空闲时段完成运算，让用户第一时
================================================================================
工具： calculate
输入： {'expression': '(15 * 37 + 228) / 3'}
计算结果：(15 * 37 + 228) / 3 = 261.0
```

## 3. 构建工具调用 Agent

当前环境使用的是 LangChain 1.x，新版推荐用 `create_agent()` 创建 Agent 图；旧教程中常见的 `AgentExecutor`、`create_tool_calling_agent` 在部分新版本里已经不再从 `langchain.agents` 直接导出。

它和普通 RAG 链的最大区别是：

- 普通 RAG 链：开发者提前决定一定走哪个 retriever；
- Agentic RAG：模型先理解问题，再决定是否调用工具、调用哪个工具、如何利用工具结果回答。

默认关闭 `debug`，输出更适合教材阅读。如果课堂上想观察模型消息、工具调用和工具返回结果，可以把代码中的 `DEBUG_AGENTIC_RAG` 改为 `True`。

In [ ]:
import os
import httpx
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

_ = load_dotenv(find_dotenv())
agent_api_key = os.getenv("vLLM_QWEN_API_KEY")


agentic_system_prompt = (
    "你是一个企业知识库问答助手。"
    "请先判断问题是否依赖 ZXVMAX-S 企业知识，再决定是否调用工具。"
    "场景一：语义描述、原因解释、概念说明等企业知识问题，调用 simple_rag_search，且不要再调用 hybrid_rag_search。"
    "场景二：包含产品名、模块名、告警码、接口名、字段名、编号等精确关键词的企业知识问题，调用 hybrid_rag_search，且不要再调用 simple_rag_search。"
    "场景三：问候、文本改写、通用能力等不依赖企业知识的问题，不调用任何检索工具，直接回答。"
    "数学计算调用 calculate；这属于非检索路径。"
    "使用检索工具回答时必须基于工具返回内容；依据不足时明确说明知识库中没有找到充分依据，并列出引用来源和页码。"
    "不走检索时不要虚构企业内部信息，也不需要伪造引用。"
)


agentic_llm = my_llm

DEBUG_AGENTIC_RAG = False

agentic_agent = create_agent(
        model=agentic_llm,
        tools=agentic_rag_tools,
        system_prompt=agentic_system_prompt,
        debug=DEBUG_AGENTIC_RAG,
    )

print("Agentic RAG Agent 已创建。")

**预期输出：**

```text
Agentic RAG Agent 已创建。
```

## 4. Agentic RAG 效果测试

这里准备三个核心场景，观察 Agent 是否选择正确路径：

1. **语义解释问题**：应调用 `simple_rag_search`，走单一 Dense 检索；
2. **精确关键词问题**：应调用 `hybrid_rag_search`，走混合检索；
3. **非知识库问题**：不调用检索工具，由 Agent 直接回答。

另外保留一个计算问题作为扩展示例：它可以调用 `calculate`，但仍属于“不走知识库检索”。

真实项目中，Agent 的工具选择不应只靠一次演示判断，需要结合 LangSmith、日志或评测集持续观察。

In [ ]:
agentic_examples = [
    {"scene": "单一检索", "question": "为什么需要做业务质量定界？", "expected_tools": ["simple_rag_search"]},
    {"scene": "混合检索", "question": "ZXVMAX-S上网日志有哪些功能？", "expected_tools": ["hybrid_rag_search"]},
    {"scene": "不走检索", "question": "你好，请用一句话介绍你能做什么。", "expected_tools": []},
    {"scene": "非检索工具（扩展）", "question": "帮我计算 (15 * 37 + 228) / 3。", "expected_tools": ["calculate"]},
]

def print_agent_answer(result):
    """兼容 LangChain 1.x create_agent 返回的 messages 结构。"""
    messages = result.get("messages", []) if isinstance(result, dict) else []
    if not messages:
        print(result)
        return
    final_message = messages[-1]
    content = getattr(final_message, "content", final_message)
    print(content)

def extract_called_tools(result):
    called = []
    for message in result.get("messages", []) if isinstance(result, dict) else []:
        for call in getattr(message, "tool_calls", []) or []:
            name = call.get("name") if isinstance(call, dict) else getattr(call, "name", None)
            if name:
                called.append(name)
    return called

if "agentic_agent" not in globals():
    print("Agentic Agent 尚未创建，通常是因为没有配置 vLLM_QWEN_API_KEY。")
else:
    for example in agentic_examples:
        question = example["question"]
        print("=" * 80)
        print("测试场景：", example["scene"])
        print("用户问题：", question)
        response = agentic_agent.invoke({
            "messages": [{"role": "user", "content": question}],
        })
        called_tools = extract_called_tools(response)
        print("预期调用：", example["expected_tools"] or "不调用工具")
        print("实际调用：", called_tools or "不调用工具")
        print("路径判断：", "通过" if called_tools == example["expected_tools"] else "需调优")
        print("最终回答：")
        print_agent_answer(response)

**预期输出：**

```text
================================================================================
测试场景： 单一检索
用户问题： 为什么需要做业务质量定界？
预期调用： ['simple_rag_search']
实际调用： ['simple_rag_search']
路径判断： 通过
最终回答：


根据知识库内容，做“业务质量定界”的必要性主要体现在以下几个方面：

1. **数据业务快速发展，质量问题更复杂**  
   随着移动智能终端用户数快速增长，移动数据业务进入快速发展阶段，数据业务流量呈几何级数增长，网络中新业务不断出现。业务类型越多，质量问题的来源也越复杂，需要能够区分问题到底来自哪类业务、哪个环节。  
   来源：ZXVMAX-S（V6.23）产品描述（数据业务），第 4 页；ZXVMAX-S（V6.23）产品描述（5GC业务），第 4 页。

2. **网络复杂度提升，不同业务对网络质量要求不同**  
   网络中存在 2/3/4/5G 混合组网，网络复杂度不断上升；同时不同业务对时延、速率、成功率、稳定性等质量指标的要求不同。如果不做业务质量定界，很难判断某个体验差的问题是网络问题、业务问题、终端问题还是应用侧问题。  
   来源：ZXVMAX-S（V6.23）产品描述（数据业务），第 4 页；ZXVMAX-S（V6.23）产品描述（5GC业务），第 4 页。

3. **降低网优运维成本，提高问题定位效率**  
   网络和业务复杂度提升后，网优运维面临更高要求，运维成本不断上升。通过业务质量分析，可以对业务持续时长、流量、响应时延、成功率、平均速率、请求次数、失败原因、用户数等指标进行统计，并对感知质量差的业务进行分析，从而帮助更快定位问题。  
   来源：ZXVMAX-S（V6.23）产品描述（数据业务），第 16 页。

4. **支撑用户感知优化和精细化运营**  
   业务分析功能可以围绕网页浏览、视频播放、即时通信、应用下载、游戏、OTT 等业务提供 KPI/KQI 指标，并构建用户业务感知 QoE 体系。业务质量定界有助于识别哪些业务体验差、影响哪些用户、问题集中在哪些维度，从而支撑网络优化和业务运营。  
   来源：ZXVMAX-S（V6.23）产品描述（数据业务），第 16 页。

5. **缓解“增量不增收”问题**  
   部分低价值业务，例如 P2P，可能占用大量带宽，而运营商需要不断扩容以满足需求，但收益未必同步增长。通过业务质量分析和定界，可以更好地识别高价值业务、低价值业务以及资源占用情况，为网络资源优化和经营决策提供依据。  
   来源：ZXVMAX-S（V6.23）产品描述（数据业务），第 4 页；ZXVMAX-S（V6.23）产品描述（5GC业务），第 4 页。

简而言之，做业务质量定界是为了在复杂网络和多样业务环境下，快速识别业务体验差的根因，区分网络、业务、终端或应用侧问题，从而提升运维效率、优化用户感知，并支撑网络资源和经营决策。
================================================================================
测试场景： 混合检索
用户问题： ZXVMAX-S上网日志有哪些功能？
预期调用： ['hybrid_rag_search']
实际调用： ['hybrid_rag_search', 'hybrid_rag_search', 'hybrid_rag_search', 'hybrid_rag_search', 'simple_rag_search']
路径判断： 需调优
最终回答：


根据《ZXVMAX-S（V6.23）产品描述（上网日志业务）》第 13 页，ZXVMAX-S 上网日志业务的主要功能包括：

1. **上网日志保存**
   - 支持使用 **Gbase 数据库** 或 **HDFS** 保存上网日志。
   - 上网日志保存时间可配置，最短保存 **7 天**，最长可保存 **1 年**。
   - 支持自动清理超过保存时间的上网日志。

2. **上网日志查询**
   - 支持通过 Web 界面查询用户上网日志。
   - 可结合公网 IP、目的 IP、IMSI、MSISDN、URL 等参数进行查询，用于追溯用户上网行为。

3. **上网日志批量导入查询**

4. **日志管理**

5. **账号管理**

6. **角色管理**

7. **资源监控**

8. **告警管理**

9. **省级网关对接**

10. **拨测结果自动比对功能**

11. **NAT 日志入库功能**

12. **北向接口**

13. **上网日志历史查询**

14. **云化上网日志 XDR 查询**

参考资料：  
- 《ZXVMAX-S（V6.23）产品描述（上网日志业务）》，第 4 页、第 13 页。
================================================================================
测试场景： 不走检索
用户问题： 你好，请用一句话介绍你能做什么。
预期调用： 不调用工具
实际调用： 不调用工具
路径判断： 通过
最终回答：


你好，我可以帮你解答 ZXVMAX-S 企业知识问题、查询相关文档依据，也可以处理计算、改写和通用问答等任务。
================================================================================
测试场景： 非检索工具（扩展）
用户问题： 帮我计算 (15 * 37 + 228) / 3。
预期调用： ['calculate']
实际调用： ['calculate']
路径判断： 通过
最终回答：


计算结果为 **261**。
```

## 5. Agentic RAG 小结

Agentic RAG 不是替代普通 RAG，而是在普通 RAG 已经稳定后增加一层“工具调度能力”。建议按下面顺序教学和落地：

1. **先做普通 RAG**：保证检索、提示词和回答质量可控；
2. **再做混合检索 + Rerank**：提升召回与排序质量；
3. **最后引入 Agent**：把不同检索链、计算工具、业务 API 封装成工具，让模型按任务选择；
4. **持续评估工具选择**：Agent 的核心风险不是代码能不能跑，而是它是否在正确场景选择正确工具。

三种核心决策可以概括为：语义问题走单一检索，精确关键词问题走混合检索，不依赖企业知识的问题不走检索。

一句话总结：普通 RAG 解决“如何基于知识库回答”，Agentic RAG 解决“是否需要查、应该怎么查、是否改用其他能力”。

# RAG进阶：效果评估

本章评估前面构建的混合检索 RAG：先检查检索层是否命中正确文档，再使用 LLM-as-Judge 对比 RAG 与无知识库基线答案的忠实度、相关性和完整性，最后说明局限性与后续扩展方向。

## 1. 检索层与生成层分开评估

### 1.1 检索指标

**对应打基础：** `C06_RAG系统评估/01_评估介绍.md` 的检索评估。先计算 Hit@K、MRR、页码和章节命中，再运行原有的忠实度、相关性和完整性评分。

In [ ]:
def is_expected_document(doc, expected):
    metadata = doc.metadata
    source_name = metadata.get("source_name") or Path(str(metadata.get("source", ""))).name
    page_number = metadata.get("page_number", metadata.get("page"))
    section_title = metadata.get("section_title")
    checks = []
    if expected.get("source_contains"):
        checks.append(expected["source_contains"] in source_name)
    if expected.get("page_number") is not None:
        checks.append(page_number == expected["page_number"])
    if expected.get("section_title"):
        checks.append(section_title == expected["section_title"])
    return all(checks) if checks else False

def evaluate_retrieval_result(docs, expected, k=5):
    top_docs = docs[:k]
    ranks = [
        rank for rank, doc in enumerate(top_docs, 1)
        if is_expected_document(doc, expected)
    ]
    first_rank = min(ranks) if ranks else None
    return {
        f"hit@{k}": int(first_rank is not None),
        "reciprocal_rank": 1 / first_rank if first_rank else 0.0,
        "first_relevant_rank": first_rank,
    }

retrieval_eval_set = [
    {"question": "数据可靠性包括哪些层？", "expected": {"source_contains": "上网日志业务", "page_number": 21, "section_title": "7.3 数据可靠性"}},
    {"question": "ZXVMAX-S上网日志有哪些功能？", "expected": {"source_contains": "上网日志业务"}},
    {"question": "什么是5GC业务？", "expected": {"source_contains": "5GC业务"}},
]
print(f"检索评测集已准备：{len(retrieval_eval_set)} 条；建议扩展到 50 条以上。")

**预期输出：**

```text
检索评测集已准备：3 条；建议扩展到 50 条以上。
```

### 1.2 生成指标

生成层使用忠实度、相关性和完整性三个指标：忠实度检查答案是否有上下文依据，相关性检查是否直接回答问题，完整性检查是否覆盖参考答案要点。后面的 LLM-as-Judge 会分别对 RAG 与无知识库基线进行评分。

## 2. 端到端生成效果评估（LLM-as-Judge）

### 2.1 评估方案

RAG 链路搭完之后，最后一步是评估它是否真的比“裸大模型”更可靠。

本章使用 **LLM-as-Judge** 的思路，做一个轻量级对比实验：

1. 准备 3 个 VMAX 领域的测试问题，并给出每个问题的参考答案要点；
2. 复用“RAG进阶：混合检索”的查询理解、Weighted 混合召回、Metadata Filter、可选 Rerank 和上下文优化流程生成 RAG 答案；
3. 使用同一个问题直接调用大模型，生成无知识库基线答案；
4. 让另一个 LLM 从 **忠实度、相关性、完整性** 三个维度打分（1-5 分，5 分最好）；
5. 用表格对比两种方案的分数，观察混合检索 RAG 对幻觉抑制和知识准确性的作用。

> 说明：这里不引入额外评估库（如 Ragas），只使用已经配置好的大模型做裁判，便于在 `env_rag` 环境中直接运行。分数仅供参考，真实业务中建议结合人工标注。

### 2.2 准备评估环境与混合检索器

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

# 评估环境准备：复用统一模型配置，连接混合检索向量库
import os
import httpx
import re
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
from langchain_milvus import Milvus, BM25BuiltInFunction
from pymilvus import MilvusClient
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 模型
eval_llm = my_llm
eval_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)

connection_args = {"uri": os.environ["MILVUS_URI"]}
_eval_milvus_client = MilvusClient(**connection_args)

eval_hybrid_db = Milvus(
    embedding_function=eval_emb,
    builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
    collection_name="ZXVMAXS_HYBRID",
    connection_args=connection_args,
    vector_field=["dense", "sparse"],
    text_field="text",
    consistency_level="Bounded",
)
# 检索时不返回向量原始字段
eval_hybrid_db.fields = [
    field for field in eval_hybrid_db.fields
    if field not in {"dense", "sparse", "vector"}
]

def _format_eval_context(docs, preview_chars=500):
    lines = []
    for i, doc in enumerate(docs, 1):
        source = (doc.metadata.get("source_name") or doc.metadata.get("source")
                  or doc.metadata.get("file_path") or "未知来源")
        page = doc.metadata.get("page_number", doc.metadata.get("page", "-"))
        section = doc.metadata.get("section_title", "未识别章节")
        content = doc.page_content[:preview_chars].replace("\n", " ")
        lines.append(f"[{i}] source={source} | page={page} | section={section}\n{content}")
    return "\n\n".join(lines)

def retrieve_docs_for_eval(question: str, k: int = 20):
    """复用混合检索章节的查询理解、召回、过滤、Rerank 与去重流程。"""
    strategy = prepare_retrieval_query(question)
    normalized_question = strategy["normalized_query"]
    docs = eval_hybrid_db.similarity_search(
        normalized_question,
        k=k,
        ranker_type="weighted",
        ranker_params={"weights": [0.6, 0.4]},
    )
    business_type = strategy["business_type"]
    if business_type:
        # 带业务类型再召回一次，避免先召回后过滤只剩少量候选。
        business_docs = eval_hybrid_db.similarity_search(
            f"{normalized_question} {business_type}",
            k=k,
            ranker_type="weighted",
            ranker_params={"weights": [0.6, 0.4]},
        )
        docs = deduplicate_documents([*docs, *business_docs])
        filtered_docs = [
            doc for doc in docs
            if doc.metadata.get("business_type") == business_type
        ]
        if filtered_docs:
            docs = filtered_docs

    if "xinference_qwen3_rerank" in globals():
        docs = xinference_qwen3_rerank(normalized_question, docs, top_n=8)
    return strategy, deduplicate_documents(docs)

def build_eval_context(question: str):
    strategy, docs = retrieve_docs_for_eval(question)
    adjacent_source_docs = globals().get("split_docs", [])
    if adjacent_source_docs:
        docs = expand_adjacent_chunks(docs[:3], adjacent_source_docs, window=1)
    docs = select_context_with_budget(docs, max_chars=6000)
    return strategy, docs, _format_eval_context(docs)

print("评估环境准备完成：查询理解 → 混合检索 → Metadata Filter → 可选 Rerank → 上下文优化。")


**预期输出：**

```text
评估环境准备完成：查询理解 → 混合检索 → Metadata Filter → 可选 Rerank → 上下文优化。
```

### 2.3 执行检索层评估

In [ ]:
retrieval_eval_rows = []
print(f"开始执行检索层评估，共 {len(retrieval_eval_set)} 条问题。")

for index, sample in enumerate(retrieval_eval_set, 1):
    question = sample["question"]
    print(f"[{index}/{len(retrieval_eval_set)}] {question}", flush=True)
    strategy, docs = retrieve_docs_for_eval(question, k=20)
    metrics = evaluate_retrieval_result(docs, sample["expected"], k=5)
    retrieval_eval_rows.append({
        "问题": question,
        "business_type": strategy["business_type"] or "未限定",
        "Hit@5": metrics["hit@5"],
        "MRR": round(metrics["reciprocal_rank"], 3),
        "首个相关结果排名": metrics["first_relevant_rank"] or "未命中",
    })
    print(
        f"    Hit@5={metrics['hit@5']} | MRR={metrics['reciprocal_rank']:.3f} "
        f"| first_rank={metrics['first_relevant_rank'] or '未命中'}",
        flush=True,
    )

retrieval_eval_df = pd.DataFrame(retrieval_eval_rows)
display(retrieval_eval_df)
print("\n检索层平均指标：")
print(retrieval_eval_df[["Hit@5", "MRR"]].mean().round(3))

**预期输出：**

```text
开始执行检索层评估，共 3 条问题。
[1/3] 数据可靠性包括哪些层？
    Hit@5=0 | MRR=0.000 | first_rank=未命中
[2/3] ZXVMAX-S上网日志有哪些功能？
    Hit@5=1 | MRR=1.000 | first_rank=1
[3/3] 什么是5GC业务？
    Hit@5=1 | MRR=1.000 | first_rank=1

检索层平均指标：
Hit@5    0.667
MRR      0.667
dtype: float64
```

### 2.4 构造 RAG 链、基线链与评测集

In [ ]:
# 构造 RAG 问答链与基线问答链
rag_prompt = PromptTemplate.from_template("""
你是 VMAX 运维助手，请严格根据以下上下文回答问题。

上下文：
{context}

问题：{question}

回答要求：
1. 只基于上下文回答，不要编造上下文之外的信息；
2. 如果上下文没有提供答案，请回答“我不知道”；
3. 回答简洁、准确，不超过500字；
4. 不输出思考过程。

/no_think

回答：""")

# 检索在生成循环中显式执行，便于打印过程并把同一份上下文复用于评分。
rag_answer_chain = (
    rag_prompt
    | eval_llm
    | StrOutputParser()
)

baseline_prompt = PromptTemplate.from_template("""
请直接、简洁地回答下面的问题，不超过300字。
如果不确定，请明确说明不确定，不要编造具体厂商、产品或功能。
不要输出思考过程。

问题：{question}

/no_think
""")
baseline_answer_chain = baseline_prompt | eval_llm | StrOutputParser()

# 评估数据集：问题 + 参考答案要点
eval_set = [
    {
        "question": "ZXVMAX-S上网日志有哪些功能？",
        "reference": "上网日志保存、上网日志查询、批量导入查询、日志管理、账号/角色管理、资源监控、告警管理、省级网关对接、拨测结果自动比对、NAT日志入库、北向接口、上网日志历史查询、云化上网日志XDR查询",
    },
    {
        "question": "什么是ZXVMAX-S的5GC业务？",
        "reference": "面向5G核心网的多维价值分析系统；包含5G精准开通、5G业务运营、一键投诉、区域感知、用户感知、实时跟踪/会话跟踪等功能",
    },
    {
        "question": "ZXVMAX-S如何保障数据可靠性？",
        "reference": "采集层采用冗余磁盘阵列并支持补充采集；存储共享层使用MPP数据库多副本、冗余磁盘阵列和双控制器；应用分析层的软件与配置数据部署在冗余磁盘阵列上，确保故障时数据不丢失",
    },
]

print(f"准备评估 {len(eval_set)} 个问题")


**预期输出：**

```text
准备评估 3 个问题
```

### 2.5 生成 RAG 与基线答案

In [ ]:
# 生成答案（逐题打印检索、RAG 和基线生成进度）
import time

total_questions = len(eval_set)
generation_started_at = time.perf_counter()

for index, item in enumerate(eval_set, 1):
    question = item["question"]
    question_started_at = time.perf_counter()
    print("=" * 80, flush=True)
    print(f"[{index}/{total_questions}] 问题：{question}", flush=True)

    print("  ① 正在执行混合检索与上下文优化...", flush=True)
    retrieval_started_at = time.perf_counter()
    strategy, docs, context = build_eval_context(question)
    item["retrieval_strategy"] = strategy
    item["retrieved_docs"] = docs
    item["retrieved_context"] = context
    print(
        f"     完成：route=hybrid | business_type={strategy['business_type'] or '未限定'} "
        f"| 最终上下文={len(docs)} 块/{len(context)} 字符 "
        f"| 耗时={time.perf_counter() - retrieval_started_at:.1f}s",
        flush=True,
    )
    for doc_index, doc in enumerate(docs[:5], 1):
        source = (doc.metadata.get("source_name") or doc.metadata.get("source")
                  or doc.metadata.get("file_path") or "未知来源")
        section = doc.metadata.get("section_title", "未识别章节")
        page = doc.metadata.get("page_number", doc.metadata.get("page", "-"))
        print(f"       [{doc_index}] {source} | page={page} | section={section}", flush=True)

    print("  ② 正在生成 RAG 答案...", flush=True)
    rag_started_at = time.perf_counter()
    item["rag_answer"] = rag_answer_chain.invoke({
        "context": context,
        "question": question,
    })
    print(f"     完成，耗时={time.perf_counter() - rag_started_at:.1f}s", flush=True)
    print("     RAG 预览：", item["rag_answer"][:160].replace("\n", " "), flush=True)

    print("  ③ 正在生成无知识库基线答案...", flush=True)
    baseline_started_at = time.perf_counter()
    item["baseline_answer"] = baseline_answer_chain.invoke({"question": question})
    print(f"     完成，耗时={time.perf_counter() - baseline_started_at:.1f}s", flush=True)
    print("     基线预览：", item["baseline_answer"][:160].replace("\n", " "), flush=True)
    print(f"  本题总耗时：{time.perf_counter() - question_started_at:.1f}s", flush=True)

print("=" * 80)
print(f"RAG 与基线答案已生成完成，总耗时：{time.perf_counter() - generation_started_at:.1f}s")


**预期输出：**

```text
================================================================================
[1/3] 问题：ZXVMAX-S上网日志有哪些功能？
  ① 正在执行混合检索与上下文优化...
     完成：route=hybrid | business_type=上网日志业务 | 最终上下文=3 块/992 字符 | 耗时=1.3s
       [1] ../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=11 | section=未识别章节
       [2] ../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=9 | section=未识别章节
       [3] ../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=0 | section=未识别章节
  ② 正在生成 RAG 答案...
     完成，耗时=1.6s
     RAG 预览：   我不知道
  ③ 正在生成无知识库基线答案...
     完成，耗时=2.0s
     基线预览：   不确定。  我无法确认“ZXVMAX-S上网日志”的具体功能清单，不能准确列出其支持的功能。为避免误导，不编造具体功能。建议查阅该产品官方文档、产品手册或联系厂商技术支持获取准确说明。
  本题总耗时：4.9s
================================================================================
[2/3] 问题：什么是ZXVMAX-S的5GC业务？
  ① 正在执行混合检索与上下文优化...
     完成：route=hybrid | business_type=5GC业务 | 最终上下文=3 块/1113 字符 | 耗时=1.6s
       [1] ../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=7 | section=未识别章节
       [2] ../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=27 | section=未识别章节
       [3] ../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（5GC业务）.pdf | page=5 | section=未识别章节
  ② 正在生成 RAG 答案...
     完成，耗时=9.0s
     RAG 预览：   上下文未直接给出“5GC业务”的明确定义。根据提供的产品描述，ZXVMAX-S（5GC业务）是面向用户网络运维和运营分析的多维价值分析系统，支持性能分析、事件分析、终端分析等多种应用，可从网元、用户、终端、业务等维度，对业务使用过程中的质量和特征进行挖掘分析；同时支持实时分析（如网络质量、数据业务质量）和事后分析（
  ③ 正在生成无知识库基线答案...
     完成，耗时=2.7s
     基线预览：   不确定具体定义。一般而言，“ZXVMAX-S 的 5GC 业务”可能指该产品/平台基于 5G 核心网（5GC）提供的通信或数据服务能力，例如会话、消息、语音、视频、短信或企业通信等业务。具体功能、支持范围和实现方式应以中兴通讯或相关官方产品文档为准。
  本题总耗时：13.3s
================================================================================
[3/3] 问题：ZXVMAX-S如何保障数据可靠性？
  ① 正在执行混合检索与上下文优化...
     完成：route=hybrid | business_type=未限定 | 最终上下文=3 块/1492 字符 | 耗时=0.9s
       [1] ../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（上网日志业务）.pdf | page=20 | section=未识别章节
       [2] ../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（数据业务）.pdf | page=32 | section=未识别章节
       [3] ../knowledge_path/VMAX-S/ZXVMAX-S（V6.23）产品描述（语音业务）.pdf | page=31 | section=未识别章节
  ② 正在生成 RAG 答案...
     完成，耗时=6.6s
     RAG 预览：   ZXVMAX-S 通过分层架构设计保障数据可靠性和可恢复性：  1. **采集层**：运行环境采用冗余磁盘阵列设计。在服务器损坏、单一硬盘故障等场景下，替换故障硬件后即可继续工作，数据不会丢失；同时支持数据补充采集，确保系统数据连续性。  2. **存储共享层**：根据业务架构采用不同存储方案，并设置多个冗余备份。
  ③ 正在生成无知识库基线答案...
     完成，耗时=2.1s
     基线预览：   不确定。我无法确认 ZXVMAX-S 的具体数据可靠性保障机制，不能编造其是否采用冗余存储、纠删码、多副本、快照、备份或容灾等技术。建议查阅中兴通讯/产品官方文档或联系厂商获取准确说明。
  本题总耗时：9.5s
================================================================================
RAG 与基线答案已生成完成，总耗时：27.7s
```

### 2.6 使用 LLM-as-Judge 评分

In [ ]:
# LLM-as-Judge：对答案进行三维度评分
judge_prompt = PromptTemplate.from_template("""
你是一名公正的评估专家。请根据下面的上下文、问题和参考答案要点，对助手的回答进行评分。

【上下文】
{context}

【问题】
{question}

【参考答案要点】
{reference}

【助手回答】
{answer}

请从以下三个维度打分（1-5 分，5 分最好），并给出简短理由：
1. 忠实度：回答内容是否完全能从上下文中找到依据。
2. 相关性：回答是否直接针对问题，没有答非所问。
3. 完整性：回答是否覆盖了参考答案要点。

请严格按以下格式输出：
忠实度: <分数>
相关性: <分数>
完整性: <分数>
理由: <一句话理由>

不要输出思考过程。
/no_think
""")

judge_chain = judge_prompt | eval_llm | StrOutputParser()

def parse_judge_result(text: str):
    result = {"faithfulness": 0, "relevance": 0, "completeness": 0, "reason": ""}
    faith = re.search(r"忠实度[:：]\s*(\d+)", text)
    rel = re.search(r"相关性[:：]\s*(\d+)", text)
    comp = re.search(r"完整性[:：]\s*(\d+)", text)
    reason = re.search(r"理由[:：]\s*(.+)", text, re.DOTALL)
    if faith:
        result["faithfulness"] = int(faith.group(1))
    if rel:
        result["relevance"] = int(rel.group(1))
    if comp:
        result["completeness"] = int(comp.group(1))
    if reason:
        result["reason"] = reason.group(1).strip().replace("\n", " ")
    return result

total_judgements = len(eval_set) * 2
judgement_index = 0
judge_started_at = time.perf_counter()

for question_index, item in enumerate(eval_set, 1):
    context = item["retrieved_context"]
    for key, label in [("rag_answer", "RAG"), ("baseline_answer", "基线")]:
        judgement_index += 1
        score_started_at = time.perf_counter()
        print(
            f"[{judgement_index}/{total_judgements}] 正在评分：问题 {question_index} / {label}",
            flush=True,
        )
        judge_text = judge_chain.invoke({
            "context": context,
            "question": item["question"],
            "reference": item["reference"],
            "answer": item[key],
        })
        item[f"{key}_score"] = parse_judge_result(judge_text)
        score = item[f"{key}_score"]
        print(
            f"    忠实度={score['faithfulness']} | 相关性={score['relevance']} "
            f"| 完整性={score['completeness']} | 耗时={time.perf_counter() - score_started_at:.1f}s",
            flush=True,
        )

print(f"评分完成，总耗时：{time.perf_counter() - judge_started_at:.1f}s")


**预期输出：**

```text
[1/6] 正在评分：问题 1 / RAG
    忠实度=1 | 相关性=2 | 完整性=1 | 耗时=4.0s
[2/6] 正在评分：问题 1 / 基线
    忠实度=5 | 相关性=4 | 完整性=1 | 耗时=3.9s
[3/6] 正在评分：问题 2 / RAG
    忠实度=5 | 相关性=5 | 完整性=2 | 耗时=10.5s
[4/6] 正在评分：问题 2 / 基线
    忠实度=1 | 相关性=2 | 完整性=1 | 耗时=5.4s
[5/6] 正在评分：问题 3 / RAG
    忠实度=5 | 相关性=5 | 完整性=3 | 耗时=13.0s
[6/6] 正在评分：问题 3 / 基线
    忠实度=1 | 相关性=2 | 完整性=1 | 耗时=4.2s
评分完成，总耗时：41.1s
```

### 2.7 汇总并对比评估结果

In [ ]:
# 汇总对比结果
rows = []
for item in eval_set:
    rows.append({
        "问题": item["question"],
        "方案": "RAG",
        "忠实度": item["rag_answer_score"]["faithfulness"],
        "相关性": item["rag_answer_score"]["relevance"],
        "完整性": item["rag_answer_score"]["completeness"],
        "平均": round(sum([item["rag_answer_score"][k] for k in ["faithfulness", "relevance", "completeness"]]) / 3, 2),
    })
    rows.append({
        "问题": item["question"],
        "方案": "基线（无RAG）",
        "忠实度": item["baseline_answer_score"]["faithfulness"],
        "相关性": item["baseline_answer_score"]["relevance"],
        "完整性": item["baseline_answer_score"]["completeness"],
        "平均": round(sum([item["baseline_answer_score"][k] for k in ["faithfulness", "relevance", "completeness"]]) / 3, 2),
    })

df = pd.DataFrame(rows)
display(df)

print("\n===== 平均分对比 =====")
print(df.groupby("方案")[["忠实度", "相关性", "完整性", "平均"]].mean().round(2))

print("\n===== 详细理由示例（第一个问题） =====")
print("问题：", eval_set[0]["question"])
print("\n[RAG 回答]")
print(eval_set[0]["rag_answer"][:300])
print("理由：", eval_set[0]["rag_answer_score"]["reason"])
print("\n[基线回答]")
print(eval_set[0]["baseline_answer"][:300])
print("理由：", eval_set[0]["baseline_answer_score"]["reason"])


**预期输出：**

```text

===== 平均分对比 =====
           忠实度   相关性  完整性    平均
方案                             
RAG       3.67  4.00  2.0  3.22
基线（无RAG）  2.33  2.67  1.0  2.00

===== 详细理由示例（第一个问题） =====
问题： ZXVMAX-S上网日志有哪些功能？

[RAG 回答]


我不知道
理由： 助手仅回答“我不知道”，未提供上下文或参考答案中的任何功能要点，虽未明显跑题但无实质答案。

[基线回答]


不确定。

我无法确认“ZXVMAX-S上网日志”的具体功能清单，不能准确列出其支持的功能。为避免误导，不编造具体功能。建议查阅该产品官方文档、产品手册或联系厂商技术支持获取准确说明。
理由： 助手未编造信息且回应了问题，但完全未覆盖参考答案中的上网日志功能要点。
```

## 3. 评估结论与局限性

### 3.1 评估结果解读

通过这个简单的 LLM-as-Judge 实验，通常可以观察到：

- **RAG 方案**：忠实度和完整性通常明显高于基线，因为答案来自本地知识库；
- **基线方案**：容易出现“看似专业、实则编造”的回答（比如把 ZXVMAX-S 说成华为产品），幻觉风险更高；
- **相关性**：如果问题表述清晰，两者通常都不错；但如果问题需要文档中的具体关键词，RAG 更稳。

### 3.2 局限性与后续优化

1. **LLM 裁判也有偏差**：评分模型本身可能偏好“更长”或“更完整”的答案，结果仅供参考；
2. **样本量小**：这里只用了 3 个问题，真实业务建议准备 50+ 条评测集；
3. **检索指标仍较基础**：当前只演示 Hit@K、MRR、页码和章节命中，后续可加入 Recall@K、NDCG 和分业务统计；
4. **可以引入 Ragas**：如果希望更系统，可以安装 `ragas`，用内置的 `faithfulness`、`answer_relevancy`、`context_recall` 等指标。

这个章节把 RAG 从“能跑通”推进到了“可评估”，让整个 Notebook 形成了一个更完整的闭环。

## 4. 延伸阅读：Ragas 评估框架

**Ragas**（Retrieval-Augmented Generation Assessment）是专为 RAG 系统设计的一个开源评估框架，用于在不需要人工标注参考答案的情况下，对检索和生成两个环节进行量化评估。

### 4.1 核心指标

- **Faithfulness（忠实度）**：衡量生成答案是否在提供的上下文中能找到依据，检测模型是否"编造"了上下文中不存在的信息。

- **Answer Relevancy（答案相关性）**：衡量生成答案与问题的相关程度，惩罚答非所问或包含无关冗余内容的回答。

- **Context Relevancy（上下文相关性）**：衡量检索到的文档片段与问题的相关程度，去除无关内容的比例。

- **Context Recall（上下文召回率）**：衡量检索到的上下文是否包含了回答问题的全部必要信息。

### 4.2 为什么使用 Ragas

1. **自动化**：无需人工准备标准答案，适合快速迭代 RAG 链路；
2. **端到端**：同时覆盖检索和生成，便于定位是"召回不够"还是"生成不准"；
3. **与 LangChain 集成友好**：可以直接传入 LangChain 的 `retriever` 和 `chain` 结果进行打分。

### 4.3 使用思路

在真实业务场景中，建议先用本节 LLM-as-Judge 的方法跑通评估流程，再逐步迁移到 Ragas。典型步骤是：准备 50 条以上的评测问题 → 用 LangChain 分别得到问题和答案 → 调用 Ragas 内置指标计算平均分 → 通过拆解"检索指标"和"生成指标"定位优化方向。

通过引入 Ragas，RAG 系统不再只是"能回答"，而是可以持续、可量化地"回答得更好"。